# WEEK 1 FULL NOTEBOOK — THÀNH VIÊN A + B  
## Collision Prediction từ Monocular Video

Notebook này gom **toàn bộ nhiệm vụ tuần 1 của cả Thành viên A và Thành viên B** vào một file duy nhất.

## Output tuần 1 cần có

- `table1_tracking_draft.csv`
- `table2_distance_velocity_draft.csv`
- `collision_events_draft.csv` có ≥15 candidates
- `negative_windows_draft.csv`
- `annotation_protocol_week1.md`
- `related_work_draft_week1.md`
- `object_id_consistency_report.csv`
- `bev_draft_*.png`
- demo video tracking
- `week1_full_status_report.json`
- `week1_full_outputs.zip`

## Lưu ý quan trọng

- Notebook tự tạo **collision candidates** từ TTC/prediction.
- Nhưng `verified_by = member_A_checked` chỉ nên set sau khi bạn xem video.  
- Final lock tuần 1 chỉ đúng khi:
  - `verified_true_positive_events >= 15`
  - `verified_true_negative_windows >= 15`

In [ ]:
# ============================================================
# 0. GLOBAL CONFIG + FOLDERS
# ============================================================

from pathlib import Path
import os, sys, json, time, math, platform, subprocess, importlib, warnings, shutil, zipfile
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------
# Project folders
# -----------------------------
WORKING_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

PROJECT_ROOT = WORKING_ROOT / "collision_prediction_project"
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
PRED_DIR = RESULTS_DIR / "predictions"
VIDEO_DIR = RESULTS_DIR / "videos"
FIGURE_DIR = PROJECT_ROOT / "figures"
PAPER_DIR = PROJECT_ROOT / "paper"
LOG_DIR = RESULTS_DIR / "logs"
REPORT_DIR = RESULTS_DIR / "reports"
REVIEW_DIR = RESULTS_DIR / "review_assets"

for d in [
    PROJECT_ROOT, DATA_DIR, SRC_DIR, RESULTS_DIR, TABLE_DIR, PRED_DIR, VIDEO_DIR,
    FIGURE_DIR, PAPER_DIR, LOG_DIR, REPORT_DIR, REVIEW_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment config
# -----------------------------
SELECTED_SEQUENCES = ["0000", "0001", "0002", "0003", "0004",
                       "0005", "0006", "0007", "0008", "0009", "0010"]
# DA SUA: mo rong tu 2 len 11 sequence de tong thoi luong danh gia du lon,
# tranh ngoai suy FAR/gio tu vai chuc giay nhu truoc. Neu Kaggle session het
# gio/OOM, co the giam lai danh sach nay, nhung CAN giu it nhat ~5-10 phut
# tong thoi luong de FAR/gio co y nghia thong ke.
MAX_FRAMES_PER_SEQ = None  # DA SUA: dung toan bo sequence (truoc la 150 frame/seq ~30s tong)
KITTI_FPS = 10.0

YOLO_WEIGHTS = "yolo11x.pt"       # Nếu OOM: đổi thành "yolo11s.pt" hoặc "yolo11n.pt"
IMG_SIZE = 640
CONF_THRES = 0.35
IOU_THRES = 0.50
TRACKER_CFG = "bytetrack.yaml"

EVAL_IOU_THRESHOLD = 0.50
FILTER_GT_TO_EVAL_RANGE = True

TTC_WARNING_SECONDS = 3.0
TTC_DANGER_SECONDS = 1.5
TTC_DISTANCE_THRESHOLD_M = 15.0
MIN_EVENT_LEN_FRAMES = 3

NEGATIVE_WINDOW_FRAMES = 10
NEGATIVE_STRIDE_FRAMES = 5

COCO_TRAFFIC_CLASSES = {"person", "bicycle", "car", "motorcycle", "bus", "truck"}
KITTI_TRAFFIC_CLASSES = {"car", "van", "truck", "pedestrian", "person_sitting", "cyclist", "tram"}

CLASS_HEIGHT_M = {
    "person": 1.70,
    "pedestrian": 1.70,
    "person_sitting": 1.20,
    "cyclist": 1.70,
    "rider": 1.70,
    "bicycle": 1.40,
    "bike": 1.40,
    "motorcycle": 1.30,
    "car": 1.55,
    "van": 1.90,
    "truck": 3.00,
    "bus": 3.20,
    "tram": 3.20,
    "unknown": 1.60,
}

DEFAULT_FOCAL_PX = 721.0
DEFAULT_CX = 621.0
DEFAULT_CY = 187.5

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)
print("TABLE_DIR   :", TABLE_DIR)
print("PRED_DIR    :", PRED_DIR)
print("VIDEO_DIR   :", VIDEO_DIR)
print("FIGURE_DIR  :", FIGURE_DIR)
print("PAPER_DIR   :", PAPER_DIR)

In [ ]:
import yaml
from pathlib import Path

custom_tracker_config = {
    "tracker_type": "bytetrack",

    "track_high_thresh": 0.5,
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.6,

    "track_buffer": 60,
    "match_thresh": 0.8,

    # Bắt buộc đối với Ultralytics mới
    "fuse_score": True
}

yaml_path = Path(RESULTS_DIR) / "custom_bytetrack.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(custom_tracker_config, f, sort_keys=False)

TRACKER_CFG = str(yaml_path)

print(TRACKER_CFG)

In [ ]:
# ============================================================
# 0.6 ADAPTIVE KALMAN FILTER + MAHALANOBIS GATING cho ByteTrack
# Monkey-patch vào ultralytics BYTETracker
# ============================================================
import numpy as np

# --- Ngưỡng Chi-square 95% với 4 bậc tự do (x, y, w, h) ---
CHI2_THRESH_95 = 9.4877

class AdaptiveKalmanFilter:
    """
    NSA-style Adaptive Kalman Filter:
    - Tự động điều chỉnh ma trận nhiễu R theo kích thước bounding box
    - Box nhỏ/xa → R lớn (nhiều nhiễu đo) → filter tin vào prediction nhiều hơn
    - Box lớn/gần → R nhỏ → filter tin vào measurement nhiều hơn
    """
    def __init__(self):
        # Import kalman gốc để kế thừa logic
        try:
            from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYAH
            self._base = KalmanFilterXYAH()
        except ImportError:
            try:
                from ultralytics.trackers.utils.kalman_filter import KalmanFilterXYWH
                self._base = KalmanFilterXYWH()
            except ImportError:
                self._base = None
                print("[WARN] Không tìm thấy KalmanFilter trong ultralytics. Dùng fallback.")
        self._std_weight_position = 1.0 / 20
        self._std_weight_velocity = 1.0 / 160

    def initiate(self, measurement):
        if self._base:
            return self._base.initiate(measurement)
        # Fallback thuần NumPy nếu import thất bại
        mean_pos = measurement
        mean_vel = np.zeros_like(mean_pos)
        mean = np.r_[mean_pos, mean_vel]
        std = [2 * self._std_weight_position * measurement[3],
               2 * self._std_weight_position * measurement[3],
               1e-2, 2 * self._std_weight_position * measurement[3],
               10 * self._std_weight_velocity * measurement[3],
               10 * self._std_weight_velocity * measurement[3],
               1e-5, 10 * self._std_weight_velocity * measurement[3]]
        covariance = np.diag(np.square(std))
        return mean, covariance

    def predict(self, mean, covariance):
        if self._base:
            return self._base.predict(mean, covariance)
        return mean, covariance

    def update(self, mean, covariance, measurement):
        if self._base:
            # Adaptive: scale R theo kích thước box (measurement[3] = height)
            h = max(float(measurement[3]), 1.0)
            scale = np.clip(h / 60.0, 0.5, 3.0)  # normalize theo box 60px
            # Lấy kết quả gốc rồi điều chỉnh
            new_mean, new_cov = self._base.update(mean, covariance, measurement)
            # Scale covariance nhẹ theo kích thước
            new_cov *= scale
            return new_mean, new_cov
        return mean, covariance

    def gating_distance(self, mean, covariance, measurements, only_position=False):
        """Tính khoảng cách Mahalanobis từ track đến các measurements."""
        if self._base and hasattr(self._base, 'gating_distance'):
            return self._base.gating_distance(mean, covariance, measurements,
                                               only_position=only_position)
        # Fallback: Euclidean distance nếu không có hàm gốc
        if only_position:
            mean_pos = mean[:2]
            diff = measurements[:, :2] - mean_pos
        else:
            mean_pos = mean[:4]
            diff = measurements[:, :4] - mean_pos
        return np.sum(diff ** 2, axis=1)


def apply_mahalanobis_gate(cost_matrix, kalman_filter, tracks, detections):
    """
    Áp dụng Mahalanobis gating: đặt cost=∞ cho cặp (track, det)
    có khoảng cách vượt ngưỡng chi2 95% (4 bậc tự do).
    """
    if len(tracks) == 0 or len(detections) == 0:
        return cost_matrix

    gated = cost_matrix.copy()
    measurements = np.array([
        [d.tlwh[0] + d.tlwh[2] / 2,
         d.tlwh[1] + d.tlwh[3] / 2,
         d.tlwh[2] / max(d.tlwh[3], 1e-6),
         d.tlwh[3]]
        for d in detections
    ], dtype=float)

    for r, track in enumerate(tracks):
        if not hasattr(track, 'mean') or track.mean is None:
            continue
        try:
            dists = kalman_filter.gating_distance(
                track.mean, track.covariance, measurements, only_position=False
            )
            gated[r, dists > CHI2_THRESH_95] = np.inf
        except Exception:
            pass  # Nếu lỗi, giữ nguyên cost matrix

    return gated


# --- Monkey-patch BYTETracker ---
try:
    from ultralytics.trackers.byte_tracker import BYTETracker

    _orig_byte_init = BYTETracker.__init__

    def _adaptive_byte_init(self, args, frame_rate=30):
        _orig_byte_init(self, args, frame_rate)
        self.kalman_filter = AdaptiveKalmanFilter()
        self._mahalanobis_gate = apply_mahalanobis_gate
        print("[OK] AdaptiveKalmanFilter đã thay thế KalmanFilter gốc trong BYTETracker")

    BYTETracker.__init__ = _adaptive_byte_init
    print("[OK] Monkey-patch BYTETracker thành công!")
    print(f"[OK] Mahalanobis gating threshold: chi2(4df, p=0.95) = {CHI2_THRESH_95}")

except Exception as e:
    print(f"[WARN] Không thể patch BYTETracker: {e}")
    print("[INFO] Tiếp tục dùng ByteTrack gốc với custom YAML config")


In [ ]:
# ============================================================
# 1. ENVIRONMENT CHECK + OPTIONAL INSTALLS
# ============================================================

def try_import(import_name, pip_name=None, install=False):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except Exception as e:
        print(f"[WARN] Cannot import {import_name}: {e}")
        if install:
            try:
                print(f"Installing {pip_name} ...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
                return importlib.import_module(import_name)
            except Exception as e2:
                print(f"[WARN] Install failed for {pip_name}: {e2}")
        return None

cv2 = try_import("cv2", "opencv-python", install=False)
plt_mod = try_import("matplotlib.pyplot", "matplotlib", install=False)
torch = try_import("torch", "torch", install=False)
psutil = try_import("psutil", "psutil", install=False)
scipy_signal = try_import("scipy.signal", "scipy", install=False)

ultralytics = try_import("ultralytics", "ultralytics", install=True)
if ultralytics is not None:
    from ultralytics import YOLO
else:
    raise ImportError("Ultralytics is required. Bật Internet ON hoặc cài ultralytics trước.")

mm = try_import("motmetrics", "motmetrics==1.4.0", install=True)

system_info = {
    "generated_at": datetime.now().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
}

if torch is not None:
    system_info["torch_version"] = torch.__version__
    system_info["cuda_available"] = bool(torch.cuda.is_available())
    system_info["cuda_version"] = getattr(torch.version, "cuda", None)
    system_info["gpu_count"] = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
    system_info["gpus"] = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            system_info["gpus"].append({
                "id": i,
                "name": props.name,
                "total_memory_gb": round(props.total_memory / (1024**3), 2),
            })

if psutil is not None:
    mem = psutil.virtual_memory()
    system_info["ram_total_gb"] = round(mem.total / (1024**3), 2)
    system_info["ram_available_gb"] = round(mem.available / (1024**3), 2)

system_info_path = LOG_DIR / "system_info_week1.json"
with open(system_info_path, "w", encoding="utf-8") as f:
    json.dump(system_info, f, ensure_ascii=False, indent=2)

print(json.dumps(system_info, ensure_ascii=False, indent=2))
print("Saved:", system_info_path)

In [ ]:
# ============================================================
# 2. FIND KITTI TRACKING DATASET
# ============================================================

def find_kitti_tracking_root(search_root=INPUT_ROOT):
    image_dirs = list(Path(search_root).rglob("image_02"))
    candidates = []
    for image_dir in image_dirs:
        root = image_dir.parent
        score = int((root / "label_02").exists()) + int((root / "calib").exists()) + int((root / "oxts").exists())
        candidates.append((score, root))
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
    return candidates[0][1]

KITTI_ROOT = find_kitti_tracking_root()

if KITTI_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy KITTI Tracking dataset trong /kaggle/input. "
        "Hãy add dataset có image_02, label_02, calib vào notebook."
    )

IMAGE_DIR = KITTI_ROOT / "image_02"
LABEL_DIR = KITTI_ROOT / "label_02"
CALIB_DIR = KITTI_ROOT / "calib"
OXT_DIR = KITTI_ROOT / "oxts"

print("KITTI_ROOT:", KITTI_ROOT)
print("IMAGE_DIR :", IMAGE_DIR, IMAGE_DIR.exists())
print("LABEL_DIR :", LABEL_DIR, LABEL_DIR.exists())
print("CALIB_DIR :", CALIB_DIR, CALIB_DIR.exists())
print("OXT_DIR   :", OXT_DIR, OXT_DIR.exists())

available_sequences = sorted([p.name for p in IMAGE_DIR.iterdir() if p.is_dir()])
SELECTED_SEQUENCES = [s for s in SELECTED_SEQUENCES if s in available_sequences]
if len(SELECTED_SEQUENCES) == 0:
    SELECTED_SEQUENCES = available_sequences[:2]

dataset_rows = []
for seq in SELECTED_SEQUENCES:
    imgs = sorted((IMAGE_DIR / seq).glob("*.png"))
    dataset_rows.append({
        "sequence_id": seq,
        "num_images_total": len(imgs),
        "max_frames_used": MAX_FRAMES_PER_SEQ if MAX_FRAMES_PER_SEQ is not None else len(imgs),
        "has_label": (LABEL_DIR / f"{seq}.txt").exists(),
        "has_calib": (CALIB_DIR / f"{seq}.txt").exists(),
        "image_dir": str(IMAGE_DIR / seq),
        "label_file": str(LABEL_DIR / f"{seq}.txt"),
        "calib_file": str(CALIB_DIR / f"{seq}.txt"),
    })

dataset_summary_df = pd.DataFrame(dataset_rows)
dataset_summary_path = TABLE_DIR / "kitti_dataset_summary_week1.csv"
dataset_summary_df.to_csv(dataset_summary_path, index=False)

print("Selected sequences:", SELECTED_SEQUENCES)
print("Saved:", dataset_summary_path)
display(dataset_summary_df)

In [ ]:
# ============================================================
# 3. COMMON HELPERS
# ============================================================

def normalize_class_name(name):
    name = str(name).lower()
    if name in ["pedestrian", "person_sitting"]:
        return "person"
    if name in ["cyclist", "bicycle", "bike"]:
        return "bicycle"
    if name in ["van"]:
        return "car"
    if name in ["tram"]:
        return "bus"
    if name in ["motorbike", "motorcycle"]:
        return "motorcycle"
    return name

def get_image_files(seq_id, max_frames=None):
    files = sorted((IMAGE_DIR / str(seq_id).zfill(4)).glob("*.png"))
    if max_frames is not None:
        files = files[:int(max_frames)]
    return files

def parse_kitti_label_file(label_path):
    columns = [
        "frame", "track_id", "type", "truncated", "occluded", "alpha",
        "x1", "y1", "x2", "y2",
        "h", "w", "l", "loc_x", "loc_y", "loc_z", "rot_y"
    ]
    rows = []
    label_path = Path(label_path)
    if not label_path.exists():
        return pd.DataFrame(columns=columns)

    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 17:
                continue
            rows.append({
                "frame": int(parts[0]),
                "track_id": int(parts[1]),
                "type": parts[2],
                "truncated": float(parts[3]),
                "occluded": int(float(parts[4])),
                "alpha": float(parts[5]),
                "x1": float(parts[6]),
                "y1": float(parts[7]),
                "x2": float(parts[8]),
                "y2": float(parts[9]),
                "h": float(parts[10]),
                "w": float(parts[11]),
                "l": float(parts[12]),
                "loc_x": float(parts[13]),
                "loc_y": float(parts[14]),
                "loc_z": float(parts[15]),
                "rot_y": float(parts[16]),
            })
    return pd.DataFrame(rows, columns=columns)

def read_kitti_calib(calib_path):
    calib_path = Path(calib_path)
    calib = {}
    if not calib_path.exists():
        return {"fx": DEFAULT_FOCAL_PX, "fy": DEFAULT_FOCAL_PX, "cx": DEFAULT_CX, "cy": DEFAULT_CY, "P2": None}

    with open(calib_path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" not in line:
                continue
            key, val = line.split(":", 1)
            calib[key] = [float(x) for x in val.strip().split()]

    P2 = calib.get("P2", None)
    if P2 is not None and len(P2) == 12:
        P2m = np.array(P2).reshape(3, 4)
        return {"fx": float(P2m[0, 0]), "fy": float(P2m[1, 1]), "cx": float(P2m[0, 2]), "cy": float(P2m[1, 2]), "P2": P2m}
    return {"fx": DEFAULT_FOCAL_PX, "fy": DEFAULT_FOCAL_PX, "cx": DEFAULT_CX, "cy": DEFAULT_CY, "P2": None}

def bbox_iou_matrix(gt_boxes, pred_boxes):
    gt_boxes = np.asarray(gt_boxes, dtype=float)
    pred_boxes = np.asarray(pred_boxes, dtype=float)
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return np.zeros((len(gt_boxes), len(pred_boxes)), dtype=float)

    gt = gt_boxes[:, None, :]
    pr = pred_boxes[None, :, :]

    ix1 = np.maximum(gt[..., 0], pr[..., 0])
    iy1 = np.maximum(gt[..., 1], pr[..., 1])
    ix2 = np.minimum(gt[..., 2], pr[..., 2])
    iy2 = np.minimum(gt[..., 3], pr[..., 3])

    iw = np.maximum(ix2 - ix1, 0)
    ih = np.maximum(iy2 - iy1, 0)
    inter = iw * ih

    gt_area = np.maximum(gt[..., 2] - gt[..., 0], 0) * np.maximum(gt[..., 3] - gt[..., 1], 0)
    pr_area = np.maximum(pr[..., 2] - pr[..., 0], 0) * np.maximum(pr[..., 3] - pr[..., 1], 0)
    union = gt_area + pr_area - inter
    return np.where(union > 0, inter / union, 0)

def estimate_depth_from_bbox(row, calib):
    cls = normalize_class_name(row.get("class_name", row.get("pred_class", row.get("type", "unknown"))))
    h_real = CLASS_HEIGHT_M.get(cls, CLASS_HEIGHT_M["unknown"])
    h_pix = max(float(row["y2"]) - float(row["y1"]), 1.0)
    return float(calib["fy"] * h_real / h_pix)

def estimate_x_from_bbox(row, z, calib):
    cx_bbox = (float(row["x1"]) + float(row["x2"])) / 2.0
    return float((cx_bbox - calib["cx"]) * z / max(calib["fx"], 1e-6))

def savgol_or_rolling(values, window=7, polyorder=2):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return values
    w = min(window, n if n % 2 == 1 else n - 1)
    if w < 3:
        return pd.Series(values).rolling(3, center=True, min_periods=1).median().values
    if scipy_signal is not None:
        try:
            return scipy_signal.savgol_filter(values, window_length=w, polyorder=min(polyorder, w - 1), mode="interp")
        except Exception:
            pass
    return pd.Series(values).rolling(w, center=True, min_periods=1).median().values

print("Helpers ready.")

In [ ]:
# ============================================================
# 4. MEMBER B N2 — YOLO + BYTETRACK + MOT CSV + DEMO VIDEO
# ============================================================

def run_yolo_bytetrack_on_sequence(seq_id):
    seq_id = str(seq_id).zfill(4)
    image_files = get_image_files(seq_id, max_frames=MAX_FRAMES_PER_SEQ)
    if len(image_files) == 0:
        print("[WARN] No images for sequence", seq_id)
        return pd.DataFrame(), None

    model = YOLO(YOLO_WEIGHTS)

    device = 0
    if torch is not None and not torch.cuda.is_available():
        device = "cpu"

    rows = []
    start_time = time.perf_counter()

    writer = None
    video_path = VIDEO_DIR / f"demo_tracking_seq_{seq_id}.mp4"

    if cv2 is not None:
        first_img = cv2.imread(str(image_files[0]))
        if first_img is not None:
            h_img, w_img = first_img.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(video_path), fourcc, KITTI_FPS, (w_img, h_img))

    for img_path in image_files:
        frame_id = int(Path(img_path).stem)

        result = model.track(
            source=str(img_path),
            persist=True,
            tracker=TRACKER_CFG,
            conf=CONF_THRES,
            iou=IOU_THRES,
            imgsz=IMG_SIZE,
            device=device,
            verbose=False,
        )[0]

        annotated = None
        if writer is not None:
            frame_img = cv2.imread(str(img_path))
            if frame_img is not None:
                annotated = frame_img.copy()

        boxes = result.boxes
        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.detach().cpu().numpy()
            confs = boxes.conf.detach().cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy))
            clss = boxes.cls.detach().cpu().numpy().astype(int) if boxes.cls is not None else np.full(len(xyxy), -1)
            ids = boxes.id.detach().cpu().numpy().astype(int) if boxes.id is not None else np.full(len(xyxy), -1)

            for j, box in enumerate(xyxy):
                class_id = int(clss[j])
                class_name = str(model.names.get(class_id, class_id)).lower()
                if class_name not in COCO_TRAFFIC_CLASSES:
                    continue

                x1, y1, x2, y2 = [float(v) for v in box]
                score = float(confs[j])
                track_id = int(ids[j])
                w = x2 - x1
                h = y2 - y1

                rows.append({
                    "sequence_id": seq_id,
                    "frame": frame_id,
                    "track_id": track_id,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "w": w,
                    "h": h,
                    "score": score,
                    "class_id": class_id,
                    "class_name": class_name,
                })

                if annotated is not None and track_id >= 0:
                    cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                    cv2.putText(
                        annotated,
                        f"{class_name} ID:{track_id} {score:.2f}",
                        (int(x1), max(int(y1) - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.45,
                        (0, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )

        if writer is not None and annotated is not None:
            writer.write(annotated)

    if writer is not None:
        writer.release()

    elapsed = time.perf_counter() - start_time
    fps = len(image_files) / elapsed if elapsed > 0 else np.nan

    pred_df = pd.DataFrame(rows)
    pred_csv = PRED_DIR / f"pred_mot_seq_{seq_id}.csv"
    pred_df.to_csv(pred_csv, index=False)

    mot_export = pred_df[["frame", "track_id", "x1", "y1", "w", "h", "score", "class_id", "class_name"]].copy() if len(pred_df) else pd.DataFrame(
        columns=["frame", "track_id", "x1", "y1", "w", "h", "score", "class_id", "class_name"]
    )
    mot_csv = PRED_DIR / f"mot_format_seq_{seq_id}.csv"
    mot_export.to_csv(mot_csv, index=False)

    runtime = {
        "sequence_id": seq_id,
        "num_frames": len(image_files),
        "num_prediction_rows": len(pred_df),
        "runtime_sec": elapsed,
        "fps": fps,
        "pred_csv": str(pred_csv),
        "mot_csv": str(mot_csv),
        "demo_video": str(video_path) if video_path.exists() else "",
    }

    print(f"seq={seq_id} frames={len(image_files)} pred_rows={len(pred_df)} fps={fps:.2f}")
    return pred_df, runtime

all_pred_dfs = []
runtime_rows = []

for seq_id in SELECTED_SEQUENCES:
    pred_seq, rt = run_yolo_bytetrack_on_sequence(seq_id)
    if len(pred_seq):
        all_pred_dfs.append(pred_seq)
    if rt is not None:
        runtime_rows.append(rt)

pred_all = pd.concat(all_pred_dfs, ignore_index=True) if all_pred_dfs else pd.DataFrame()
pred_all_path = PRED_DIR / "pred_mot_all_sequences.csv"
pred_all.to_csv(pred_all_path, index=False)

runtime_df = pd.DataFrame(runtime_rows)
runtime_path = TABLE_DIR / "tracking_runtime_week1.csv"
runtime_df.to_csv(runtime_path, index=False)

print("Saved:", pred_all_path, "rows:", len(pred_all))
print("Saved:", runtime_path)
display(runtime_df)
display(pred_all.head())

In [ ]:
# ============================================================
# 5. MEMBER B N3 — TABLE 1 TRACKING DRAFT
# ============================================================

def load_gt_for_sequence(seq_id):
    seq_id = str(seq_id).zfill(4)
    gt = parse_kitti_label_file(LABEL_DIR / f"{seq_id}.txt")
    if len(gt) == 0:
        return gt
    gt["sequence_id"] = seq_id
    gt["class_name"] = gt["type"].apply(normalize_class_name)
    gt = gt[gt["track_id"] >= 0].copy()
    gt = gt[gt["type"].str.lower() != "dontcare"].copy()
    gt = gt[gt["type"].str.lower().isin(KITTI_TRAFFIC_CLASSES)].copy()
    return gt

def evaluate_tracking_sequence(seq_id, pred_df, iou_thr=0.5):
    seq_id = str(seq_id).zfill(4)
    gt = load_gt_for_sequence(seq_id)
    pred = pred_df[pred_df["sequence_id"].astype(str).str.zfill(4) == seq_id].copy()

    gt_before = len(gt)
    if FILTER_GT_TO_EVAL_RANGE and len(pred):
        min_f, max_f = int(pred["frame"].min()), int(pred["frame"].max())
        gt = gt[(gt["frame"] >= min_f) & (gt["frame"] <= max_f)].copy()

    frames = sorted(set(gt["frame"].unique()).union(set(pred["frame"].unique())))

    if mm is not None:
        acc = mm.MOTAccumulator(auto_id=True)

        for fr in frames:
            gtf = gt[gt["frame"] == fr].copy()
            prf = pred[pred["frame"] == fr].copy()

            gt_ids = gtf["track_id"].astype(int).tolist()
            pr_ids = prf["track_id"].astype(int).tolist()

            gt_boxes = gtf[["x1", "y1", "x2", "y2"]].values
            pr_boxes = prf[["x1", "y1", "x2", "y2"]].values
            ious = bbox_iou_matrix(gt_boxes, pr_boxes)
            dists = 1.0 - ious
            dists[ious < iou_thr] = np.nan

            acc.update(gt_ids, pr_ids, dists)

        mh = mm.metrics.create()
        summary = mh.compute(
            acc,
            metrics=["mota", "idf1", "num_false_positives", "num_misses", "num_switches", "num_objects", "num_predictions"],
            name=seq_id,
        )
        row = summary.loc[seq_id].to_dict()
        return {
            "sequence_id": seq_id,
            "MOTA": float(row.get("mota", np.nan)),
            "IDF1": float(row.get("idf1", np.nan)),
            "HOTA_official": np.nan,
            "FP": int(row.get("num_false_positives", 0)),
            "FN": int(row.get("num_misses", 0)),
            "IDS": int(row.get("num_switches", 0)),
            "gt_objects_before_frame_filter": gt_before,
            "gt_objects_after_frame_filter": len(gt),
            "pred_objects": len(pred),
            "num_frames": len(frames),
            "status": "ok_hota_not_computed",
        }

    # fallback no motmetrics
    fp = fn = ids = 0
    prev_match = {}
    for fr in frames:
        gtf = gt[gt["frame"] == fr].reset_index(drop=True)
        prf = pred[pred["frame"] == fr].reset_index(drop=True)
        ious = bbox_iou_matrix(gtf[["x1", "y1", "x2", "y2"]].values, prf[["x1", "y1", "x2", "y2"]].values)

        pairs = []
        for i in range(ious.shape[0]):
            for j in range(ious.shape[1]):
                if ious[i, j] >= iou_thr:
                    pairs.append((ious[i, j], i, j))
        pairs.sort(reverse=True)

        used_g, used_p = set(), set()
        for _, i, j in pairs:
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)
            gt_id = int(gtf.iloc[i]["track_id"])
            pr_id = int(prf.iloc[j]["track_id"])
            if gt_id in prev_match and prev_match[gt_id] != pr_id:
                ids += 1
            prev_match[gt_id] = pr_id

        fn += len(gtf) - len(used_g)
        fp += len(prf) - len(used_p)

    mota = 1.0 - (fp + fn + ids) / max(len(gt), 1)
    return {
        "sequence_id": seq_id,
        "MOTA": mota,
        "IDF1": np.nan,
        "HOTA_official": np.nan,
        "FP": fp,
        "FN": fn,
        "IDS": ids,
        "gt_objects_before_frame_filter": gt_before,
        "gt_objects_after_frame_filter": len(gt),
        "pred_objects": len(pred),
        "num_frames": len(frames),
        "status": "fallback_no_motmetrics",
    }

table1_rows = []
for seq_id in SELECTED_SEQUENCES:
    row = evaluate_tracking_sequence(seq_id, pred_all, iou_thr=EVAL_IOU_THRESHOLD)
    rtf = runtime_df[runtime_df["sequence_id"].astype(str).str.zfill(4) == str(seq_id).zfill(4)]
    if len(rtf):
        row["FPS"] = float(rtf.iloc[0]["fps"])
        row["runtime_sec"] = float(rtf.iloc[0]["runtime_sec"])
    row["model"] = YOLO_WEIGHTS
    row["tracker"] = "ByteTrack"
    row["conf_threshold"] = CONF_THRES
    table1_rows.append(row)

table1_df = pd.DataFrame(table1_rows)
table1_path = TABLE_DIR / "table1_tracking_draft.csv"
table1_df.to_csv(table1_path, index=False)

print("Saved:", table1_path)
display(table1_df)

In [ ]:
# ==============================================================
# HOTA PATCH — Thêm vào notebook SAU cell tạo table1_df
# Dán toàn bộ cell này vào ngay sau dòng:
#   table1_df.to_csv(table1_path, index=False)
#
# KHÔNG cần thay đổi bất kỳ cell nào khác.
# KHÔNG cần cài TrackEval hay thư viện ngoài.
# Chỉ dùng scipy.optimize.linear_sum_assignment (có sẵn trên Kaggle).
#
# Nguồn tham khảo:
#   Luiten et al. (2021). HOTA: A Higher Order Metric for
#   Evaluating Multi-Object Tracking. IJCV.
#   https://arxiv.org/abs/2009.14894
# ==============================================================

from scipy.optimize import linear_sum_assignment
import numpy as np
import warnings

# ─────────────────────────────────────────────────────────────
# PHẦN 1: Hàm compute_hota_sequence
# ─────────────────────────────────────────────────────────────

def compute_hota_sequence(seq_id, pred_df, gt_override=None,
                          alpha_range=None):
    """
    Tính HOTA, DetA, AssA cho một sequence, trung bình trên
    các ngưỡng IoU alpha ∈ {0.05, 0.10, ..., 0.95} (19 giá trị).

    Định nghĩa cốt lõi (Luiten et al., 2021):
        HOTA_α  = sqrt(DetA_α × AssA_α)
        DetA_α  = TP_α / (TP_α + FP_α + FN_α)   [Jaccard detection]
        AssA_α  = (1/TP_α) Σ_{c∈TP} A_α(c)       [weighted mean assoc.]

    Với A_α(c) cho cặp (gt_id, pred_id) tại frame f:
        TPA(c) = số frame mà gt_id và pred_id được match với nhau
        FPA(c) = số frame mà pred_id match với GT khác
        FNA(c) = số frame mà gt_id match với pred khác
        A_α(c) = TPA(c) / (TPA(c) + FPA(c) + FNA(c))

    Args:
        seq_id     : string sequence ID (e.g. "0000")
        pred_df    : DataFrame pred_all (có sẵn trong notebook)
        gt_override: DataFrame GT nếu đã load sẵn (None = load từ KITTI)
        alpha_range: mảng threshold IoU (default: 0.05..0.95 step 0.05)

    Returns:
        dict với HOTA, DetA, AssA (float), và per-alpha lists
    """

    if alpha_range is None:
        alpha_range = np.round(np.arange(0.05, 0.96, 0.05), 2)  # 19 thresholds

    seq_str = str(seq_id).zfill(4)

    # Load GT
    if gt_override is not None:
        gt = gt_override.copy()
    else:
        try:
            gt = load_gt_for_sequence(seq_str)          # dùng hàm có sẵn trong notebook
        except Exception as e:
            warnings.warn(f"[HOTA] Không load được GT seq {seq_str}: {e}")
            return {"HOTA": np.nan, "DetA": np.nan, "AssA": np.nan,
                    "status": f"gt_load_error: {e}"}

    if len(gt) == 0:
        return {"HOTA": np.nan, "DetA": np.nan, "AssA": np.nan,
                "status": "gt_empty"}

    pred = pred_df[pred_df["sequence_id"].astype(str).str.zfill(4) == seq_str].copy()

    all_frames = sorted(
        set(gt["frame"].unique()) | set(pred["frame"].unique())
    )

    # Precompute IoU matrices một lần cho tất cả frames
    # Tránh lặp bbox_iou_matrix cho từng alpha threshold
    iou_cache: dict = {}
    for f in all_frames:
        gtf  = gt[gt["frame"] == f]
        prf  = pred[pred["frame"] == f]
        if len(gtf) == 0 or len(prf) == 0:
            iou_cache[f] = None
            continue
        gt_boxes = gtf[["x1", "y1", "x2", "y2"]].values.astype(float)
        pr_boxes = prf[["x1", "y1", "x2", "y2"]].values.astype(float)
        iou_cache[f] = (
            gtf["track_id"].astype(int).values,   # gt IDs
            prf["track_id"].astype(int).values,   # pred IDs
            bbox_iou_matrix(gt_boxes, pr_boxes),  # dùng hàm có sẵn
        )

    n_gt_per_frame   = {f: len(gt[gt["frame"] == f])   for f in all_frames}
    n_pred_per_frame = {f: len(pred[pred["frame"] == f]) for f in all_frames}

    # ── Sweep qua từng alpha ──────────────────────────────────
    hota_per_alpha = []
    deta_per_alpha = []
    assa_per_alpha = []

    for alpha in alpha_range:
        tp_total = 0
        fp_total = 0
        fn_total = 0

        # Association bookkeeping
        match_counts   = {}   # (gt_id, pred_id) -> số frame được match
        gt_tp_total    = {}   # gt_id   -> tổng TP frame (với bất kỳ pred)
        pred_tp_total  = {}   # pred_id -> tổng TP frame (với bất kỳ gt)

        for f in all_frames:
            n_gt   = n_gt_per_frame[f]
            n_pred = n_pred_per_frame[f]

            # Xử lý trường hợp không có detections
            if n_gt == 0 and n_pred == 0:
                continue
            if n_gt == 0:
                fp_total += n_pred
                continue
            if n_pred == 0:
                fn_total += n_gt
                continue

            cache = iou_cache[f]
            if cache is None:           # fallback an toàn
                fn_total += n_gt
                fp_total += n_pred
                continue

            gt_ids, pred_ids, iou_mat = cache

            # Hungarian assignment tối đa hoá IoU
            row_ind, col_ind = linear_sum_assignment(-iou_mat)

            matched_gt_idx   = set()
            matched_pred_idx = set()

            for r, c in zip(row_ind, col_ind):
                if iou_mat[r, c] >= alpha:
                    tp_total += 1
                    matched_gt_idx.add(r)
                    matched_pred_idx.add(c)

                    gt_id   = int(gt_ids[r])
                    pred_id = int(pred_ids[c])
                    key = (gt_id, pred_id)

                    match_counts[key]          = match_counts.get(key, 0) + 1
                    gt_tp_total[gt_id]         = gt_tp_total.get(gt_id, 0) + 1
                    pred_tp_total[pred_id]     = pred_tp_total.get(pred_id, 0) + 1

            fn_total += n_gt   - len(matched_gt_idx)
            fp_total += n_pred - len(matched_pred_idx)

        # ── DetA ─────────────────────────────────────────────
        denom_det = tp_total + fp_total + fn_total
        deta = tp_total / denom_det if denom_det > 0 else 0.0

        # ── AssA ─────────────────────────────────────────────
        if tp_total == 0:
            assa = 0.0
        else:
            assa_num = 0.0
            for (gt_id, pred_id), count in match_counts.items():
                # TPA(c) = count
                # FNA(c) = gt_tp_total[gt_id]   - count
                # FPA(c) = pred_tp_total[pred_id] - count
                union = gt_tp_total[gt_id] + pred_tp_total[pred_id] - count
                a_c = count / union if union > 0 else 0.0
                assa_num += a_c * count      # mỗi TP frame đóng góp như nhau
            assa = assa_num / tp_total

        # ── HOTA_α ───────────────────────────────────────────
        hota_alpha = float(np.sqrt(deta * assa)) if (deta > 0 and assa > 0) else 0.0

        hota_per_alpha.append(hota_alpha)
        deta_per_alpha.append(float(deta))
        assa_per_alpha.append(float(assa))

    return {
        "HOTA":             float(np.mean(hota_per_alpha)),
        "DetA":             float(np.mean(deta_per_alpha)),
        "AssA":             float(np.mean(assa_per_alpha)),
        "HOTA_per_alpha":   hota_per_alpha,
        "DetA_per_alpha":   deta_per_alpha,
        "AssA_per_alpha":   assa_per_alpha,
        "alpha_range":      list(alpha_range),
        "status":           "ok",
    }


# ─────────────────────────────────────────────────────────────
# PHẦN 2: Chạy HOTA cho tất cả sequences và ghép vào table1_df
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("Đang tính HOTA cho từng sequence (19 alpha thresholds × N frames)...")
print("Ước tính: ~30-90 giây tùy GPU/CPU và số frame mỗi sequence.")
print("=" * 60)

hota_rows = []
for seq_id in SELECTED_SEQUENCES:
    print(f"  [HOTA] Sequence {seq_id} ...", end=" ", flush=True)
    try:
        result = compute_hota_sequence(seq_id, pred_all)
        hota_rows.append({
            "sequence_id": str(seq_id).zfill(4),
            "HOTA": result["HOTA"],
            "DetA": result["DetA"],
            "AssA": result["AssA"],
            "hota_status": result.get("status", "unknown"),
        })
        print(f"HOTA={result['HOTA']:.4f}  DetA={result['DetA']:.4f}  AssA={result['AssA']:.4f}")
    except Exception as e:
        print(f"LỖI: {e}")
        hota_rows.append({
            "sequence_id": str(seq_id).zfill(4),
            "HOTA": np.nan,
            "DetA": np.nan,
            "AssA": np.nan,
            "hota_status": f"error: {e}",
        })

hota_df = pd.DataFrame(hota_rows)

# Ghép HOTA vào table1_df
# Xoá cột HOTA_official cũ (= NaN) nếu có, thay bằng giá trị tính được
table1_df["sequence_id"] = table1_df["sequence_id"].astype(str).str.zfill(4)

if "HOTA_official" in table1_df.columns:
    table1_df = table1_df.drop(columns=["HOTA_official"])

table1_df = table1_df.merge(
    hota_df[["sequence_id", "HOTA", "DetA", "AssA", "hota_status"]],
    on="sequence_id",
    how="left",
)

# Sắp xếp lại cột cho đẹp: đặt HOTA ngay sau IDF1
cols = list(table1_df.columns)
for col in ["HOTA", "DetA", "AssA"]:
    if col in cols:
        cols.remove(col)

# Chèn HOTA, DetA, AssA sau IDF1
if "IDF1" in cols:
    idx = cols.index("IDF1") + 1
    cols[idx:idx] = [c for c in ["HOTA", "DetA", "AssA"] if c in table1_df.columns]
else:
    cols += [c for c in ["HOTA", "DetA", "AssA"] if c in table1_df.columns]

table1_df = table1_df[cols]

# Lưu lại CSV (overwrite file cũ + tạo file mới với HOTA)
table1_path_hota = TABLE_DIR / "table1_tracking_with_hota.csv"
table1_df.to_csv(table1_path_hota, index=False)
table1_df.to_csv(TABLE_DIR / "table1_tracking_draft.csv", index=False)   # overwrite file gốc

print("\n" + "=" * 60)
print("KẾT QUẢ Table 1 — Tracking Metrics (với HOTA):")
print("=" * 60)

# Hiện bảng gọn
display_cols = ["sequence_id", "MOTA", "IDF1", "HOTA", "DetA", "AssA", "FP", "FN", "IDS"]
display_cols = [c for c in display_cols if c in table1_df.columns]
display(table1_df[display_cols].style.format({
    "MOTA":  "{:.4f}",
    "IDF1":  "{:.4f}",
    "HOTA":  "{:.4f}",
    "DetA":  "{:.4f}",
    "AssA":  "{:.4f}",
    "FP":    "{:.0f}",
    "FN":    "{:.0f}",
    "IDS":   "{:.0f}",
}).highlight_max(subset=["MOTA","IDF1","HOTA"], color="lightgreen")
  .highlight_min(subset=["FP","FN","IDS"],      color="lightyellow"))

# Summary statistics
print("\n── Mean / Std trên tất cả sequences ──")
for col in ["MOTA", "IDF1", "HOTA", "DetA", "AssA"]:
    if col in table1_df.columns:
        vals = table1_df[col].dropna()
        print(f"  {col:6s}: mean={vals.mean():.4f}  std={vals.std():.4f}  "
              f"min={vals.min():.4f}  max={vals.max():.4f}")

print(f"\nĐã lưu: {table1_path_hota}")
print("Cột HOTA_official (N/A) đã được thay bằng HOTA, DetA, AssA thực.")


# HOTA THẬT bằng official TrackEval (thay cho bản tự viết ở cell trên)

**Lý do thay:** Comment review yêu cầu rõ dùng *công cụ TrackEval*
(github.com/JonathonLuiten/TrackEval), không phải code tự viết lại công thức.
Cell trên (`compute_hota_sequence` tự viết bằng scipy) **vẫn được giữ lại
không xoá** — dùng để đối chiếu (sanity check) với kết quả chính thức ở dưới.

**Khác biệt quan trọng cần biết trước khi chạy:**
TrackEval bản KITTI 2D Box (`Kitti2DBox`) chỉ đánh giá **2 lớp riêng biệt**:
`car` và `pedestrian` (giới hạn cứng của chính công cụ). MOTA/IDF1 ở Table 1
hiện tại đang gộp CHUNG tất cả lớp giao thông thành một bài toán tracking duy
nhất. Nghĩa là: HOTA chính thức (cell này) ra 2 số mỗi sequence
(HOTA_car, HOTA_pedestrian). Bản "toàn bộ class gộp" nằm ở cell phía dưới.

**Lỗi numpy đã gặp và đã sửa:** TrackEval viết từ thời NumPy <1.20, dùng
`np.float`/`np.int`/`np.bool` — các alias này đã bị NumPy xoá hẳn ở bản mới
(Kaggle hiện cài NumPy 1.24+) gây lỗi `AttributeError: module 'numpy' has no
attribute 'float'`. Cell dưới có patch lại alias này — patch này không sửa
1 dòng nào trong code gốc TrackEval, chỉ phục hồi tên gọi cũ mà bản thân
NumPy xác nhận là tương đương 100%, an toàn (xem chú thích deprecation chính
thức của NumPy). Đã tự kiểm tra patch này trên 1 ví dụ KITTI giả lập (3 frame,
2 object) chạy qua đúng TrackEval thật — ra HOTA=1.0 cho Car khi box gần như
trùng khớp hoàn hảo, xác nhận patch không che giấu lỗi nào khác.

**Cách chạy:** Bật **Internet ON** trong Kaggle Notebook Settings, chạy theo
thứ tự.


In [ ]:
# ==============================================================
# HOTA THẬT — PHẦN 1: Clone TrackEval thật + export GT/predictions thật
# ==============================================================
# KHÔNG tự viết lại công thức HOTA. Toàn bộ phép tính HOTA nằm trong code
# gốc của TrackEval (github.com/JonathonLuiten/TrackEval), mình chỉ chuẩn bị
# dữ liệu đúng format để nó đọc. Input (GT thật từ LABEL_DIR, predictions thật
# từ pred_all) — không sinh số giả ở bất kỳ bước nào.

import sys, subprocess, shutil
from pathlib import Path
import numpy as np

TRACKEVAL_DIR = Path("/kaggle/working/TrackEval")

if not TRACKEVAL_DIR.exists():
    print("Đang clone TrackEval thật từ GitHub (cần Internet ON trong Settings)...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/JonathonLuiten/TrackEval.git",
        str(TRACKEVAL_DIR),
    ])
else:
    print("TrackEval đã có sẵn tại:", TRACKEVAL_DIR)

if str(TRACKEVAL_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKEVAL_DIR))

import trackeval  # nếu lỗi ở đây -> dừng luôn, KHÔNG fallback sang code tự viết
print("Import trackeval OK, module path:", trackeval.__file__)

# ----------------------------------------------------------------
# FIX tương thích NumPy mới: TrackEval viết từ thời np.float/np.int/np.bool
# còn tồn tại (NumPy <1.20). Bản NumPy mới trên Kaggle đã xoá hẳn các alias
# này -> AttributeError khi chạy evaluate(). Patch lại alias (CHỈ patch SAU
# khi import trackeval xong, để không phá phần khởi tạo nội bộ của
# scipy/numpy.ma). Đây không phải sửa công thức/logic - np.float vốn luôn
# là tên gọi khác của float, NumPy tự xác nhận việc này an toàn 100%.
# ----------------------------------------------------------------
np.float = float
np.int = int
np.bool = bool
print("Đã patch np.float/np.int/np.bool (alias cũ, an toàn) để tương thích NumPy mới")

# ----------------------------------------------------------------
# 1) Cấu trúc thư mục đúng chuẩn Kitti2DBox
#    GT_FOLDER/label_02/<seq>.txt
#    GT_FOLDER/evaluate_tracking.seqmap.training
#    TRACKERS_FOLDER/<tracker_name>/data/<seq>.txt
# ----------------------------------------------------------------
TE_ROOT      = Path("/kaggle/working/trackeval_kitti")
GT_FOL       = TE_ROOT / "gt"
GT_LABEL_FOL = GT_FOL / "label_02"
TRACKER_FOL  = TE_ROOT / "trackers"
TRACKER_NAME = "yolo11x_bytetrack_real"

GT_LABEL_FOL.mkdir(parents=True, exist_ok=True)
(TRACKER_FOL / TRACKER_NAME / "data").mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# 2) Copy NGUYÊN VĂN file GT thật (không sửa 1 ký tự) sang đúng cấu trúc
# ----------------------------------------------------------------
for seq_id in SELECTED_SEQUENCES:
    src_label = LABEL_DIR / f"{seq_id}.txt"
    dst_label = GT_LABEL_FOL / f"{seq_id}.txt"
    if not src_label.exists():
        raise FileNotFoundError(f"Không tìm thấy GT thật cho sequence {seq_id}: {src_label}")
    shutil.copyfile(src_label, dst_label)
print(f"Đã copy {len(SELECTED_SEQUENCES)} file GT thật sang {GT_LABEL_FOL}")

# ----------------------------------------------------------------
# 3) Export predictions thật (pred_all) sang format KITTI submission
#    (17 cột chuẩn KITTI + cột 18 = confidence score thật từ YOLO)
#    Chỉ 'car' và 'pedestrian' được Kitti2DBox đánh giá (giới hạn của
#    chính TrackEval, valid_classes=['car','pedestrian']) -> các lớp khác
#    (bicycle, motorcycle, bus, truck) bị loại, KHÔNG gán nhầm sang Car/Ped.
# ----------------------------------------------------------------
COCO_TO_KITTI_TYPE = {
    "car": "Car",
    "person": "Pedestrian",
}

def export_pred_to_kitti_format(seq_id, pred_df, out_path):
    seq_id = str(seq_id).zfill(4)
    seq_pred = pred_df[pred_df["sequence_id"].astype(str).str.zfill(4) == seq_id].copy()
    lines = []
    for _, row in seq_pred.iterrows():
        if int(row["track_id"]) < 0:
            continue  # bỏ track chưa được gán ID hợp lệ (id=-1)
        kitti_type = COCO_TO_KITTI_TYPE.get(str(row["class_name"]).lower())
        if kitti_type is None:
            continue  # không phải car/pedestrian -> không nộp dòng này
        fields = [
            int(row["frame"]), int(row["track_id"]), kitti_type,
            -1, -1, -10.0,
            float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"]),
            -1, -1, -1, -1000, -1000, -1000, -10.0,
            float(row["score"]),
        ]
        lines.append(" ".join(str(x) for x in fields))
    with open(out_path, "w") as f:
        f.write(chr(10).join(lines) + (chr(10) if lines else ""))
    return len(lines)

print()
print("Đang export predictions thật sang format KITTI submission...")
total_lines = 0
for seq_id in SELECTED_SEQUENCES:
    out_path = TRACKER_FOL / TRACKER_NAME / "data" / f"{seq_id}.txt"
    n = export_pred_to_kitti_format(seq_id, pred_all, out_path)
    total_lines += n
    print(f"  seq {seq_id}: {n} dòng (car+pedestrian, id>=0) đã export -> {out_path.name}")
print("Tổng số dòng prediction export:", total_lines)

# ----------------------------------------------------------------
# 4) Seqmap — số frame lấy từ chính số ảnh thật trong KITTI image dir
# ----------------------------------------------------------------
seqmap_path = GT_FOL / "evaluate_tracking.seqmap.training"
with open(seqmap_path, "w") as f:
    for seq_id in SELECTED_SEQUENCES:
        n_frames_real = len(get_image_files(seq_id))
        f.write(f"{seq_id} empty 000000 {n_frames_real:06d}" + chr(10))
        print(f"  seqmap {seq_id}: {n_frames_real} frame thật")
print("Đã ghi seqmap:", seqmap_path)

# ----------------------------------------------------------------
# 5) Preview nhanh để tự kiểm tra format trước khi chạy evaluator
# ----------------------------------------------------------------
print()
print("--- Preview 3 dòng đầu GT seq 0000 (copy nguyên văn) ---")
with open(GT_LABEL_FOL / "0000.txt") as f:
    for _ in range(3):
        print(" ", f.readline().strip())

print()
print("--- Preview 3 dòng đầu prediction seq 0000 (vừa export) ---")
with open(TRACKER_FOL / TRACKER_NAME / "data" / "0000.txt") as f:
    for _ in range(3):
        line = f.readline().strip()
        if line:
            print(" ", line)

print()
print("--- Preview seqmap ---")
with open(seqmap_path) as f:
    print(f.read())


In [ ]:
# ==============================================================
# HOTA THẬT — PHẦN 2: Gọi trackeval.Evaluator THẬT, in kết quả gốc
# ==============================================================
# Không parse/chỉnh output_res theo hướng có lợi cho số đẹp — in toàn bộ
# cấu trúc trả về trước, rồi mới trích xuất.

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config["PRINT_RESULTS"]   = True
eval_config["PRINT_CONFIG"]    = True
eval_config["TIME_PROGRESS"]   = False
eval_config["OUTPUT_SUMMARY"]  = True
eval_config["OUTPUT_DETAILED"] = True
eval_config["PLOT_CURVES"]     = False

dataset_config = trackeval.datasets.Kitti2DBox.get_default_dataset_config()
dataset_config["GT_FOLDER"]        = str(GT_FOL)
dataset_config["TRACKERS_FOLDER"]  = str(TRACKER_FOL)
dataset_config["TRACKERS_TO_EVAL"] = [TRACKER_NAME]
dataset_config["CLASSES_TO_EVAL"]  = ["car", "pedestrian"]
dataset_config["SPLIT_TO_EVAL"]    = "training"

evaluator    = trackeval.Evaluator(eval_config)
dataset_list = [trackeval.datasets.Kitti2DBox(dataset_config)]
metrics_list = [trackeval.metrics.HOTA()]   # chỉ lấy HOTA — MOTA/IDF1 đã có từ motmetrics ở cell trước

output_res, output_msg = evaluator.evaluate(dataset_list, metrics_list)

import numpy as np

print()
print("=== RAW output_res keys (kiểm tra trước khi trích số) ===")
print("Level 1 (dataset):", list(output_res.keys()))
ds_name = list(output_res.keys())[0]
print("Level 2 (tracker):", list(output_res[ds_name].keys()))
tr_name = list(output_res[ds_name].keys())[0]
res_root = output_res[ds_name][tr_name]
print("Level 3 (sequence / COMBINED_SEQ):", list(res_root.keys()))
print("Level 4 (class, trong sequence đầu tiên):", list(res_root[SELECTED_SEQUENCES[0]].keys()))
# Cấu trúc thật đã verify: res_root[seq_id][class]['HOTA']  (SEQUENCE TRƯỚC, CLASS SAU)

real_hota_rows = []
for seq_id in SELECTED_SEQUENCES:
    if seq_id not in res_root:
        print(f"[CẢNH BÁO] Thiếu seq {seq_id} trong kết quả.")
        continue
    for cls in ["car", "pedestrian"]:
        if cls not in res_root[seq_id]:
            continue
        hota_block = res_root[seq_id][cls]["HOTA"]
        real_hota_rows.append({
            "sequence_id": seq_id,
            "class": cls,
            "HOTA_official": float(np.mean(hota_block["HOTA"])),
            "DetA_official": float(np.mean(hota_block["DetA"])),
            "AssA_official": float(np.mean(hota_block["AssA"])),
        })

for cls in ["car", "pedestrian"]:
    if "COMBINED_SEQ" in res_root and cls in res_root["COMBINED_SEQ"]:
        comb = res_root["COMBINED_SEQ"][cls]["HOTA"]
        print()
        print(f"[{cls}] COMBINED_SEQ (official, gộp toàn bộ TP/FP/FN trước khi tính — "
              f"KHÔNG phải trung bình cộng): HOTA={np.mean(comb['HOTA']):.4f}  "
              f"DetA={np.mean(comb['DetA']):.4f}  AssA={np.mean(comb['AssA']):.4f}")

real_hota_df = pd.DataFrame(real_hota_rows)
print()
print("=== HOTA THẬT theo từng sequence + class (official TrackEval, car/pedestrian riêng) ===")
display(real_hota_df)

real_hota_path = TABLE_DIR / "table1_hota_official_trackeval_carped.csv"
real_hota_df.to_csv(real_hota_path, index=False)
print("Đã lưu:", real_hota_path)


# HOTA THẬT — TOÀN BỘ CLASS GỘP (không giới hạn car/pedestrian)

**Vấn đề của cell trên:** `Kitti2DBox` (dataset adapter KITTI của TrackEval) **hard-code**
`valid_classes = ['car', 'pedestrian']` — bất kỳ class nào khác sẽ bị loại/raise exception.
Đây là giới hạn của riêng adapter KITTI (đúng cách KITTI leaderboard chính thức báo cáo:
tách riêng Car/Pedestrian), không sửa được bằng config.

**Cách giải quyết — vẫn dùng TrackEval THẬT, không viết lại công thức:**
`trackeval.metrics.HOTA` (file `hota.py`) chứa toàn bộ thuật toán HOTA thật (Hungarian
matching theo global alignment score, công thức AssA/DetA/HOTA) — không quan tâm class là gì.
Giới hạn class nằm ở tầng *dataset adapter* (`Kitti2DBox`), không nằm ở tầng *metric* (`HOTA`).
Nên mình gọi trực tiếp `trackeval.metrics.HOTA().eval_sequence(data)` — đúng method thật,
đúng code thật — nhưng tự chuẩn bị `data` (gt_ids, tracker_ids, similarity_scores mỗi frame)
từ TẤT CẢ class giao thông gộp chung (giống đúng cách MOTA/IDF1 ở cell 6 đang làm) — KHÔNG
đi qua `Kitti2DBox`.

Việc chuẩn bị `data` (đếm ID, ghép ID liên tục 0..N-1, tính IoU) là chuẩn bị dữ liệu, không
phải "công thức HOTA". Phần tính toán HOTA 100% là code gốc của TrackEval:
`eval_sequence()` và `combine_sequences()` — đã verify cả 2 hàm này chạy đúng trên ví dụ
giả lập (gt 2 track, pred 2 track, 3 frame, box gần trùng -> HOTA ra ~0.98, hợp lý).

**Lưu ý báo cáo trong paper:** Số ra từ cell này KHÔNG phải số "HOTA chuẩn KITTI leaderboard"
(leaderboard tách Car/Pedestrian riêng — xem cell trên). Đây là HOTA tính bằng core engine
thật của TrackEval, trên cùng phạm vi gộp-class như MOTA/IDF1 hiện tại. Cần ghi rõ trong
Section IV để không bị hiểu nhầm là số leaderboard chính thức.


In [ ]:
# ==============================================================
# HOTA THẬT — TOÀN BỘ CLASS GỘP — dùng trực tiếp trackeval.metrics.HOTA
# ==============================================================
# Bắt buộc đã chạy 2 cell "PHẦN 1" + "PHẦN 2" ở trên (đã có biến `trackeval`,
# đã patch np.float/np.int/np.bool). Không tạo file submission KITTI, không
# qua Kitti2DBox -> không bị chặn class.

import numpy as np

assert "trackeval" in globals(), "Chưa chạy cell clone TrackEval ở trên."
# Patch lại cho chắc (idempotent, vô hại) nếu cell này được chạy độc lập sau khi restart kernel
np.float = float
np.int = int
np.bool = bool

def build_hota_data_all_classes(seq_id, pred_df):
    # Dựng dict `data` đúng format mà trackeval.metrics.HOTA.eval_sequence() cần,
    # gộp TẤT CẢ class giao thông (giống scope của MOTA/IDF1 ở cell 6), KHÔNG qua Kitti2DBox.
    seq_id = str(seq_id).zfill(4)

    gt = load_gt_for_sequence(seq_id)  # GT thật, dùng lại hàm đã có (đã filter dontcare đúng)
    pred = pred_df[pred_df["sequence_id"].astype(str).str.zfill(4) == seq_id].copy()

    if FILTER_GT_TO_EVAL_RANGE and len(pred):
        min_f, max_f = int(pred["frame"].min()), int(pred["frame"].max())
        gt = gt[(gt["frame"] >= min_f) & (gt["frame"] <= max_f)].copy()

    frames = sorted(set(gt["frame"].unique()) | set(pred["frame"].unique()))

    unique_gt_ids = sorted(gt["track_id"].astype(int).unique().tolist())
    unique_pr_ids = sorted(pred["track_id"].astype(int).unique().tolist())
    gt_id_map = {tid: i for i, tid in enumerate(unique_gt_ids)}
    pr_id_map = {tid: i for i, tid in enumerate(unique_pr_ids)}

    gt_ids_list, tracker_ids_list, sim_list = [], [], []
    num_gt_dets = 0
    num_tracker_dets = 0

    for f in frames:
        gtf = gt[gt["frame"] == f]
        prf = pred[pred["frame"] == f]

        gt_ids_t = np.array([gt_id_map[int(t)] for t in gtf["track_id"]], dtype=int)
        pr_ids_t = np.array([pr_id_map[int(t)] for t in prf["track_id"]], dtype=int)

        gt_boxes = gtf[["x1", "y1", "x2", "y2"]].values.astype(float)
        pr_boxes = prf[["x1", "y1", "x2", "y2"]].values.astype(float)
        sim = bbox_iou_matrix(gt_boxes, pr_boxes)  # hàm IoU đã có sẵn trong notebook, không viết lại

        gt_ids_list.append(gt_ids_t)
        tracker_ids_list.append(pr_ids_t)
        sim_list.append(sim)
        num_gt_dets += len(gt_ids_t)
        num_tracker_dets += len(pr_ids_t)

    data = {
        "num_timesteps":    len(frames),
        "num_gt_ids":       len(unique_gt_ids),
        "num_tracker_ids":  len(unique_pr_ids),
        "num_gt_dets":      num_gt_dets,
        "num_tracker_dets": num_tracker_dets,
        "gt_ids":           gt_ids_list,
        "tracker_ids":      tracker_ids_list,
        "similarity_scores": sim_list,
    }
    return data

hota_metric = trackeval.metrics.HOTA()   # object thật của thư viện, không subclass/sửa gì

all_res = {}
pooled_rows = []
for seq_id in SELECTED_SEQUENCES:
    data = build_hota_data_all_classes(seq_id, pred_all)
    res = hota_metric.eval_sequence(data)   # gọi đúng method thật trong hota.py
    all_res[seq_id] = res
    pooled_rows.append({
        "sequence_id": seq_id,
        "HOTA_official_allclass": float(np.mean(res["HOTA"])),
        "DetA_official_allclass": float(np.mean(res["DetA"])),
        "AssA_official_allclass": float(np.mean(res["AssA"])),
        "num_gt_ids": data["num_gt_ids"],
        "num_tracker_ids": data["num_tracker_ids"],
    })
    print(f"  seq {seq_id}: HOTA={np.mean(res['HOTA']):.4f}  DetA={np.mean(res['DetA']):.4f}  "
          f"AssA={np.mean(res['AssA']):.4f}")

pooled_df = pd.DataFrame(pooled_rows)
print()
print("=== HOTA THẬT — toàn bộ class gộp (per-sequence) ===")
display(pooled_df)

combined_res = hota_metric.combine_sequences(all_res)
print()
print("COMBINED (official, 11 sequence, weighted theo lượng detection thật):")
print(f"  HOTA={np.mean(combined_res['HOTA']):.4f}  "
      f"DetA={np.mean(combined_res['DetA']):.4f}  "
      f"AssA={np.mean(combined_res['AssA']):.4f}")
print(f"  (so sánh: trung bình cộng đơn giản 11 sequence = "
      f"{pooled_df['HOTA_official_allclass'].mean():.4f} — hai số này được phép khác nhau)")

pooled_path = TABLE_DIR / "table1_hota_official_trackeval_allclass.csv"
pooled_df.to_csv(pooled_path, index=False)
print()
print("Đã lưu:", pooled_path)

table1_df_official = table1_df.drop(columns=[c for c in ["HOTA", "DetA", "AssA"] if c in table1_df.columns]) \
    .merge(pooled_df.rename(columns={
        "HOTA_official_allclass": "HOTA",
        "DetA_official_allclass": "DetA",
        "AssA_official_allclass": "AssA",
    })[["sequence_id", "HOTA", "DetA", "AssA"]], on="sequence_id", how="left")

display(table1_df_official[["sequence_id", "MOTA", "IDF1", "HOTA", "DetA", "AssA", "FP", "FN", "IDS"]])
table1_df_official.to_csv(TABLE_DIR / "table1_tracking_with_hota_OFFICIAL.csv", index=False)
print("Đã lưu bảng Table 1 cuối cùng (HOTA = TrackEval thật, toàn bộ class):",
      TABLE_DIR / "table1_tracking_with_hota_OFFICIAL.csv")

compare3 = table1_df[["sequence_id", "HOTA"]].rename(columns={"HOTA": "HOTA_tu_viet_allclass"}).merge(
    pooled_df[["sequence_id", "HOTA_official_allclass"]], on="sequence_id", how="left"
)
compare3["chenh_lech_tuyet_doi"] = (compare3["HOTA_tu_viet_allclass"] - compare3["HOTA_official_allclass"]).abs()
print()
print("=== ĐỐI CHIẾU công bằng: tự viết vs TrackEval thật (CÙNG phạm vi toàn class) ===")
display(compare3)
print(f"Chênh lệch tuyệt đối trung bình: {compare3['chenh_lech_tuyet_doi'].mean():.4f}")
print("Nếu chênh lệch nhỏ (~<0.01-0.02): bản tự viết ở cell trước đáng tin cậy.")
print("Nếu chênh lệch lớn: dùng số ở cell này (official) để báo cáo, không dùng bản tự viết.")


In [ ]:
# ============================================================
# 6. MEMBER B N4 — DISTANCE ESTIMATION + TABLE 2 PART 1
# ============================================================

def match_pred_to_gt_for_distance(seq_id, pred_df, iou_thr=0.5):
    seq_id = str(seq_id).zfill(4)
    gt = load_gt_for_sequence(seq_id)
    pred = pred_df[pred_df["sequence_id"].astype(str).str.zfill(4) == seq_id].copy()
    calib = read_kitti_calib(CALIB_DIR / f"{seq_id}.txt")

    rows = []
    if len(gt) == 0 or len(pred) == 0:
        return pd.DataFrame()

    common_frames = sorted(set(gt["frame"].unique()).intersection(set(pred["frame"].unique())))
    for fr in common_frames:
        gtf = gt[gt["frame"] == fr].reset_index(drop=True)
        prf = pred[pred["frame"] == fr].reset_index(drop=True)

        ious = bbox_iou_matrix(gtf[["x1", "y1", "x2", "y2"]].values, prf[["x1", "y1", "x2", "y2"]].values)
        pairs = []
        for i in range(ious.shape[0]):
            for j in range(ious.shape[1]):
                if ious[i, j] >= iou_thr:
                    gt_cls = normalize_class_name(gtf.iloc[i]["type"])
                    pr_cls = normalize_class_name(prf.iloc[j]["class_name"])
                    compatible = (gt_cls == pr_cls) or (gt_cls in ["car", "truck", "bus"] and pr_cls in ["car", "truck", "bus"])
                    if compatible:
                        pairs.append((ious[i, j], i, j))

        pairs.sort(reverse=True)
        used_g, used_p = set(), set()

        for iou, i, j in pairs:
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)

            gr = gtf.iloc[i]
            pr = prf.iloc[j]

            pred_z = estimate_depth_from_bbox(pr, calib)
            pred_x = estimate_x_from_bbox(pr, pred_z, calib)

            gt_z = float(gr["loc_z"])
            gt_x = float(gr["loc_x"])

            rows.append({
                "sequence_id": seq_id,
                "frame": int(fr),
                "gt_track_id": int(gr["track_id"]),
                "pred_track_id": int(pr["track_id"]),
                "gt_class": normalize_class_name(gr["type"]),
                "pred_class": normalize_class_name(pr["class_name"]),
                "iou": float(iou),
                "gt_x": gt_x,
                "gt_z": gt_z,
                "gt_distance": float(np.sqrt(gt_x**2 + gt_z**2)),
                "pred_x_pinhole": pred_x,
                "pred_z_pinhole": pred_z,
                "pred_distance_pinhole": float(np.sqrt(pred_x**2 + pred_z**2)),
                "score": float(pr.get("score", 1.0)),
            })

    return pd.DataFrame(rows)

distance_dfs = []
for seq_id in SELECTED_SEQUENCES:
    ddf = match_pred_to_gt_for_distance(seq_id, pred_all, iou_thr=EVAL_IOU_THRESHOLD)
    if len(ddf):
        distance_dfs.append(ddf)

distance_pairs = pd.concat(distance_dfs, ignore_index=True) if distance_dfs else pd.DataFrame()
distance_pairs_path = TABLE_DIR / "distance_eval_pairs.csv"
distance_pairs.to_csv(distance_pairs_path, index=False)

def distance_range_label(d):
    if d < 10:
        return "0-10m"
    if d < 20:
        return "10-20m"
    if d < 30:
        return "20-30m"
    return ">30m"

dist_rows = []
if len(distance_pairs):
    distance_pairs["range"] = distance_pairs["gt_distance"].apply(distance_range_label)
    distance_pairs["abs_err_pinhole"] = (distance_pairs["pred_distance_pinhole"] - distance_pairs["gt_distance"]).abs()
    distance_pairs["sq_err_pinhole"] = (distance_pairs["pred_distance_pinhole"] - distance_pairs["gt_distance"]) ** 2

    for r in ["0-10m", "10-20m", "20-30m", ">30m"]:
        sub = distance_pairs[distance_pairs["range"] == r]
        if len(sub):
            dist_rows.append({
                "metric_group": "distance",
                "method": "pinhole_bbox_height",
                "range": r,
                "count": len(sub),
                "distance_mae_m": float(sub["abs_err_pinhole"].mean()),
                "distance_rmse_m": float(np.sqrt(sub["sq_err_pinhole"].mean())),
            })

    sub = distance_pairs
    dist_rows.append({
        "metric_group": "distance",
        "method": "pinhole_bbox_height",
        "range": "overall",
        "count": len(sub),
        "distance_mae_m": float(sub["abs_err_pinhole"].mean()),
        "distance_rmse_m": float(np.sqrt(sub["sq_err_pinhole"].mean())),
    })

distance_metrics = pd.DataFrame(dist_rows)
distance_metrics_path = TABLE_DIR / "distance_metrics_by_range.csv"
distance_metrics.to_csv(distance_metrics_path, index=False)

print("Saved:", distance_pairs_path, "rows:", len(distance_pairs))
print("Saved:", distance_metrics_path)
display(distance_metrics)

In [ ]:
# ============================================================
# 7. MEMBER B N5 — VELOCITY + SAVITZKY-GOLAY + TABLE 2
# ============================================================

def compute_velocity_pairs(distance_pairs, fps=10.0):
    rows = []
    if distance_pairs is None or len(distance_pairs) == 0:
        return pd.DataFrame()

    df = distance_pairs.sort_values(["sequence_id", "pred_track_id", "frame"]).copy()
    for (seq, tid), g in df.groupby(["sequence_id", "pred_track_id"], sort=False):
        g = g.drop_duplicates(subset=["frame"], keep="first").sort_values("frame").reset_index(drop=True)
        if len(g) < 3:
            continue

        frames = g["frame"].astype(int).values
        pred_z_raw = g["pred_z_pinhole"].astype(float).values
        gt_z_raw = g["gt_z"].astype(float).values

        pred_z_sg = savgol_or_rolling(pred_z_raw, window=7, polyorder=2)
        gt_z_sg = savgol_or_rolling(gt_z_raw, window=7, polyorder=2)

        for i in range(1, len(g)):
            dframe = frames[i] - frames[i - 1]
            if dframe <= 0:
                continue
            dt = dframe / fps

            pred_v_raw = (pred_z_raw[i] - pred_z_raw[i - 1]) / dt
            gt_v_raw = (gt_z_raw[i] - gt_z_raw[i - 1]) / dt
            pred_v_sg = (pred_z_sg[i] - pred_z_sg[i - 1]) / dt
            gt_v_sg = (gt_z_sg[i] - gt_z_sg[i - 1]) / dt

            rows.append({
                "sequence_id": seq,
                "frame": int(frames[i]),
                "pred_track_id": int(tid),
                "gt_track_id": int(g.iloc[i]["gt_track_id"]),
                "dt": dt,
                "pred_vz_raw_mps": float(pred_v_raw),
                "gt_vz_mps": float(gt_v_raw),
                "pred_vz_sg_mps": float(pred_v_sg),
                "gt_vz_sg_mps": float(gt_v_sg),
                "abs_err_v_raw_mps": float(abs(pred_v_raw - gt_v_raw)),
                "abs_err_v_sg_mps": float(abs(pred_v_sg - gt_v_sg)),
                "pred_z_smooth": float(pred_z_sg[i]),
                "gt_z_smooth": float(gt_z_sg[i]),
            })

    return pd.DataFrame(rows)

velocity_pairs = compute_velocity_pairs(distance_pairs, fps=KITTI_FPS)
velocity_pairs_path = TABLE_DIR / "velocity_eval_pairs.csv"
velocity_pairs.to_csv(velocity_pairs_path, index=False)

if len(velocity_pairs):
    clean = velocity_pairs[velocity_pairs["pred_vz_raw_mps"].abs() < 80].copy()
    velocity_summary = pd.DataFrame([{
        "metric_group": "velocity",
        "count_raw": len(velocity_pairs),
        "count_clean": len(clean),
        "velocity_mae_raw_mps": float(velocity_pairs["abs_err_v_raw_mps"].mean()),
        "velocity_mae_sg_mps": float(velocity_pairs["abs_err_v_sg_mps"].mean()),
        "velocity_mae_raw_clean_mps": float(clean["abs_err_v_raw_mps"].mean()) if len(clean) else np.nan,
        "velocity_mae_sg_clean_mps": float(clean["abs_err_v_sg_mps"].mean()) if len(clean) else np.nan,
        "velocity_mae_sg_clean_kmh": float(clean["abs_err_v_sg_mps"].mean() * 3.6) if len(clean) else np.nan,
        "sg_window": 7,
        "sg_polyorder": 2,
        "removed_outliers_pred_abs_speed_gt_80mps": int(len(velocity_pairs) - len(clean)),
    }])
else:
    velocity_summary = pd.DataFrame([{
        "metric_group": "velocity",
        "count_raw": 0,
        "count_clean": 0,
        "velocity_mae_raw_mps": np.nan,
        "velocity_mae_sg_mps": np.nan,
        "velocity_mae_raw_clean_mps": np.nan,
        "velocity_mae_sg_clean_mps": np.nan,
        "velocity_mae_sg_clean_kmh": np.nan,
        "sg_window": 7,
        "sg_polyorder": 2,
        "removed_outliers_pred_abs_speed_gt_80mps": 0,
    }])

velocity_summary_path = TABLE_DIR / "velocity_summary.csv"
velocity_summary.to_csv(velocity_summary_path, index=False)

# Combined Table 2
table2_rows = []
if len(distance_metrics):
    for _, r in distance_metrics.iterrows():
        row = r.to_dict()
        row["velocity_mae_raw_clean_mps"] = np.nan
        row["velocity_mae_sg_clean_mps"] = np.nan
        row["velocity_mae_sg_clean_kmh"] = np.nan
        table2_rows.append(row)

vrow = velocity_summary.iloc[0].to_dict()
table2_rows.append({
    "metric_group": "velocity",
    "method": "finite_difference_plus_savgol",
    "range": "overall",
    "count": int(vrow["count_clean"]),
    "distance_mae_m": np.nan,
    "distance_rmse_m": np.nan,
    "velocity_mae_raw_clean_mps": vrow["velocity_mae_raw_clean_mps"],
    "velocity_mae_sg_clean_mps": vrow["velocity_mae_sg_clean_mps"],
    "velocity_mae_sg_clean_kmh": vrow["velocity_mae_sg_clean_kmh"],
})

table2_df = pd.DataFrame(table2_rows)
table2_path = TABLE_DIR / "table2_distance_velocity_draft.csv"
table2_df.to_csv(table2_path, index=False)

print("Saved:", velocity_pairs_path, "rows:", len(velocity_pairs))
print("Saved:", velocity_summary_path)
print("Saved:", table2_path)
display(velocity_summary)
display(table2_df)

In [ ]:
# ============================================================
# 8. MEMBER B N5/N6 — PRED TRAJECTORY + TTC + CANDIDATES
# ============================================================

def build_pred_trajectory_from_predictions(pred_df):
    rows = []
    if pred_df is None or len(pred_df) == 0:
        return pd.DataFrame()

    for seq_id, g in pred_df.groupby(pred_df["sequence_id"].astype(str).str.zfill(4)):
        calib = read_kitti_calib(CALIB_DIR / f"{seq_id}.txt")
        for _, r in g.iterrows():
            z = estimate_depth_from_bbox(r, calib)
            x = estimate_x_from_bbox(r, z, calib)
            rows.append({
                "sequence_id": seq_id,
                "frame": int(r["frame"]),
                "pred_track_id": int(r["track_id"]),
                "pred_class": normalize_class_name(r["class_name"]),
                "score": float(r.get("score", 1.0)),
                "x1": float(r["x1"]),
                "y1": float(r["y1"]),
                "x2": float(r["x2"]),
                "y2": float(r["y2"]),
                "bbox_h": float(r["y2"]) - float(r["y1"]),
                "pred_x_pinhole": float(x),
                "pred_z_pinhole": float(z),
            })
    return pd.DataFrame(rows)

def add_ttc_to_trajectory(traj_df, fps=10.0, smooth_window=5):
    outs = []
    if traj_df is None or len(traj_df) == 0:
        return pd.DataFrame()

    for (seq, tid), g in traj_df.sort_values(["sequence_id", "pred_track_id", "frame"]).groupby(["sequence_id", "pred_track_id"], sort=False):
        g = g.drop_duplicates(subset=["frame"], keep="first").sort_values("frame").reset_index(drop=True)
        if len(g) < 2:
            continue

        z_raw = g["pred_z_pinhole"].astype(float).values
        x_raw = g["pred_x_pinhole"].astype(float).values
        z_smooth = pd.Series(z_raw).rolling(smooth_window, center=True, min_periods=1).median().values
        x_smooth = pd.Series(x_raw).rolling(smooth_window, center=True, min_periods=1).median().values

        frames = g["frame"].astype(int).values
        vx = np.full(len(g), np.nan)
        vz = np.full(len(g), np.nan)
        ttc = np.full(len(g), np.nan)

        for i in range(1, len(g)):
            dframe = frames[i] - frames[i - 1]
            if dframe <= 0:
                continue
            dt = dframe / fps
            vx_i = (x_smooth[i] - x_smooth[i - 1]) / dt
            vz_i = (z_smooth[i] - z_smooth[i - 1]) / dt
            vx[i] = vx_i
            vz[i] = vz_i
            if vz_i < 0:
                ttc[i] = z_smooth[i] / max(-vz_i, 1e-6)

        g["pred_x_smooth"] = x_smooth
        g["pred_z_smooth"] = z_smooth
        g["pred_vx"] = vx
        g["pred_vz"] = vz
        g["ttc"] = ttc
        outs.append(g)

    return pd.concat(outs, ignore_index=True) if outs else pd.DataFrame()

pred_traj = build_pred_trajectory_from_predictions(pred_all)
pred_traj_path = TABLE_DIR / "pred_trajectory_3d_all.csv"
pred_traj.to_csv(pred_traj_path, index=False)

pred_ttc = add_ttc_to_trajectory(pred_traj, fps=KITTI_FPS)
pred_ttc_path = TABLE_DIR / "pred_ttc_all.csv"
pred_ttc.to_csv(pred_ttc_path, index=False)

print("Saved:", pred_traj_path, "rows:", len(pred_traj))
print("Saved:", pred_ttc_path, "rows:", len(pred_ttc))
display(pred_ttc.head())

In [ ]:
# ============================================================
# 9. MEMBER A/B N6 — GENERATE COLLISION EVENTS + NEGATIVE WINDOWS
# ============================================================

def normalize_event_columns(df):
    required = {
        "event_id": "",
        "video_id": "",
        "sequence_id": "",
        "start_frame": np.nan,
        "end_frame": np.nan,
        "start_time_sec": np.nan,
        "end_time_sec": np.nan,
        "obj_id_1": "ego",
        "obj_id_2": "",
        "event_type": "ego_object_ttc_candidate",
        "severity": "",
        "ttc_min": np.nan,
        "min_distance": np.nan,
        "mean_score": np.nan,
        "verified_by": "pending_review",
        "is_true_event": "",
        "review_note": "",
        "note": "",
    }
    out = df.copy()
    for col, default in required.items():
        if col not in out.columns:
            out[col] = default
    out["verified_by"] = out["verified_by"].fillna("pending_review").replace("", "pending_review")
    return out[list(required.keys())]

def generate_collision_candidates(pred_ttc, fps=10.0, min_needed=20):
    df = pred_ttc.copy()
    if df is None or len(df) == 0:
        return normalize_event_columns(pd.DataFrame())

    if "pred_track_id" not in df.columns and "track_id" in df.columns:
        df["pred_track_id"] = df["track_id"]
    if "pred_z_smooth" not in df.columns and "pred_z_pinhole" in df.columns:
        df["pred_z_smooth"] = df["pred_z_pinhole"]
    if "pred_vz" not in df.columns:
        df["pred_vz"] = np.nan
    if "ttc" not in df.columns:
        df["ttc"] = np.nan
    if "score" not in df.columns:
        df["score"] = 1.0

    df["sequence_id"] = df["sequence_id"].astype(str).str.zfill(4)
    for c in ["frame", "pred_track_id", "pred_z_smooth", "pred_vz", "ttc", "score"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["frame", "pred_track_id", "pred_z_smooth"]).copy()
    df["frame"] = df["frame"].astype(int)
    df["pred_track_id"] = df["pred_track_id"].astype(int)

    df["approaching"] = df["pred_vz"] < 0
    df["ttc_for_rank"] = df["ttc"].replace([np.inf, -np.inf], np.nan).fillna(999.0)
    df["risk_score"] = (
        (1.0 / np.maximum(df["ttc_for_rank"], 0.2)) * 4.0 +
        (1.0 / np.maximum(df["pred_z_smooth"], 1.0)) * 8.0 +
        df["approaching"].astype(float) * 2.0 +
        df["score"].fillna(0.5)
    )

    rows = []
    used_keys = set()
    event_idx = 1

    def add_event(seq, tid, sub, event_type):
        nonlocal event_idx
        if len(sub) == 0:
            return
        start_frame = int(sub["frame"].min())
        end_frame = int(sub["frame"].max())
        key = (str(seq).zfill(4), int(tid), start_frame, end_frame)
        if key in used_keys:
            return
        used_keys.add(key)

        ttc_min = float(sub["ttc_for_rank"].min())
        if ttc_min >= 900:
            ttc_min = np.nan
        min_dist = float(sub["pred_z_smooth"].min())
        mean_score = float(sub["score"].mean()) if "score" in sub.columns else np.nan

        if pd.notna(ttc_min) and ttc_min < TTC_DANGER_SECONDS:
            severity = "danger"
        elif min_dist < 10:
            severity = "warning"
        else:
            severity = "review_required"

        rows.append({
            "event_id": f"E{event_idx:03d}",
            "video_id": f"KITTI_{str(seq).zfill(4)}",
            "sequence_id": str(seq).zfill(4),
            "start_frame": start_frame,
            "end_frame": end_frame,
            "start_time_sec": start_frame / fps,
            "end_time_sec": end_frame / fps,
            "obj_id_1": "ego",
            "obj_id_2": int(tid),
            "event_type": event_type,
            "severity": severity,
            "ttc_min": ttc_min,
            "min_distance": min_dist,
            "mean_score": mean_score,
            "verified_by": "pending_review",
            "is_true_event": "",
            "review_note": "",
            "note": "Auto-generated candidate for manual review. Not final ground truth until verified.",
        })
        event_idx += 1

    # strict -> relaxed -> fallback
    masks = [
        (df["ttc_for_rank"] < 3.0) & (df["pred_z_smooth"] < 15.0) & (df["approaching"]),
        (df["ttc_for_rank"] < 5.0) & (df["pred_z_smooth"] < 25.0),
        (df["ttc_for_rank"] < 8.0) & (df["pred_z_smooth"] < 40.0),
    ]

    for mi, mask in enumerate(masks):
        risk = df[mask].copy()
        if len(risk) == 0:
            continue
        for (seq, tid), g in risk.sort_values(["sequence_id", "pred_track_id", "frame"]).groupby(["sequence_id", "pred_track_id"], sort=False):
            g = g.sort_values("frame").reset_index(drop=True)
            frames = g["frame"].astype(int).values
            groups = []
            cur = [0]
            for i in range(1, len(g)):
                if frames[i] <= frames[i - 1] + 2:
                    cur.append(i)
                else:
                    groups.append(cur)
                    cur = [i]
            groups.append(cur)

            for inds in groups:
                sub = g.iloc[inds].copy()
                add_event(seq, tid, sub, f"ttc_candidate_pass_{mi+1}")
                if len(rows) >= min_needed:
                    break
            if len(rows) >= min_needed:
                break
        if len(rows) >= min_needed:
            break

    if len(rows) < min_needed:
        top = df.sort_values("risk_score", ascending=False).head(min_needed * 3)
        for _, r in top.iterrows():
            seq = str(r["sequence_id"]).zfill(4)
            tid = int(r["pred_track_id"])
            fr = int(r["frame"])
            sub = df[(df["sequence_id"] == seq) & (df["pred_track_id"] == tid) & (df["frame"].between(fr - 2, fr + 2))].copy()
            add_event(seq, tid, sub, "fallback_top_risk_candidate")
            if len(rows) >= min_needed:
                break

    return normalize_event_columns(pd.DataFrame(rows))

def normalize_negative_columns(df):
    required = {
        "negative_id": "",
        "video_id": "",
        "sequence_id": "",
        "start_frame": np.nan,
        "end_frame": np.nan,
        "start_time_sec": np.nan,
        "end_time_sec": np.nan,
        "window_size_frames": np.nan,
        "reason": "",
        "verified_by": "pending_review",
        "is_true_negative": "",
        "review_note": "",
    }
    out = df.copy()
    for col, default in required.items():
        if col not in out.columns:
            out[col] = default
    out["verified_by"] = out["verified_by"].fillna("pending_review").replace("", "pending_review")
    return out[list(required.keys())]

def generate_negative_windows(pred_ttc, events, fps=10.0, window=10, stride=5, max_per_seq=25, buffer_frames=5):
    rows = []
    neg_idx = 1

    if pred_ttc is not None and len(pred_ttc):
        seq_frames = pred_ttc.copy()
        seq_frames["sequence_id"] = seq_frames["sequence_id"].astype(str).str.zfill(4)
        seq_frames = seq_frames.groupby("sequence_id")["frame"].agg(["min", "max"]).reset_index()
    else:
        seq_frames = pd.DataFrame({"sequence_id": ["0000", "0001"], "min": [0, 0], "max": [150, 150]})

    events = events.copy()
    if len(events):
        events["sequence_id"] = events["sequence_id"].astype(str).str.zfill(4)

    for _, s in seq_frames.iterrows():
        seq = str(s["sequence_id"]).zfill(4)
        start_min = int(s["min"])
        end_max = int(s["max"])
        ev_seq = events[events["sequence_id"] == seq] if len(events) else pd.DataFrame()

        count = 0
        for start in range(start_min, max(start_min + 1, end_max - window + 1), stride):
            end = start + window - 1
            overlap = False

            for _, e in ev_seq.iterrows():
                ev_start = int(e["start_frame"]) - buffer_frames
                ev_end = int(e["end_frame"]) + buffer_frames
                if not (end < ev_start or start > ev_end):
                    overlap = True
                    break

            if overlap:
                continue

            rows.append({
                "negative_id": f"N{neg_idx:03d}",
                "video_id": f"KITTI_{seq}",
                "sequence_id": seq,
                "start_frame": int(start),
                "end_frame": int(end),
                "start_time_sec": start / fps,
                "end_time_sec": end / fps,
                "window_size_frames": window,
                "reason": "no_overlap_with_collision_candidate_plus_buffer",
                "verified_by": "pending_review",
                "is_true_negative": "",
                "review_note": "",
            })
            neg_idx += 1
            count += 1
            if count >= max_per_seq:
                break

    return normalize_negative_columns(pd.DataFrame(rows))

collision_events = generate_collision_candidates(pred_ttc, fps=KITTI_FPS, min_needed=20)
collision_path = TABLE_DIR / "collision_events_draft.csv"
risk_path = TABLE_DIR / "ttc_risk_candidates.csv"
collision_events.to_csv(collision_path, index=False)
collision_events.to_csv(risk_path, index=False)

negative_windows = generate_negative_windows(pred_ttc, collision_events, fps=KITTI_FPS)
negative_path = TABLE_DIR / "negative_windows_draft.csv"
negative_windows.to_csv(negative_path, index=False)

annotation_path = TABLE_DIR / "annotation_review_sheet_week1.csv"
collision_events.to_csv(annotation_path, index=False)

print("Saved:", collision_path, "rows:", len(collision_events))
print("Saved:", negative_path, "rows:", len(negative_windows))
if len(collision_events) < 15:
    print("WARNING: Chưa đủ 15 candidates, cần chạy thêm sequence/frames hoặc kiểm tra prediction.")
else:
    print("OK: Đủ >=15 collision candidates để review.")
display(collision_events.head(25))
display(negative_windows.head(25))

In [ ]:
# ============================================================
# 10. MEMBER B N5 — BEV FIGURE
# ============================================================

import matplotlib.pyplot as plt

def choose_bev_frame(pred_ttc_df):
    if pred_ttc_df is None or len(pred_ttc_df) == 0:
        return None, None
    df = pred_ttc_df.copy()
    df = df[np.isfinite(df["pred_z_smooth"])]
    df = df[(df["pred_z_smooth"] > 0) & (df["pred_z_smooth"] < 80)]
    if len(df) == 0:
        return None, None
    counts = df.groupby(["sequence_id", "frame"]).size().reset_index(name="count")
    best = counts.sort_values("count", ascending=False).iloc[0]
    return str(best["sequence_id"]).zfill(4), int(best["frame"])

bev_seq, bev_frame = choose_bev_frame(pred_ttc)

if bev_seq is None:
    print("[WARN] Cannot create BEV: no valid pred_ttc rows.")
else:
    bev_df = pred_ttc[(pred_ttc["sequence_id"].astype(str).str.zfill(4) == bev_seq) & (pred_ttc["frame"] == bev_frame)].copy()
    bev_df = bev_df[np.isfinite(bev_df["pred_z_smooth"])]
    bev_df = bev_df[(bev_df["pred_z_smooth"] > 0) & (bev_df["pred_z_smooth"] < 80)]

    fig = plt.figure(figsize=(8, 8))
    plt.scatter(bev_df["pred_x_smooth"], bev_df["pred_z_smooth"], s=40)

    for _, r in bev_df.iterrows():
        x = float(r["pred_x_smooth"])
        z = float(r["pred_z_smooth"])
        vx = float(r["pred_vx"]) if np.isfinite(r["pred_vx"]) else 0
        vz = float(r["pred_vz"]) if np.isfinite(r["pred_vz"]) else 0
        tid = int(r["pred_track_id"])
        cls = str(r.get("pred_class", "obj"))
        plt.arrow(x, z, vx * 0.2, vz * 0.2, head_width=0.3, length_includes_head=True)
        plt.text(x, z, f"{cls}-{tid}\\n{z:.1f}m", fontsize=8)

    plt.xlabel("Lateral X (m)")
    plt.ylabel("Longitudinal Z (m)")
    plt.title(f"BEV Draft — sequence {bev_seq}, frame {bev_frame}")
    plt.grid(True)
    plt.gca().invert_yaxis()
    plt.tight_layout()

    bev_path = FIGURE_DIR / f"bev_draft_seq_{bev_seq}_frame_{bev_frame:06d}.png"
    plt.savefig(bev_path, dpi=180)
    plt.show()

    print("Saved:", bev_path)

In [ ]:
# ============================================================
# 11. MEMBER A N1/N2/N3/N5 — CONFIG, PROGRESS, RELATED WORK, PROTOCOL
# ============================================================

member_a_config = {
    "project": "Collision Prediction from Monocular Video",
    "week": 1,
    "member": "A",
    "role": "Lead / Paper / Collision / Ablation / Writing",
    "generated_at": datetime.now().isoformat(),
    "common_config": {
        "fps": KITTI_FPS,
        "nms_threshold": IOU_THRES,
        "det_conf_threshold": CONF_THRES,
        "ttc_warning_threshold_seconds": TTC_WARNING_SECONDS,
        "ttc_danger_threshold_seconds": TTC_DANGER_SECONDS,
        "collision_distance_threshold_m": TTC_DISTANCE_THRESHOLD_M,
        "tracker": "ByteTrack",
    },
    "week1_lock_criteria": {
        "min_verified_true_positive_events": 15,
        "min_verified_true_negative_windows": 15,
        "annotation_protocol_required": True,
        "related_work_draft_required": True,
        "object_id_consistency_report_required": True,
    },
}

config_path = PAPER_DIR / "member_a_week1_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(member_a_config, f, ensure_ascii=False, indent=2)

progress_df = pd.DataFrame([
    ["N1", "Tạo repo/cấu trúc thư mục, config chung, progress tracker", "done"],
    ["N2", "Đọc + ghi chú TrackFormer/TransTrack, MonoDepth2/DPT/MiDaS, LSTM/GRU/Transformer trajectory", "done"],
    ["N3", "Viết annotation protocol, tạo collision_events_draft.csv", "done"],
    ["N4", "Kiểm tra consistency object ID, chuẩn bị review events", "done"],
    ["N5", "Viết 2–3 đoạn Related Work draft", "done"],
    ["N6", "Tạo negative windows và manual review workflow", "pending_manual_review"],
    ["N7", "Sync/report/handoff tuần 2", "generated_later"],
], columns=["day", "member_a_task", "status"])

progress_path = TABLE_DIR / "member_a_week1_progress_tracker.csv"
progress_df.to_csv(progress_path, index=False)

related_work_notes = '''
# Related Work Notes — Week 1

## TrackFormer / TransTrack [GY1]
TrackFormer and TransTrack are transformer-based multi-object tracking methods. They are relevant because they can improve temporal association and identity consistency. This project uses ByteTrack as the main baseline due to limited time, GPU resources, and the need for a reproducible pipeline.

## MonoDepth2 / DPT / MiDaS [GY2]
Geometric monocular depth is explainable but sensitive to object-height and flat-ground assumptions. MonoDepth2, DPT, and MiDaS are discussed as learned alternatives and future work.

## LSTM / GRU / Transformer trajectory [GY4]
The current TTC module assumes short-term constant velocity. LSTM/GRU/Transformer trajectory models can better handle nonlinear motion, sudden braking, turning, and lane changes.
'''

related_work_draft = '''
# Related Work Draft — Week 1

## Transformer-based Multi-Object Tracking [GY1]

Recent multi-object tracking methods increasingly adopt transformer-based architectures. TrackFormer formulates multi-object tracking as a set-prediction problem and uses track queries to preserve object identities across frames. TransTrack also uses transformer attention by combining previous-frame object queries with learned object queries for newly appearing objects. These methods are important because they can reduce the dependence on handcrafted association and improve identity consistency in crowded scenes.

However, this project adopts YOLO with ByteTrack as the main tracking baseline. This decision is motivated by real-time constraints, limited GPU resources, and the need to connect tracking outputs directly with distance estimation, velocity estimation, BEV visualization, and TTC-based warning logic. Therefore, TrackFormer and TransTrack are included as important related methods, while the main experiment focuses on the reproducible YOLO + ByteTrack pipeline.

## Monocular Depth Estimation [GY2]

Depth estimation is a key limitation of monocular collision prediction because a single camera does not directly provide metric 3D information. The current system uses geometric depth estimation based on calibration and bounding-box geometry. This method is simple and explainable, but it depends on object-height assumptions and flat-ground geometry. Such assumptions can introduce errors for trucks, motorcycles, pedestrians, truncated objects, and sloped road surfaces.

Learning-based monocular depth methods such as MonoDepth2, DPT, and MiDaS provide an alternative direction. They can use semantic and global image cues to infer depth from a single image, potentially reducing the dependence on fixed object-height assumptions. Nevertheless, they introduce scale-alignment issues, additional runtime cost, and evaluation complexity. For this reason, learned monocular depth is discussed as future work or an optional comparison, while the main Week 1 pipeline uses geometric depth.

## Trajectory Prediction for Collision Risk [GY4]

The current TTC module assumes that recent motion can be approximated by constant velocity. This assumption is efficient and interpretable, but it may fail in scenarios involving sudden braking, acceleration, lane changes, turning motion, or pedestrian behavior. Trajectory prediction methods based on LSTM, GRU, graph neural networks, and transformers can model longer temporal history and interactions among road users.

In this project, trajectory prediction is not implemented as the main Week 1 experiment because the priority is to establish a working tracking-distance-velocity-TTC pipeline. However, LSTM/GRU/Transformer-based prediction is discussed as a future direction for reducing missed warnings and false alarms in nonlinear traffic scenarios.
'''

references_bib = '''
@inproceedings{meinhardt2022trackformer,
  title={TrackFormer: Multi-Object Tracking with Transformers},
  author={Meinhardt, Tim and Kirillov, Alexander and Leal-Taixe, Laura and Feichtenhofer, Christoph},
  booktitle={CVPR},
  year={2022}
}

@article{sun2020transtrack,
  title={TransTrack: Multiple Object Tracking with Transformer},
  author={Sun, Peize and Cao, Jinkun and Jiang, Yi and Zhang, Rufeng and Xie, Enze and Yuan, Zehuan and Wang, Changhu and Luo, Ping},
  journal={arXiv preprint arXiv:2012.15460},
  year={2020}
}

@inproceedings{godard2019monodepth2,
  title={Digging Into Self-Supervised Monocular Depth Estimation},
  author={Godard, Clement and Mac Aodha, Oisin and Firman, Michael and Brostow, Gabriel},
  booktitle={ICCV},
  year={2019}
}

@inproceedings{ranftl2021dpt,
  title={Vision Transformers for Dense Prediction},
  author={Ranftl, Rene and Bochkovskiy, Alexey and Koltun, Vladlen},
  booktitle={ICCV},
  year={2021}
}
'''

annotation_protocol = '''
# Collision Event Annotation Protocol — Week 1

## Goal
This protocol defines how TTC-based candidates are manually reviewed before being used as collision-risk ground truth.

## Positive event rule
Mark a candidate as positive if the object is visually relevant to collision or near-collision risk, distance decreases clearly, object ID is stable, and the warning is not caused only by one-frame depth jitter.

## False candidate rule
Mark a candidate as false if it is caused by depth jitter, ID switch, wrong detection, another lane, irrelevant object, or ambiguous/too short motion.

## Negative window rule
A negative window is valid if it does not overlap with reviewed positive events and no visible collision risk is present.

## Required fields
`video_id`, `start_frame`, `end_frame`, `obj_id_1`, `obj_id_2`, `event_type`, `severity`, `verified_by`, `is_true_event`, `review_note`.

## Lock criterion
Week 1 is locked only when at least 15 true positive events and 15 true negative windows are manually verified.
'''

(PAPER_DIR / "related_work_notes_week1.md").write_text(related_work_notes.strip(), encoding="utf-8")
(PAPER_DIR / "related_work_draft_week1.md").write_text(related_work_draft.strip(), encoding="utf-8")
(PAPER_DIR / "related_work_outline_week1.md").write_text(related_work_notes.strip(), encoding="utf-8")
(PAPER_DIR / "references_week1.bib").write_text(references_bib.strip(), encoding="utf-8")
(PAPER_DIR / "annotation_protocol_week1.md").write_text(annotation_protocol.strip(), encoding="utf-8")

print("Saved Member A docs.")
display(progress_df)

In [ ]:
# ============================================================
# 12. MEMBER A N4/N6 — REVIEW SHEETS + OBJECT ID CONSISTENCY
# ============================================================

events_df = pd.read_csv(collision_path)
negatives_df = pd.read_csv(negative_path)

review_events = events_df.copy()
if "ttc_min" in review_events.columns and "min_distance" in review_events.columns:
    review_events = review_events.sort_values(["ttc_min", "min_distance"], na_position="last").reset_index(drop=True)

review_events["member_A_instruction"] = (
    "Open demo video around start_time_sec/end_time_sec. "
    "Set verified_by=member_A_checked; is_true_event=1 or 0; write review_note."
)
review_events_path = TABLE_DIR / "annotation_review_sheet_week1_member_A.csv"
review_events.to_csv(review_events_path, index=False)

review_neg = negatives_df.copy()
review_neg["member_A_instruction"] = (
    "Open demo video around start_time_sec/end_time_sec. "
    "Set verified_by=member_A_checked; is_true_negative=1 or 0; write review_note."
)
review_neg_path = TABLE_DIR / "negative_windows_review_sheet_week1_member_A.csv"
review_neg.to_csv(review_neg_path, index=False)

id_rows = []
for _, r in events_df.iterrows():
    start = int(r.get("start_frame", 0))
    end = int(r.get("end_frame", start))
    duration = end - start + 1
    obj_id = str(r.get("obj_id_2", ""))

    if obj_id.lower() in ["", "nan", "none"]:
        id_status = "missing_object_id"
        note = "Missing object ID."
    elif duration < 2:
        id_status = "very_short_event"
        note = "Very short event; inspect carefully."
    elif duration < 3:
        id_status = "short_event"
        note = "Short event; check if real or depth spike."
    else:
        id_status = "needs_video_review"
        note = "Check track ID consistency in video."

    id_rows.append({
        "event_id": r.get("event_id", ""),
        "sequence_id": str(r.get("sequence_id", "")).zfill(4),
        "obj_id_2": obj_id,
        "start_frame": start,
        "end_frame": end,
        "duration_frames": duration,
        "id_status": id_status,
        "note": note,
    })

id_report = pd.DataFrame(id_rows)
id_report_path = TABLE_DIR / "object_id_consistency_report.csv"
id_report.to_csv(id_report_path, index=False)

suggest_events = review_events["event_id"].astype(str).head(15).tolist()
suggest_negs = review_neg["negative_id"].astype(str).head(15).tolist()

manual_template = f'''# Sau khi xem video, copy IDs thật vào cell Manual Review.
VALID_EVENT_IDS = {suggest_events}
FALSE_EVENT_IDS = []

VALID_NEGATIVE_IDS = {suggest_negs}
INVALID_NEGATIVE_IDS = []

REVIEWER_NAME = "member_A_checked"
'''
manual_template_path = REVIEW_DIR / "manual_review_ids_template.py"
manual_template_path.write_text(manual_template, encoding="utf-8")

print("Saved:", review_events_path)
print("Saved:", review_neg_path)
print("Saved:", id_report_path)
print("Saved:", manual_template_path)
print(manual_template)
display(review_events.head(20))
display(review_neg.head(20))
display(id_report.head(20))

In [ ]:
# ============================================================
# 13. MEMBER A N6 — MANUAL REVIEW APPLY
# ============================================================
# QUAN TRỌNG:
# Hãy xem video trước khi điền các danh sách bên dưới.
# Không nên khóa tuần 1 bằng các ID chưa được xem video.

VALID_EVENT_IDS = [
    # "E001", "E002", ...
]

FALSE_EVENT_IDS = [
    # "E003", ...
]

VALID_NEGATIVE_IDS = [
    # "N001", "N002", ...
]

INVALID_NEGATIVE_IDS = [
    # "N003", ...
]

REVIEWER_NAME = "member_A_checked"

def apply_event_review(path, valid_ids, false_ids, reviewer):
    df = pd.read_csv(path)
    valid_ids = set(str(x) for x in valid_ids)
    false_ids = set(str(x) for x in false_ids)

    overlap = valid_ids & false_ids
    if overlap:
        raise ValueError("Event IDs duplicated in valid/false: " + str(sorted(overlap)))

    for col in ["verified_by", "is_true_event", "review_note"]:
        if col not in df.columns:
            df[col] = ""

    df["event_id"] = df["event_id"].astype(str)
    df["verified_by"] = df["verified_by"].fillna("pending_review").replace("", "pending_review")

    valid_mask = df["event_id"].isin(valid_ids)
    false_mask = df["event_id"].isin(false_ids)

    df.loc[valid_mask, "verified_by"] = reviewer
    df.loc[valid_mask, "is_true_event"] = 1
    df.loc[valid_mask, "review_note"] = df.loc[valid_mask, "review_note"].replace("", "valid TTC risk after video review")

    df.loc[false_mask, "verified_by"] = reviewer
    df.loc[false_mask, "is_true_event"] = 0
    df.loc[false_mask, "review_note"] = df.loc[false_mask, "review_note"].replace("", "false alarm after video review")

    df.to_csv(path, index=False)
    reviewed_path = path.parent / "collision_events_member_A_reviewed.csv"
    df.to_csv(reviewed_path, index=False)
    return df, reviewed_path

def apply_negative_review(path, valid_ids, invalid_ids, reviewer):
    df = pd.read_csv(path)
    valid_ids = set(str(x) for x in valid_ids)
    invalid_ids = set(str(x) for x in invalid_ids)

    overlap = valid_ids & invalid_ids
    if overlap:
        raise ValueError("Negative IDs duplicated in valid/invalid: " + str(sorted(overlap)))

    for col in ["verified_by", "is_true_negative", "review_note"]:
        if col not in df.columns:
            df[col] = ""

    df["negative_id"] = df["negative_id"].astype(str)
    df["verified_by"] = df["verified_by"].fillna("pending_review").replace("", "pending_review")

    valid_mask = df["negative_id"].isin(valid_ids)
    invalid_mask = df["negative_id"].isin(invalid_ids)

    df.loc[valid_mask, "verified_by"] = reviewer
    df.loc[valid_mask, "is_true_negative"] = 1
    df.loc[valid_mask, "review_note"] = df.loc[valid_mask, "review_note"].replace("", "safe negative window after video review")

    df.loc[invalid_mask, "verified_by"] = reviewer
    df.loc[invalid_mask, "is_true_negative"] = 0
    df.loc[invalid_mask, "review_note"] = df.loc[invalid_mask, "review_note"].replace("", "invalid/ambiguous negative window after video review")

    df.to_csv(path, index=False)
    reviewed_path = path.parent / "negative_windows_member_A_reviewed.csv"
    df.to_csv(reviewed_path, index=False)
    return df, reviewed_path

events_reviewed, events_reviewed_path = apply_event_review(collision_path, VALID_EVENT_IDS, FALSE_EVENT_IDS, REVIEWER_NAME)
neg_reviewed, neg_reviewed_path = apply_negative_review(negative_path, VALID_NEGATIVE_IDS, INVALID_NEGATIVE_IDS, REVIEWER_NAME)

verified_true_events = int(
    (
        events_reviewed["verified_by"].astype(str).ne("pending_review") &
        events_reviewed["is_true_event"].astype(str).eq("1")
    ).sum()
)
verified_true_negatives = int(
    (
        neg_reviewed["verified_by"].astype(str).ne("pending_review") &
        neg_reviewed["is_true_negative"].astype(str).eq("1")
    ).sum()
)

print("Saved:", events_reviewed_path)
print("Saved:", neg_reviewed_path)
print("verified_true_events:", verified_true_events)
print("verified_true_negatives:", verified_true_negatives)

if verified_true_events >= 15 and verified_true_negatives >= 15:
    print("OK: Đủ điều kiện manual review để khóa tuần 1.")
else:
    print("CHƯA KHÓA: cần >=15 verified positive events và >=15 verified negative windows.")

In [ ]:
# ============================================================
# 14. WEEK 1 FINAL STATUS + REPORT + PACKAGE
# ============================================================

def exists(path):
    return Path(path).exists()

def count_rows(path):
    path = Path(path)
    if not path.exists():
        return 0
    try:
        return len(pd.read_csv(path))
    except Exception:
        return 0

def count_verified_events(path):
    if not exists(path):
        return 0, 0
    df = pd.read_csv(path)
    if "verified_by" not in df.columns:
        return 0, 0
    verified = df["verified_by"].astype(str).ne("pending_review")
    true_mask = df["is_true_event"].astype(str).eq("1") if "is_true_event" in df.columns else False
    return int((verified & true_mask).sum()), int(verified.sum())

def count_verified_negatives(path):
    if not exists(path):
        return 0, 0
    df = pd.read_csv(path)
    if "verified_by" not in df.columns:
        return 0, 0
    verified = df["verified_by"].astype(str).ne("pending_review")
    true_mask = df["is_true_negative"].astype(str).eq("1") if "is_true_negative" in df.columns else False
    return int((verified & true_mask).sum()), int(verified.sum())

verified_true_events, verified_event_total = count_verified_events(collision_path)
verified_true_negatives, verified_negative_total = count_verified_negatives(negative_path)

video_files = list(VIDEO_DIR.glob("*.mp4"))
bev_files = list(FIGURE_DIR.glob("*bev*.png")) + list(FIGURE_DIR.glob("*BEV*.png"))

status = {
    "generated_at": datetime.now().isoformat(),
    "week": 1,
    "scope": "Full Week 1 — Member A + Member B",
    "can_start_week2_draft": bool(count_rows(collision_path) >= 15 and count_rows(negative_path) >= 15),
    "can_finalize_week2_collision_metrics": bool(verified_true_events >= 15 and verified_true_negatives >= 15),
    "can_lock_week1": bool(verified_true_events >= 15 and verified_true_negatives >= 15),
    "counts": {
        "selected_sequences": SELECTED_SEQUENCES,
        "prediction_rows": count_rows(pred_all_path),
        "distance_pairs": count_rows(distance_pairs_path),
        "velocity_pairs": count_rows(velocity_pairs_path),
        "collision_event_candidates": count_rows(collision_path),
        "negative_windows": count_rows(negative_path),
        "verified_true_positive_events": verified_true_events,
        "verified_event_rows_total": verified_event_total,
        "verified_true_negative_windows": verified_true_negatives,
        "verified_negative_rows_total": verified_negative_total,
        "demo_video_count": len(video_files),
        "bev_figure_count": len(bev_files),
    },
    "artifact_checks": {
        "table1_tracking_draft": exists(table1_path),
        "table2_distance_velocity_draft": exists(table2_path),
        "collision_events_draft": exists(collision_path),
        "negative_windows_draft": exists(negative_path),
        "annotation_protocol": exists(PAPER_DIR / "annotation_protocol_week1.md"),
        "related_work_draft": exists(PAPER_DIR / "related_work_draft_week1.md"),
        "object_id_consistency_report": exists(id_report_path),
        "demo_video_exists": len(video_files) > 0,
        "bev_figure_exists": len(bev_files) > 0,
    },
    "files": {
        "system_info": str(system_info_path),
        "dataset_summary": str(dataset_summary_path),
        "predictions_all": str(pred_all_path),
        "table1_tracking_draft": str(table1_path),
        "table2_distance_velocity_draft": str(table2_path),
        "distance_eval_pairs": str(distance_pairs_path),
        "velocity_eval_pairs": str(velocity_pairs_path),
        "pred_ttc_all": str(pred_ttc_path),
        "collision_events_draft": str(collision_path),
        "negative_windows_draft": str(negative_path),
        "annotation_review_sheet": str(review_events_path),
        "negative_review_sheet": str(review_neg_path),
        "object_id_consistency_report": str(id_report_path),
        "manual_review_template": str(manual_template_path),
        "related_work_draft": str(PAPER_DIR / "related_work_draft_week1.md"),
        "annotation_protocol": str(PAPER_DIR / "annotation_protocol_week1.md"),
        "demo_videos": [str(p) for p in video_files],
        "bev_figures": [str(p) for p in bev_files],
    },
    "note": "Rows with verified_by=pending_review are candidates only, not final ground truth.",
}

status_path = TABLE_DIR / "week1_full_status_report.json"
with open(status_path, "w", encoding="utf-8") as f:
    json.dump(status, f, ensure_ascii=False, indent=2)

report_text = (
    "# Week 1 Full Report — Member A + Member B\\n\\n"
    "## Summary\\n\\n"
    "This notebook completed the Week 1 pipeline for tracking, distance estimation, velocity estimation, BEV visualization, "
    "TTC candidate generation, annotation protocol, Related Work draft, review sheets, and handoff status.\\n\\n"
    "## Key outputs\\n\\n"
    f"- Table 1 tracking draft exists: {exists(table1_path)}\\n"
    f"- Table 2 distance/velocity draft exists: {exists(table2_path)}\\n"
    f"- Collision event candidates: {count_rows(collision_path)}\\n"
    f"- Negative windows: {count_rows(negative_path)}\\n"
    f"- Verified true positive events: {verified_true_events}/15\\n"
    f"- Verified true negative windows: {verified_true_negatives}/15\\n"
    f"- Demo videos: {len(video_files)}\\n"
    f"- BEV figures: {len(bev_files)}\\n\\n"
    "## Lock status\\n\\n"
    f"- Can start Week 2 draft: {status['can_start_week2_draft']}\\n"
    f"- Can finalize Week 2 collision metrics: {status['can_finalize_week2_collision_metrics']}\\n"
    f"- Can lock Week 1: {status['can_lock_week1']}\\n\\n"
    "## Important note\\n\\n"
    "`collision_events_draft.csv` contains candidates. Final ground truth requires manual video review.\\n"
)

report_path = REPORT_DIR / "week1_full_report.md"
report_path.write_text(report_text, encoding="utf-8")

# Evidence checklist
evidence_rows = []
for key, value in status["files"].items():
    if isinstance(value, list):
        evidence_rows.append({"artifact": key, "path": "; ".join(value), "exists": len(value) > 0})
    else:
        evidence_rows.append({"artifact": key, "path": value, "exists": Path(value).exists()})
evidence_df = pd.DataFrame(evidence_rows)
evidence_path = TABLE_DIR / "week1_full_evidence_checklist.csv"
evidence_df.to_csv(evidence_path, index=False)

# Package all core outputs
package_path = RESULTS_DIR / "week1_full_A_B_outputs.zip"
with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for _, row in evidence_df.iterrows():
        if not row["exists"]:
            continue
        for p_str in str(row["path"]).split("; "):
            p = Path(p_str)
            if p.exists() and p.is_file():
                try:
                    z.write(p, arcname=str(p.relative_to(PROJECT_ROOT)))
                except Exception:
                    z.write(p, arcname=p.name)
    for extra in [status_path, report_path, evidence_path]:
        if Path(extra).exists():
            z.write(extra, arcname=str(Path(extra).relative_to(PROJECT_ROOT)))

print("Saved status:", status_path)
print("Saved report:", report_path)
print("Saved evidence:", evidence_path)
print("Saved package:", package_path)
display(evidence_df)
print(json.dumps(status, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
import pandas as pd

TABLE_DIR = Path("/kaggle/working/collision_prediction_project/results/tables")

events = pd.read_csv(TABLE_DIR / "annotation_review_sheet_week1_member_A.csv")
negatives = pd.read_csv(TABLE_DIR / "negative_windows_review_sheet_week1_member_A.csv")

print("EVENT IDS:")
display(events[[
    "event_id", "sequence_id", "start_time_sec", "end_time_sec",
    "obj_id_2", "ttc_min", "min_distance", "severity"
]].head(30))

print("NEGATIVE IDS:")
display(negatives[[
    "negative_id", "sequence_id", "start_time_sec", "end_time_sec",
    "reason"
]].head(30))

# GHI CHÚ SỬA LỖI — TUẦN 2 (PHIÊN BẢN ĐÃ SỬA)

Phần **Tuần 2/Tuần 3 gốc (cell cũ 18-31) đã bị xoá hoàn toàn** và viết lại từ đầu, vì chứa các vấn đề sau (đã xác minh bằng cách đọc trực tiếp source code):

| # | Vấn đề trong bản gốc | Vị trí (cell cũ) | Cách đã sửa ở dưới |
|---|---|---|---|
| 1 | `fpr = min(far/30.0, 1.0)` — không phải FPR thật (không có TN) | cell 17, 20 | Tính `FPR = FP/(FP+TN)` đúng chuẩn, có TN thật |
| 2 | `return 2.84` hardcode + chuỗi text gõ tay `"MAE 2.84 km/h"` trong legend | cell 30, 31 | MAE lấy trực tiếp từ biến đã tính bằng `mean_absolute_error`, không gõ tay |
| 3 | `corrected_v = final_v*(1-0.15) + gt_v_array[idx]*0.15` — trộn ground-truth vào "dự đoán" | cell 30 | Monte Carlo chỉ nhiễu hoá *giá trị dự đoán*, không truy cập `gt_v` ở bất kỳ bước nào |
| 4 | Bảng ablation/runtime/context gõ tay (`"F1-Score": 0.890` …) | cell 21, 22, 23, 26, 27, 28 | Tính lại từ `eval_base`/`pred_ttc` thật; dòng chưa chạy được ghi rõ `NOT_YET_RUN` |
| 5 | Fallback `np.random.uniform(...)` khi thiếu biến — chạy ra số ngẫu nhiên mà không cảnh báo rõ | cell 19, 24, 30 | Cell đầu tiên `raise RuntimeError` nếu thiếu biến Tuần 1, không fallback ngẫu nhiên |
| 6 | GT collision có fallback dùng `pred_z` thay `gt_z` (circular) | cell 17 | Bắt buộc dùng `gt_z` (LiDAR thật) trong `distance_pairs`, không fallback sang pred |

**Hạn chế còn lại cần ghi rõ trong paper, không thể sửa bằng code:** KITTI là dữ liệu lái xe an toàn (không có va chạm thật), nên "ground truth collision" ở đây là *near-conflict proxy event* dựa trên ngưỡng khoảng cách/TTC từ LiDAR thật, **không phải nhãn va chạm thật được con người xác nhận**. Cần nêu rõ giới hạn này trong Section V (Discussion) thay vì gọi là "ground-truth collision" không kèm chú thích.

**Quan trọng:** các cell dưới đây **chưa được chạy** (không có GPU/dataset KITTI trong môi trường tạo file này). Cần chạy lại toàn bộ notebook (cell 0 → cuối) trên Kaggle để có số liệu thật, sau đó dùng số liệu đó để viết lại Table 1-5 và Fig. 2-4 trong paper.

---

## Cap nhat lan 2 (sau khi chay thu va thay ket qua qua khac so voi PDF)

Sau khi chay ban sua lan 1 tren Kaggle, ket qua THAT cho thay F1 thuc te (~0.07) rat
thap so voi so PDF cu (0.701 - da xac nhan la fabricate). Hai van de duoc phat hien va
sua tiep o ban nay:

1. **`MAX_FRAMES_PER_SEQ` (Cell 1, Tuan 1)**: truoc gioi han 150 frame/sequence (~30s
   tong cho 2 sequence) -> FAR/gio ngoai suy tu mau qua nho, khong on dinh thong ke.
   Da sua thanh `None` de dung toan bo sequence. **Can chay lai TOAN BO Tuan 1** (YOLO
   inference) sau khi sua dong nay.
2. **Frame-level -> Event-level (debounce)**: confusion matrix truoc tinh tren tung
   frame doc lap, khien 1 canh bao keo dai N frame lien tiep bi dem thanh N FP/TP
   rieng le, lam FAR/gio bi phong dai bat thuong. Da sua sang tinh theo WINDOW
   (`EVENT_WINDOW_FRAMES = 10` frame ~ 1s), dung phep OR tren frame trong cung window.

**Tat ca output trong file nay da duoc xoa (chua chay)** vi thay doi (1) lam vo hieu
toan bo cac bien Tuan 1 (`pred_ttc`, `distance_pairs`, `velocity_pairs`, ...). Can chay
lai notebook nay tu dau tren Kaggle de co so lieu thuc moi.

---

## Cap nhat lan 3 (mo rong du lieu + them context heuristics thuc te)

1. **Mo rong sequence (Cell 1, Tuan 1)**: tu `["0000","0001"]` (601 frame, ~58s)
   len 11 sequence `0000-0010`. Muc tieu: co vai phut du lieu thuc de FAR/gio
   khong con bi ngoai suy tu mau qua nho. **Can chay lai TOAN BO Tuan 1** (YOLO
   se chay lau hon vi nhieu frame hon - kiem tra GPU-hour Kaggle con du khong).
2. **Context-aware heuristics (cell moi sau Ablation)**: implement THAT 2/3
   heuristic paper da mo ta (trajectory monotonicity + lane corridor check thay
   cho future IoU overlap). Heuristic vertical-separation KHONG implement duoc
   vi kien truc hien tai la ego-centric (1 track vs camera), khong phai pairwise
   (2 track voi nhau) nhu paper mo ta - day la gioi han can ghi ro trong Section
   V/VI cua paper, khong phai loi can sua.

**Tat ca output da duoc xoa (chua chay)** vi Fix #1 lam vo hieu toan bo Tuan 1.
Chay lai tu dau tren Kaggle de co so lieu thuc moi.

---

## Cap nhat lan 4 (hoan thien YOLOv8x ablation + runtime full pipeline)

1. **YOLOv8x ablation (cell moi sau Heuristics)**: chay lai THAT detection+tracking+
   distance+velocity+eval voi `yolo8x.pt`, dien so thuc vao dong NOT_YET_RUN cua
   Table 4. Ton GPU-hour tuong duong 1 lan chay Tuan 1 day du.
2. **Runtime full pipeline (cell Runtime)**: them do (b) detection+ByteTrack qua
   `model.track()` de tach overhead ByteTrack, va (d) thoi gian tinh TTC tren ca
   sequence mau. FPS cuoi cung phan anh dung pipeline day du, khong chi detection.

**Tat ca output da xoa (chua chay)**. Chay lai tren Kaggle - luu y cell YOLOv8x se
chay lau (~tuong duong lan chay YOLOv11x ban dau tren 11 sequence).

---

## Cap nhat lan 5 (ap dung 4 chien luoc tu ban phan bien Q1 toan dien)

Tai lieu phan bien de xuat ~15 huong cai thien. 4 viec sau duoc chon vi Impact
"Cao" theo bang Muc 10 cua tai lieu VA kha thi ngay (khong can dataset moi):

1. **BoT-SORT ablation** - doi tracker, khong can cai dependency moi (co san trong
   ultralytics). So sanh truc tiep voi ByteTrack tren cung dataset/protocol.
2. **TTC+DRAC fusion (Multi-SSM)** - implement DRAC tu cong thuc vat ly, hoc
   trong so fusion bang logistic regression **CO TRAIN/TEST SPLIT THEO SEQUENCE**
   (khong leakage). PET KHONG implement duoc vi can kien truc pairwise.
3. **Bootstrap CI (1000 resample) + McNemar's test** - kiem dinh y nghia thong ke
   cho so sanh co/khong heuristics.
4. TTC-only baseline (Table 4, dong "khong filter") da co tu truoc.

**Chua lam trong ban nay**: mo rong BDD100K/nuScenes/Waymo, dataset co va cham
thuc DAD/DoTA/CCD, robustness corruption test, trajectory forecasting upgrade -
can dataset/xac nhan tu Lan truoc.

**Tat ca output da xoa**. Chay lai tren Kaggle.

---

## Cap nhat lan 6 (robustness corruption test)

Them Phan 6: corrupt anh KITTI THAT (Gaussian noise, blur, low-light, fog) o 4
loai x 2 muc do + baseline sach = 9 cau hinh, chay lai detection+tracking+eval
THAT tren tung ban. Gioi han 2 sequence x 60 frame de kiem soat thoi gian chay -
tang ROBUSTNESS_SEQUENCES/ROBUSTNESS_MAX_FRAMES trong cell neu can manh hon.

**Chua lam (can quyet dinh/dataset tu Lan)**: nuScenes, BDD100K, DAD/DoTA/CCD,
MTR++/Social-GAT - xem giai thich ly do va de xuat thu tu trong tin nhan chinh.

**Tat ca output da xoa**. Chay lai tren Kaggle.

---

## Cap nhat lan 7 (Depth Anything V2 + nuScenes + BDD100K + CCD-discovery + trajectory)

1. **Depth Anything V2 Metric (VKITTI)**: them module depth pretrained (NeurIPS
   2024) song song voi pinhole+class-height cu, so sanh THAT MAE/RMSE tren cung
   sequence. Day la upgrade duoc chon vi co co so literature ro rang VA tan
   cong dung diem yeu co he thong (depth/velocity error) - KHONG phai "thay het
   code", chi them module va so sanh trung thuc.
2. **nuScenes**: generalization zero-shot cho distance (xap xi, ghi ro han che).
3. **BDD100K**: sanity check detection 2D (khong co nhan 3D).
4. **CCD**: cell discovery (chua danh gia, can xac nhan cau truc thuc truoc).
5. **DoTA, DAD**: KHONG tich hop (rao can truy cap du lieu thuc, xem chi tiet
   trong tin nhan chinh).
6. **Trajectory forecasting**: scoped xuong constant-velocity vs GRU don gian -
   MTR++/Social-GAT day du van la Future Work.

**Nguyen tac khong doi**: khong xoa pipeline cu, khong leakage, khong fake
fallback, bao cao trung thuc ca khi ket qua moi khong tot hon ket qua cu.

**Tat ca output da xoa**. Chay lai tren Kaggle (nho bat Internet trong Settings,
va them 3 dataset nuScenes/BDD100K/CCD truoc khi chay Phan 8).

In [ ]:
# ==============================================================================
# TUẦN 2 (ĐÃ SỬA) — 0. NẠP LẠI DỮ LIỆU TUẦN 1 + KIỂM TRA TÍNH TOÀN VẸN
# ==============================================================================
# Khác với bản gốc: KHÔNG fallback sang dữ liệu random/giả nếu thiếu biến.
# Nếu thiếu artifact nào của Tuần 1 -> raise lỗi rõ ràng và DỪNG LẠI.

import os
import time
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = Path("/kaggle/working/collision_prediction_project")
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "figures"
PAPER_DIR = PROJECT_ROOT / "paper"
for d in [TABLE_DIR, FIGURE_DIR, PAPER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REQUIRED_VARS = ["pred_ttc", "distance_pairs", "velocity_pairs", "clean",
                  "collision_events", "negative_windows", "KITTI_FPS",
                  "get_image_files", "read_kitti_calib", "estimate_depth_from_bbox"]
missing = [v for v in REQUIRED_VARS if v not in globals()]
if missing:
    raise RuntimeError(
        "Thieu cac bien/ham Tuan 1 sau trong kernel: " + ", ".join(missing) + ".\n"
        "Ban can chay DAY DU cac cell Tuan 1 (cell 0-17 cua notebook) trong cung session nay\n"
        "TRUOC KHI chay phan Tuan 2 ben duoi.\n"
        "Luu y: cell nay CO TINH khong co fallback du lieu gia - neu thay loi nay, do la dung,\n"
        "hay chay lai Tuan 1 thay vi bo qua."
    )

print("OK: Da co du", len(REQUIRED_VARS), "bien/ham Tuan 1 can thiet trong session.")
print(f"pred_ttc: {len(pred_ttc)} rows | distance_pairs: {len(distance_pairs)} rows | "
      f"velocity_pairs: {len(velocity_pairs)} rows | clean: {len(clean)} rows | "
      f"collision_events: {len(collision_events)} rows | negative_windows: {len(negative_windows)} rows")

In [ ]:
# ==============================================================================
# 19. GHEP DU LIEU GT + PRED O MUC FRAME-LEVEL (DA SUA)
# ==============================================================================
# SUA: Them tinh FPR = FP/(FP+TN) dung chuan, co TN that
GT_COLLISION_DIST_M = 7.0   # dinh nghia "tinh huong nguy hiem thuc" theo LiDAR
GT_COLLISION_TTC_S = 3.0    # khop voi "3.0s forward window" mo ta trong paper

eval_base = distance_pairs.merge(
    velocity_pairs[["sequence_id", "frame", "pred_track_id", "gt_vz_mps",
                    "pred_vz_raw_mps", "pred_vz_sg_mps"]],
    on=["sequence_id", "frame", "pred_track_id"],
    how="inner",
)

print("eval_base rows (co ca GT va pred matched):", len(eval_base))

# --- Ground truth risk: TU gt_z, gt_vz (LiDAR) - khong dung pred_z/pred_vz ---
eval_base["gt_approaching"] = eval_base["gt_vz_mps"] < 0
eval_base["gt_ttc"] = np.where(
    eval_base["gt_approaching"],
    eval_base["gt_z"] / np.maximum(-eval_base["gt_vz_mps"], 1e-6),
    np.inf,
)
eval_base["gt_is_risk"] = (
    (eval_base["gt_z"] < GT_COLLISION_DIST_M)
    & eval_base["gt_approaching"]
    & (eval_base["gt_ttc"] < GT_COLLISION_TTC_S)
)

# --- Predicted warning: TU pred_z_pinhole, pred_vz_sg (camera+IPM) - khong dung gt_* ---
eval_base["pred_approaching"] = eval_base["pred_vz_sg_mps"] < 0
eval_base["pred_ttc_eval"] = np.where(
    eval_base["pred_approaching"],
    eval_base["pred_z_pinhole"] / np.maximum(-eval_base["pred_vz_sg_mps"], 1e-6),
    np.inf,
)

n_risk = int(eval_base["gt_is_risk"].sum())
print(f"So sample (frame, track) co gt_is_risk=True theo nguong "
      f"{GT_COLLISION_DIST_M}m / {GT_COLLISION_TTC_S}s: {n_risk} / {len(eval_base)}")

if n_risk == 0:
    print("CANH BAO: 0 positive case voi nguong GT hien tai.")
    print("KITTI la du lieu lai xe AN TOAN (khong co va cham thuc), nen voi nguong qua chat")
    print("(vi du 1.5m nhu paper dang viet) rat de ra 0 positive thuc su.")
    print("-> Hay noi GT_COLLISION_DIST_M / GT_COLLISION_TTC_S cho hop ly voi du lieu,")
    print("   va ghi RO trong paper day la 'near-conflict proxy event', khong phai va cham thuc.")

In [ ]:
# ==============================================================================
# 2. QUET 8 NGUONG CANH BAO TTC - CONFUSION MATRIX THAT, EVENT-LEVEL (DEBOUNCE)
# ==============================================================================
# DA SUA (ban truoc la frame-level): chuyen sang WINDOW-LEVEL co debounce, tranh
# dem mot canh bao keo dai N frame lien tiep thanh N FP/TP rieng le. Co che: chia
# timeline cua moi track thanh window co do dai EVENT_WINDOW_FRAMES, dung phep OR
# (ANY) tren cac frame trong window de quyet dinh gt_is_risk/pred_warn cua window,
# roi tinh TP/FP/FN/TN o muc WINDOW. Day la cach debounce pho bien cho he thong canh bao.

EVENT_WINDOW_FRAMES = 10  # ~1s o KITTI_FPS=10 - co the doi tuy bao cao

ttc_thresholds = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]

TOTAL_RECORDING_SECONDS = sum(
    (g["frame"].max() - g["frame"].min() + 1) / KITTI_FPS
    for _, g in eval_base.groupby("sequence_id")
)
TOTAL_RECORDING_HOURS = TOTAL_RECORDING_SECONDS / 3600.0
print(f"Tong thoi luong danh gia: {TOTAL_RECORDING_SECONDS:.1f}s "
      f"({TOTAL_RECORDING_HOURS:.5f} gio)")
if TOTAL_RECORDING_SECONDS < 300:
    print("CANH BAO: tong thoi luong van con kha ngan (<5 phut). FAR/gio ngoai suy tu")
    print("khoang thoi gian ngan se kem on dinh thong ke. Neu MAX_FRAMES_PER_SEQ van")
    print("dang bi gioi han o Cell 1 (Tuan 1), hay kiem tra lai da sua thanh None chua.")


def windowed_confusion_matrix(eval_df, th, window_frames=EVENT_WINDOW_FRAMES):
    """
    Gop cac frame lien tiep cua CUNG MOT track thanh window do dai window_frames,
    roi tinh TP/FP/FN/TN o muc WINDOW (event-level, co debounce) - khong phai frame-level.
    """
    df = eval_df.copy()
    df["window_id"] = (df["frame"] // window_frames).astype(int)
    df["pred_warn"] = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)

    grouped = (
        df.groupby(["sequence_id", "pred_track_id", "window_id"])
        .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
        .reset_index()
    )

    TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    TN = int((~grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    return TP, FP, FN, TN, len(grouped)


roc_rows = []
for th in ttc_thresholds:
    TP, FP, FN, TN, n_windows = windowed_confusion_matrix(eval_base, th)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr_real = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    far_per_hour = FP / TOTAL_RECORDING_HOURS if TOTAL_RECORDING_HOURS > 0 else float("nan")

    roc_rows.append({
        "TTC_Threshold_s": th, "TP": TP, "FP": FP, "FN": FN, "TN": TN, "N_windows": n_windows,
        "FPR": round(fpr_real, 4), "TPR_Recall": round(recall, 4),
        "Precision": round(precision, 4), "F1": round(f1, 4),
        "FAR_per_hour": round(far_per_hour, 2),
    })

roc_points_df = pd.DataFrame(roc_rows)
roc_points_df.to_csv(TABLE_DIR / "table3_roc_sweep_FIXED.csv", index=False)
print("Da luu:", TABLE_DIR / "table3_roc_sweep_FIXED.csv")
print(f"(Debounce window = {EVENT_WINDOW_FRAMES} frame ~ {EVENT_WINDOW_FRAMES / KITTI_FPS:.1f}s)")
display(roc_points_df)

plt.figure(figsize=(7, 5.5))
plt.plot(roc_points_df["FPR"], roc_points_df["TPR_Recall"], marker="s", linewidth=2,
         label="He thong de xuat (FPR thuc, event-level)")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random guess")
for _, row in roc_points_df.iterrows():
    plt.annotate(f"{row['TTC_Threshold_s']}s",
                 (row["FPR"] + 0.01, row["TPR_Recall"] - 0.02), fontsize=9)
plt.xlabel("False Positive Rate = FP / (FP + TN)  [window-level]")
plt.ylabel("True Positive Rate / Recall")
plt.xlim([-0.02, 1.02]); plt.ylim([-0.02, 1.02])
plt.title(f"Collision Prediction ROC Curve - event-level (debounce {EVENT_WINDOW_FRAMES} frame)")
plt.legend(loc="lower right")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
fig4_path = FIGURE_DIR / "roc_curve_FIXED.png"
plt.savefig(fig4_path, dpi=300)
plt.show()
print("Da luu:", fig4_path)

In [ ]:
# ==============================================================================
# 21. MONTE CARLO UNCERTAINTY CHO VAN TOC - KHONG TRON GROUND TRUTH (DA SUA)
# ==============================================================================
# SUA: Xoa dong "corrected_v = final_v*(1-0.15) + gt_v_array[idx]*0.15"
# Chi nhieu hoa CHINH gia tri du doan, KHONG bao gio doc gt_v

def monte_carlo_velocity_uncertainty(pred_v_raw_array, n_samples=100,
                                     pixel_jitter_std=1.0, relative_noise_std=0.05, seed=42):
    """
    Lan truyen bat dinh bang cach nhieu hoa CHINH gia tri du doan,
    KHONG su dung nhan thuc o buoc nao.
    """
    rng = np.random.default_rng(seed)
    pred_v_raw_array = np.asarray(pred_v_raw_array, dtype=float)
    means, stds = [], []
    
    for v in pred_v_raw_array:
        noise_scale = abs(v) * relative_noise_std + pixel_jitter_std * 0.1
        samples = v + rng.normal(0, max(noise_scale, 1e-3), n_samples)
        means.append(np.mean(samples))
        stds.append(np.std(samples))
    
    return np.array(means), np.array(stds)

raw_pred_velocity_mps = clean["pred_vz_sg_mps"].values
gt_velocity_mps = clean["gt_vz_mps"].values

mc_mean_mps, mc_std_mps = monte_carlo_velocity_uncertainty(raw_pred_velocity_mps)

mae_raw_kmh = float(mean_absolute_error(gt_velocity_mps, raw_pred_velocity_mps)) * 3.6
mae_mc_kmh = float(mean_absolute_error(gt_velocity_mps, mc_mean_mps)) * 3.6

print(f"Velocity MAE - Savitzky-Golay (baseline thuc): {mae_raw_kmh:.2f} km/h")
print(f"Velocity MAE - sau Monte Carlo smoothing (KHONG dung GT): {mae_mc_kmh:.2f} km/h")
print("Luu y: neu mae_mc_kmh khong cai thien nhieu so voi mae_raw_kmh, DO LA KET QUA THAT.")

# Ve do thi
sample_n = min(60, len(clean))
plt.figure(figsize=(8.5, 4.5))
x = np.arange(sample_n)

plt.plot(x, gt_velocity_mps[:sample_n] * 3.6, linewidth=2.5, label="Ground Truth (LiDAR)")
plt.plot(x, raw_pred_velocity_mps[:sample_n] * 3.6, linestyle=":", alpha=0.6,
         label=f"Savitzky-Golay (MAE {mae_raw_kmh:.2f} km/h)")
plt.plot(x, mc_mean_mps[:sample_n] * 3.6, linestyle="--",
         label=f"Monte Carlo, no leakage (MAE {mae_mc_kmh:.2f} km/h)")
plt.fill_between(x,
                 (mc_mean_mps[:sample_n] - 1.96 * mc_std_mps[:sample_n]) * 3.6,
                 (mc_mean_mps[:sample_n] + 1.96 * mc_std_mps[:sample_n]) * 3.6,
                 alpha=0.15, label="95% CI")

plt.xlabel("Sample index")
plt.ylabel("Velocity (km/h)")
plt.title("Velocity Estimation - Monte Carlo Uncertainty (No GT Leakage)")
plt.legend()
plt.tight_layout()

fig2_path = FIGURE_DIR / "velocity_mc_FIXED.png"
plt.savefig(fig2_path, dpi=300)
plt.show()

print("Da luu:", fig2_path)

In [ ]:
# ==============================================================================
# 4. ABLATION STUDY - TINH THAT TU TOGGLE HAU XU LY, EVENT-LEVEL (DUNG HAM O CELL TRUOC)
# ==============================================================================
# - Doi FILTER (co/khong Savitzky-Golay): tinh duoc ngay tu velocity_pairs da co
#   (cung mot bo detection, chi khac hau xu ly) -> KHONG can chay lai YOLO.
# - Doi DETECTOR (YOLOv8x vs YOLOv11x): can chay lai TOAN BO Tuan 1 voi
#   YOLO_WEIGHTS="yolo8x.pt" va luu ket qua rieng -> de ro NOT_YET_RUN, KHONG dien so gia.
# - Dung CUNG ham windowed_confusion_matrix() (event-level, debounce) da dinh nghia
#   o cell ROC truoc, de Table 3 va Table 4 nhat quan voi nhau.


def f1_far_for_threshold(eval_df, th, total_hours, window_frames=EVENT_WINDOW_FRAMES):
    TP, FP, FN, TN, _ = windowed_confusion_matrix(eval_df, th, window_frames)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    far = FP / total_hours if total_hours > 0 else float("nan")
    return f1, far


BEST_TH = float(roc_points_df.sort_values("F1", ascending=False).iloc[0]["TTC_Threshold_s"])
print(f"Nguong tot nhat theo F1 tu bang ROC vua tinh: {BEST_TH}s")

# --- CFG: YOLOv11x + ByteTrack, KHONG filter (dung pred_vz_raw_mps) ---
eval_raw_v = distance_pairs.merge(
    velocity_pairs[["sequence_id", "frame", "pred_track_id", "gt_vz_mps",
                     "pred_vz_raw_mps", "pred_vz_sg_mps"]],
    on=["sequence_id", "frame", "pred_track_id"], how="inner",
)
eval_raw_v["gt_approaching"] = eval_raw_v["gt_vz_mps"] < 0
eval_raw_v["gt_ttc"] = np.where(
    eval_raw_v["gt_approaching"],
    eval_raw_v["gt_z"] / np.maximum(-eval_raw_v["gt_vz_mps"], 1e-6), np.inf)
eval_raw_v["gt_is_risk"] = ((eval_raw_v["gt_z"] < GT_COLLISION_DIST_M)
                            & eval_raw_v["gt_approaching"]
                            & (eval_raw_v["gt_ttc"] < GT_COLLISION_TTC_S))
eval_raw_v["pred_approaching"] = eval_raw_v["pred_vz_raw_mps"] < 0
eval_raw_v["pred_ttc_eval"] = np.where(
    eval_raw_v["pred_approaching"],
    eval_raw_v["pred_z_pinhole"] / np.maximum(-eval_raw_v["pred_vz_raw_mps"], 1e-6), np.inf)
f1_no_filter, far_no_filter = f1_far_for_threshold(eval_raw_v, BEST_TH, TOTAL_RECORDING_HOURS)

# --- CFG: YOLOv11x + ByteTrack + Savitzky-Golay (eval_base da tinh truoc do) ---
f1_sg, far_sg = f1_far_for_threshold(eval_base, BEST_TH, TOTAL_RECORDING_HOURS)

ablation_rows = [
    {"Config": "YOLOv8x + ByteTrack (chua chay lai)", "Detector": "YOLOv8x",
     "Filter": "N/A", "F1": "NOT_YET_RUN", "FAR_per_hour": "NOT_YET_RUN"},
    {"Config": "YOLOv11x + ByteTrack, khong filter", "Detector": "YOLOv11x",
     "Filter": "None (finite-diff)", "F1": round(f1_no_filter, 3), "FAR_per_hour": round(far_no_filter, 2)},
    {"Config": "YOLOv11x + ByteTrack + Savitzky-Golay", "Detector": "YOLOv11x",
     "Filter": "Savitzky-Golay", "F1": round(f1_sg, 3), "FAR_per_hour": round(far_sg, 2)},
]
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(TABLE_DIR / "table4_ablation_FIXED.csv", index=False)
display(ablation_df)
print("\nDong YOLOv8x: phai chay lai Tuan 1 voi YOLO_WEIGHTS='yolo8x.pt' roi dien so THAT vao day.")
print("KHONG dien so gia vao o NOT_YET_RUN truoc khi nop paper.")

In [ ]:
# ==============================================================================
# 23. CONTEXT-AWARE HEURISTICS (THEM MOI)
# ==============================================================================
# Them 2 heuristics giam bao sai:
# (1) trajectory monotonicity: d(TTC)/dt < 0 trong 5 frame lien tiep
# (2) lane corridor check: |pred_x| < 2.0m (vat the nam trong lane ego)

TRAJ_MONOTONIC_WINDOW = 5
LANE_CORRIDOR_HALF_WIDTH_M = 2.0

# Tinh monotonicity
pttc = pred_ttc.sort_values(["sequence_id", "pred_track_id", "frame"]).copy()
pttc["ttc_diff"] = pttc.groupby(["sequence_id", "pred_track_id"])["ttc"].diff()
pttc["ttc_decreasing"] = pttc["ttc_diff"] < 0

MONOTONIC_MIN_FRACTION = 0.8  # >=4/5 frame giam la du
pttc["monotonic_ok"] = (
    pttc.groupby(["sequence_id", "pred_track_id"])["ttc_decreasing"]
    .transform(lambda s: s.rolling(TRAJ_MONOTONIC_WINDOW, min_periods=TRAJ_MONOTONIC_WINDOW)
               .apply(lambda w: bool(w.mean() >= MONOTONIC_MIN_FRACTION)))
)
pttc["monotonic_ok"] = pttc["monotonic_ok"].fillna(False).astype(bool)

# Tinh lane corridor
pttc["lane_corridor_ok"] = pttc["pred_x_smooth"].abs() <= LANE_CORRIDOR_HALF_WIDTH_M

# Merge vao eval_base
heuristic_cols = pttc[["sequence_id", "frame", "pred_track_id", "monotonic_ok", "lane_corridor_ok"]]
eval_heur = eval_base.merge(heuristic_cols, on=["sequence_id", "frame", "pred_track_id"], how="left")
eval_heur["monotonic_ok"] = eval_heur["monotonic_ok"].fillna(False).astype(bool)
eval_heur["lane_corridor_ok"] = eval_heur["lane_corridor_ok"].fillna(False).astype(bool)

# Ket hop 2 heuristics bang OR (de nhe hon)
eval_heur["heuristics_passed"] = eval_heur["monotonic_ok"] | eval_heur["lane_corridor_ok"]

print(f"So sample dat monotonic_ok: {int(eval_heur['monotonic_ok'].sum())} / {len(eval_heur)}")
print(f"So sample dat lane_corridor_ok: {int(eval_heur['lane_corridor_ok'].sum())} / {len(eval_heur)}")
print(f"So sample dat CA HAI heuristics (OR): {int(eval_heur['heuristics_passed'].sum())} / {len(eval_heur)}")

# Tinh lai F1 voi heuristics
def windowed_confusion_matrix_heur(eval_df, th, window_frames=10, apply_heuristics=False):
    df = eval_df.copy()
    df["window_id"] = (df["frame"] // window_frames).astype(int)
    pred_warn = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
    
    if apply_heuristics:
        pred_warn = pred_warn & df["heuristics_passed"]
    
    df["pred_warn"] = pred_warn
    
    grouped = (
        df.groupby(["sequence_id", "pred_track_id", "window_id"])
        .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
        .reset_index()
    )
    
    TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    TN = int((~grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    
    return TP, FP, FN, TN

# Tim nguong tot nhat
BEST_TH = 1.0  # nguong tot nhat tu Cell 20

TP_no, FP_no, FN_no, TN_no = windowed_confusion_matrix_heur(eval_base, BEST_TH, apply_heuristics=False)
TP_heur, FP_heur, FN_heur, TN_heur = windowed_confusion_matrix_heur(eval_heur, BEST_TH, apply_heuristics=True)

f1_no = 2 * (TP_no / (TP_no + FP_no)) * (TP_no / (TP_no + FN_no)) / ((TP_no / (TP_no + FP_no)) + (TP_no / (TP_no + FN_no))) if (TP_no + FP_no) > 0 and (TP_no + FN_no) > 0 else 0
f1_heur = 2 * (TP_heur / (TP_heur + FP_heur)) * (TP_heur / (TP_heur + FN_heur)) / ((TP_heur / (TP_heur + FP_heur)) + (TP_heur / (TP_heur + FN_heur))) if (TP_heur + FP_heur) > 0 and (TP_heur + FN_heur) > 0 else 0

print(f"\nF1 KHONG heuristics: {f1_no:.3f}")
print(f"F1 CO heuristics: {f1_heur:.3f}")

if f1_heur > f1_no:
    print("-> Heuristics CAI THIEN F1 thuc su.")
else:
    print("-> Heuristics CHUA cai thien F1 voi nguong hien tai.")

In [ ]:
# ==============================================================================
# 4b. CONTEXT-AWARE HEURISTICS (THEO MO TA PAPER, MUC III.D.2 / Eq.17) - THUC TE
# ==============================================================================
# Paper mo ta 3 heuristics giam bao sai: (1) future IoU overlap, (2) vertical
# separation |yi-yj|<0.5m, (3) trajectory monotonicity d(TTC)/dt<0 trong 5 frame.
#
# QUAN TRONG - gioi han kien truc can ghi ro trong paper:
# Paper mo ta he thong PAIRWISE (2 doi tuong i,j bat ky), nhung notebook nay
# dang cai dat kieu EGO-CENTRIC (TTC tu camera/ego toi 1 track, chi theo truc Z),
# khong co doi tuong j thu hai. Vi vay:
#   - Heuristic (3) Trajectory monotonicity: ap dung DUNG nhu mo ta (thuoc tinh
#     cua 1 track qua thoi gian, khong can doi tuong thu hai).
#   - Heuristic (1) duoc thay bang "lane corridor check": doi tuong phai nam
#     trong hanh lang lan duong cua ego (|pred_x| nho) moi tinh la "tren duong
#     va cham", thay cho IoU tuong lai giua 2 bbox (can kien truc pairwise).
#   - Heuristic (2) (vertical separation giua 2 doi tuong) KHONG co dang tuong
#     duong hop ly trong kien truc ego-centric hien tai -> BO QUA o day, can ghi
#     ro trong paper (Section V/VI) la huong mo rong can chuyen sang kien truc
#     pairwise object-object that.

TRAJ_MONOTONIC_WINDOW = 5          # khop "5 frame lien tiep" trong paper
LANE_CORRIDOR_HALF_WIDTH_M = 2.0   # nguong hanh lang lan duong - co the chinh lai

pttc = pred_ttc.sort_values(["sequence_id", "pred_track_id", "frame"]).copy()
pttc["ttc_diff"] = pttc.groupby(["sequence_id", "pred_track_id"])["ttc"].diff()
pttc["ttc_decreasing"] = pttc["ttc_diff"] < 0
MONOTONIC_MIN_FRACTION = 0.8  # DA NOI: >=4/5 frame giam la du, khong can CA 5 frame (qua chat)
pttc["monotonic_ok"] = (
    pttc.groupby(["sequence_id", "pred_track_id"])["ttc_decreasing"]
    .transform(lambda s: s.rolling(TRAJ_MONOTONIC_WINDOW, min_periods=TRAJ_MONOTONIC_WINDOW)
               .apply(lambda w: bool(w.mean() >= MONOTONIC_MIN_FRACTION)))
)
pttc["monotonic_ok"] = pttc["monotonic_ok"].fillna(False).astype(bool)
pttc["lane_corridor_ok"] = pttc["pred_x_smooth"].abs() <= LANE_CORRIDOR_HALF_WIDTH_M

heuristic_cols = pttc[["sequence_id", "frame", "pred_track_id", "monotonic_ok", "lane_corridor_ok"]]
eval_heur = eval_base.merge(heuristic_cols, on=["sequence_id", "frame", "pred_track_id"], how="left")
eval_heur["monotonic_ok"] = eval_heur["monotonic_ok"].fillna(False).astype(bool)
eval_heur["lane_corridor_ok"] = eval_heur["lane_corridor_ok"].fillna(False).astype(bool)
# DA SUA: AND qua chat (0.09% sample dat ca 2) -> doi sang OR, kem in ro ca 2 phuong
# an (AND/OR) de Lan tu chon phuong an nao bao cao trong paper.
eval_heur["heuristics_passed_AND"] = eval_heur["monotonic_ok"] & eval_heur["lane_corridor_ok"]
eval_heur["heuristics_passed_OR"] = eval_heur["monotonic_ok"] | eval_heur["lane_corridor_ok"]
HEURISTIC_COMBINE_MODE = "OR"  # doi thanh "AND" neu muon chat hon
eval_heur["heuristics_passed"] = (
    eval_heur["heuristics_passed_OR"] if HEURISTIC_COMBINE_MODE == "OR"
    else eval_heur["heuristics_passed_AND"]
)
print(f"heuristics_passed_AND: {int(eval_heur['heuristics_passed_AND'].sum())} / {len(eval_heur)}")
print(f"heuristics_passed_OR : {int(eval_heur['heuristics_passed_OR'].sum())} / {len(eval_heur)}")
print(f"Dang dung che do: {HEURISTIC_COMBINE_MODE}")

print(f"So sample dat monotonic_ok      : {int(eval_heur['monotonic_ok'].sum())} / {len(eval_heur)}")
print(f"So sample dat lane_corridor_ok  : {int(eval_heur['lane_corridor_ok'].sum())} / {len(eval_heur)}")
print(f"So sample dat CA HAI heuristics : {int(eval_heur['heuristics_passed'].sum())} / {len(eval_heur)}")


def windowed_confusion_matrix_heur(eval_df, th, window_frames=EVENT_WINDOW_FRAMES, apply_heuristics=False):
    df = eval_df.copy()
    df["window_id"] = (df["frame"] // window_frames).astype(int)
    pred_warn = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
    if apply_heuristics:
        pred_warn = pred_warn & df["heuristics_passed"]
    df["pred_warn"] = pred_warn

    grouped = (
        df.groupby(["sequence_id", "pred_track_id", "window_id"])
        .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
        .reset_index()
    )
    TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    TN = int((~grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    return TP, FP, FN, TN


def metrics_row(th, TP, FP, FN, TN, total_hours):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    far = FP / total_hours if total_hours > 0 else float("nan")
    return {"TTC_Threshold_s": th, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "FPR": round(fpr, 4), "TPR_Recall": round(recall, 4),
            "Precision": round(precision, 4), "F1": round(f1, 4),
            "FAR_per_hour": round(far, 2)}


rows_with_heur = []
for th in ttc_thresholds:
    TP, FP, FN, TN = windowed_confusion_matrix_heur(eval_heur, th, apply_heuristics=True)
    rows_with_heur.append(metrics_row(th, TP, FP, FN, TN, TOTAL_RECORDING_HOURS))

roc_with_heuristics_df = pd.DataFrame(rows_with_heur)
roc_with_heuristics_df.to_csv(TABLE_DIR / "table3_roc_with_heuristics_FIXED.csv", index=False)

print("\n--- SO SANH: KHONG heuristics (truoc) vs CO heuristics (sau) ---")
compare_df = roc_points_df[["TTC_Threshold_s", "Precision", "F1", "FAR_per_hour"]].rename(
    columns={"Precision": "Precision_no_heur", "F1": "F1_no_heur", "FAR_per_hour": "FAR_no_heur"}
).merge(
    roc_with_heuristics_df[["TTC_Threshold_s", "Precision", "F1", "FAR_per_hour"]].rename(
        columns={"Precision": "Precision_with_heur", "F1": "F1_with_heur", "FAR_per_hour": "FAR_with_heur"}
    ),
    on="TTC_Threshold_s",
)
display(compare_df)

best_no = roc_points_df.sort_values("F1", ascending=False).iloc[0]
best_with = roc_with_heuristics_df.sort_values("F1", ascending=False).iloc[0]
print(f"\nF1 tot nhat KHONG heuristics: {best_no['F1']:.3f} (threshold {best_no['TTC_Threshold_s']}s)")
print(f"F1 tot nhat CO heuristics   : {best_with['F1']:.3f} (threshold {best_with['TTC_Threshold_s']}s)")
if best_with["F1"] > best_no["F1"]:
    print("-> Heuristics CAI THIEN F1 thuc su. Co the bao cao trong paper nhu mot ablation tich cuc.")
else:
    print("-> Heuristics CHUA cai thien F1 voi nguong LANE_CORRIDOR_HALF_WIDTH_M hien tai.")
    print("   Thu chinh LANE_CORRIDOR_HALF_WIDTH_M hoac TRAJ_MONOTONIC_WINDOW roi chay lai cell nay,")
    print("   hoac bao cao trung thuc ca 2 truong hop trong Table 4 (khong an so xau).")

ablation_df = pd.concat([
    ablation_df,
    pd.DataFrame([{
        "Config": f"YOLOv11x + ByteTrack + SG + Heuristics (monotonic+lane, w={TRAJ_MONOTONIC_WINDOW})",
        "Detector": "YOLOv11x", "Filter": "Savitzky-Golay + Heuristics",
        "F1": round(float(best_with["F1"]), 3), "FAR_per_hour": round(float(best_with["FAR_per_hour"]), 2),
    }])
], ignore_index=True)
ablation_df.to_csv(TABLE_DIR / "table4_ablation_FIXED.csv", index=False)
display(ablation_df)

## TABLE 4 — SỬA TRIỆT ĐỂ THEO RESEARCH INTEGRITY AUDITOR

### Vấn đề được phát hiện (đọc trực tiếp từ notebook outputs)

| # | Vấn đề | Severity |
|---|---|---|
| A | Paper Row "Unfiltered" (F1=0.3654, FAR=4125.10) ≠ cell 28 (F1=0.372, FAR=2721.44) | Critical |
| B | Paper Row "Rolling Median" (F1=0.3692, FAR=3982.50): **không có cell nào tính** | Critical |
| C | Paper text "FAR=10.0/F1=0.701": **không có evidence** — heuristics thật cho F1≈0.397, FAR≈988.82 | Critical |
| D | Threshold không nhất quán: ablation dùng BEST_TH=1.0s, "Proposed Framework" lấy từ Table 3 tại 1.5s | Major |

### Cách sửa

1. **Cố định ABLATION_TH = 1.5s** cho tất cả dòng ablation → nhất quán với "Proposed Framework"
2. **Tính mới Rolling Median velocity** từ `distance_pairs` (rolling window=3 center median)
3. **Cross-validate**: SG tại 1.5s **phải** cho ra đúng F1=0.3743, FAR=3859.02 (từ Table 3) → nếu khớp thì Table 4 nhất quán nội bộ
4. **Heuristics row**: dùng `windowed_confusion_matrix_heur(apply_heuristics=True)` đúng hàm, tại 1.5s → cho số thật (≠ 0.701/10.0)
5. Lưu `table4_paper_VERIFIED.csv` — đây là file duy nhất được dùng để điền vào paper

### Số liệu thật thay thế trong paper text

Thay toàn bộ câu "FAR to **10.0 alerts/hour** and raises F1 to **0.701**" bằng số từ Row 4 output của cell dưới.


In [ ]:
# ==============================================================================
# TABLE 4 — SỬA TRIỆT ĐỂ (4 dòng, ABLATION_TH=1.5s, cross-validated)
# ==============================================================================
# Vấn đề A: Paper "Unfiltered" F1=0.3654/FAR=4125.10 ≠ notebook F1=0.372/FAR=2721.44
# Vấn đề B: Paper "Rolling Median" F1=0.3692/FAR=3982.50 → chưa có cell nào tính
# Vấn đề C: Paper text FAR=10.0/F1=0.701 → FABRICATED (heuristics thật ≈ 0.397/988.82)
# Vấn đề D: Threshold inconsistent (ablation 1.0s vs Proposed Framework 1.5s từ Table 3)
# Fix: dùng ABLATION_TH=1.5s cho tất cả, tính Rolling mới, cross-validate vs Table 3

ABLATION_TH = 1.5   # cố định 1.5s — khớp row "Proposed Framework" đã verify trong Table 3
ROLLING_WINDOW = 3  # rolling(3) center median, symmetric — standard choice cho trajectory smoothing

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 1: Tính Rolling Median velocity từ distance_pairs
#         (chưa có trước đây — đây là cấu hình còn thiếu trong paper Table 4)
# ─────────────────────────────────────────────────────────────────────────────
roll_rows = []
for (seq, tid), g in distance_pairs.sort_values(
    ["sequence_id", "pred_track_id", "frame"]
).groupby(["sequence_id", "pred_track_id"], sort=False):
    g = g.drop_duplicates(subset=["frame"], keep="first")          .sort_values("frame").reset_index(drop=True)
    if len(g) < 3:
        continue
    frames  = g["frame"].astype(int).values
    z_raw   = g["pred_z_pinhole"].astype(float).values
    # Rolling(3) center median — khác với SG (polynomial bậc 2 window=7)
    z_roll  = pd.Series(z_raw).rolling(ROLLING_WINDOW, center=True, min_periods=1)                                .median().values
    for i in range(1, len(g)):
        dframe = frames[i] - frames[i - 1]
        if dframe <= 0:
            continue
        dt = dframe / KITTI_FPS
        roll_rows.append({
            "sequence_id": seq,
            "frame": int(frames[i]),
            "pred_track_id": int(tid),
            "pred_vz_roll_mps": float((z_roll[i] - z_roll[i - 1]) / dt),
        })

roll_vel_df = pd.DataFrame(roll_rows)
print(f"Rolling velocity rows: {len(roll_vel_df)} "
      f"(so sanh raw: {len(eval_raw_v)}, SG: {len(eval_base)})")

# Xây dựng eval_rolling — cùng GT risk columns với eval_raw_v, chỉ thay velocity
eval_rolling = eval_raw_v.drop(columns=["pred_approaching", "pred_ttc_eval"])     .merge(roll_vel_df, on=["sequence_id", "frame", "pred_track_id"], how="inner")
eval_rolling["pred_approaching"] = eval_rolling["pred_vz_roll_mps"] < 0
eval_rolling["pred_ttc_eval"] = np.where(
    eval_rolling["pred_approaching"],
    eval_rolling["pred_z_pinhole"] / np.maximum(-eval_rolling["pred_vz_roll_mps"], 1e-6),
    np.inf,
)
print(f"eval_rolling rows: {len(eval_rolling)}")

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 2: Hàm tính F1/FAR cho Heuristics (phải dùng windowed_confusion_matrix_heur
#         với apply_heuristics=True — không thể dùng windowed_confusion_matrix thường)
# ─────────────────────────────────────────────────────────────────────────────
def f1_far_heur(eval_df, th, total_hours, window_frames=EVENT_WINDOW_FRAMES):
    # windowed_confusion_matrix_heur tra ve 4 gia tri (TP,FP,FN,TN), KHONG co n_windows
    # (khac windowed_confusion_matrix thuong tra ve 5 gia tri)
    TP, FP, FN, TN = windowed_confusion_matrix_heur(eval_df, th, window_frames,
                                                      apply_heuristics=True)
    p = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    r = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    far = FP / total_hours if total_hours > 0 else float("nan")
    return f1, far, TP, FP, FN, TN

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 3: Tính F1/FAR tại ABLATION_TH=1.5s cho TẤT CẢ 4 cấu hình
# ─────────────────────────────────────────────────────────────────────────────
f1_raw,  far_raw  = f1_far_for_threshold(eval_raw_v,  ABLATION_TH, TOTAL_RECORDING_HOURS)
f1_roll, far_roll = f1_far_for_threshold(eval_rolling, ABLATION_TH, TOTAL_RECORDING_HOURS)
f1_sg,   far_sg   = f1_far_for_threshold(eval_base,   ABLATION_TH, TOTAL_RECORDING_HOURS)
f1_heur, far_heur, TP_h, FP_h, FN_h, TN_h = f1_far_heur(
    eval_heur, ABLATION_TH, TOTAL_RECORDING_HOURS)

print()
print("=== KẾT QUẢ 4 CẤU HÌNH tại ABLATION_TH=1.5s ===")
print(f"  [1] Unfiltered (Raw FD) : F1={f1_raw:.4f}, FAR={far_raw:.2f}")
print(f"  [2] Rolling Median (w=3): F1={f1_roll:.4f}, FAR={far_roll:.2f}")
print(f"  [3] SG (Proposed)       : F1={f1_sg:.4f},  FAR={far_sg:.2f}")
print(f"  [4] SG + Heuristics     : F1={f1_heur:.4f}, FAR={far_heur:.2f}")
print(f"      (Heuristics TP={TP_h} FP={FP_h} FN={FN_h} TN={TN_h})")

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 4: Cross-validate Row 3 (SG tại 1.5s) vs Table 3 — kiểm tra nhất quán nội bộ
#         NẾU không khớp → Table 4 có vấn đề cấu trúc, DỪNG và báo lỗi
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=== CROSS-VALIDATE: SG tại 1.5s vs Table 3 (nguồn đã verified) ===")
try:
    t3_f1  = float(roc_points_df.loc[
        roc_points_df["TTC_Threshold_s"] == 1.5, "F1"].iloc[0])
    t3_far = float(roc_points_df.loc[
        roc_points_df["TTC_Threshold_s"] == 1.5, "FAR_per_hour"].iloc[0])
    print(f"  Table 3 TTC=1.5s (đã verify): F1={t3_f1:.4f}, FAR={t3_far:.2f}")
    print(f"  Ablation SG vừa tính       : F1={f1_sg:.4f}, FAR={far_sg:.2f}")
    delta_f1  = abs(f1_sg  - t3_f1)
    delta_far = abs(far_sg - t3_far)
    if delta_f1 < 0.001 and delta_far < 1.0:
        print(f"  KIEM TRA OK (delta_F1={delta_f1:.5f}, delta_FAR={delta_far:.3f})")
        print("  -> Table 4 Row 3 nhat quan voi Table 3 — co the bao cao.")
    else:
        print(f"  CANH BAO: delta_F1={delta_f1:.4f}, delta_FAR={delta_far:.2f}")
        print("  eval_base (SG) va roc_points_df co the dung khac nhau GT_COLLISION_DIST_M")
        print("  hoac khac cach tinh windowed_confusion_matrix. Kiem tra lai cell 25 va 26.")
except Exception as e:
    print("  LOI khi truy cap roc_points_df:", e)

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 5: Tạo Table 4 chính thức cho paper
# ─────────────────────────────────────────────────────────────────────────────
table4_verified = pd.DataFrame([
    {
        "Pipeline_Modality_Configuration": "YOLOv11x + ByteTrack (Unfiltered)",
        "Detector_Core": "YOLOv11x",
        "Trajectory_Filter_Mode": "Raw Finite Difference",
        "F1_Score": round(f1_raw,  4),
        "FAR_per_hour": round(far_raw,  2),
        "TTC_Threshold_s": ABLATION_TH,
        "Evidence_source": "eval_raw_v, pred_vz_raw_mps, cell 28",
    },
    {
        "Pipeline_Modality_Configuration": "YOLOv11x + ByteTrack + Rolling Filter",
        "Detector_Core": "YOLOv11x",
        "Trajectory_Filter_Mode": f"Rolling Median Smooth (window={ROLLING_WINDOW})",
        "F1_Score": round(f1_roll, 4),
        "FAR_per_hour": round(far_roll, 2),
        "TTC_Threshold_s": ABLATION_TH,
        "Evidence_source": "eval_rolling, pred_vz_roll_mps, cell nay (Table4_fix)",
    },
    {
        "Pipeline_Modality_Configuration": "Proposed Framework (TTC-Window)",
        "Detector_Core": "YOLOv11x",
        "Trajectory_Filter_Mode": "Savitzky-Golay + Debounce",
        "F1_Score": round(f1_sg,   4),
        "FAR_per_hour": round(far_sg,   2),
        "TTC_Threshold_s": ABLATION_TH,
        "Evidence_source": "eval_base (SG), cell 25 + cross-validated vs Table 3",
    },
    {
        "Pipeline_Modality_Configuration": "Proposed + Visual Heuristics",
        "Detector_Core": "YOLOv11x",
        "Trajectory_Filter_Mode": "SG + Heuristics (monotonic OR lane) + Debounce",
        "F1_Score": round(f1_heur, 4),
        "FAR_per_hour": round(far_heur, 2),
        "TTC_Threshold_s": ABLATION_TH,
        "Evidence_source": "eval_heur, OR(monotonic_ok, lane_corridor_ok), cell 29-30",
    },
])

print()
print("=== TABLE 4 CHÍNH THỨC (4 DÒNG THẬT — dùng để điền vào paper) ===")
display(table4_verified[["Pipeline_Modality_Configuration",
                           "Trajectory_Filter_Mode", "F1_Score", "FAR_per_hour"]])

table4_verified.to_csv(TABLE_DIR / "table4_paper_VERIFIED.csv", index=False)
print("Da luu:", TABLE_DIR / "table4_paper_VERIFIED.csv")

# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 6: Số liệu thay thế trong paper TEXT (thay FAR=10.0/F1=0.701 bằng số thật)
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("SO LIEU THAT de viet lai paper text (thay FAR=10.0/F1=0.701 FABRICATED):")
print("=" * 70)
print()
print("Cu (fabricated, can xoa hoan toan):")
print('  "further lowers FAR to 10.0 alerts/hour and raises F1 to 0.701"')
print()
print("Moi (verified, dung du lieu thuc te):")
print(f'  "further lowers FAR to {far_heur:.0f} alerts/hour'
      f' and raises F1 to {f1_heur:.4f},"')
print(f'  "the best result obtained across the four configurations tested."')
print()
print("Ghi chu them trong paper (bat buoc de tranh reviewer reject):")
print(f"  TTC threshold = {ABLATION_TH}s (consistent with Table 3 knee-point)")
print(f"  Heuristics = OR(trajectory monotonicity, lane corridor check)")
print(f"  Evaluation = event-level windowed confusion matrix (window={EVENT_WINDOW_FRAMES} frames)")


In [ ]:
# ==============================================================================
# 4c. ABLATION YOLOv8x - CHAY THAT (THAY NOT_YET_RUN BANG SO THUC)
# ==============================================================================
# Chay lai detection+tracking+distance+velocity+eval voi YOLOv8x thay YOLOv11x,
# GIU NGUYEN moi thu khac (ByteTrack, IPM, Savitzky-Golay, nguong GT, debounce
# window) de so sanh cong bang.
# CANH BAO: cell nay chay lai YOLO inference tu dau tren TOAN BO SELECTED_SEQUENCES
# -> ton GPU-hour tuong duong lan chay Tuan 1 ban dau. Kiem tra quota Kaggle truoc.

_YOLO_WEIGHTS_BACKUP = YOLO_WEIGHTS
YOLO_WEIGHTS = "yolov8x.pt"
print(f"Chuyen tam thoi YOLO_WEIGHTS -> {YOLO_WEIGHTS} de chay ablation...")

pred_v8_dfs = []
for seq_id in SELECTED_SEQUENCES:
    pred_seq_v8, _rt = run_yolo_bytetrack_on_sequence(seq_id)
    if len(pred_seq_v8):
        pred_v8_dfs.append(pred_seq_v8)

YOLO_WEIGHTS = _YOLO_WEIGHTS_BACKUP
print(f"Da khoi phuc YOLO_WEIGHTS -> {YOLO_WEIGHTS}")

pred_v8_all = pd.concat(pred_v8_dfs, ignore_index=True) if pred_v8_dfs else pd.DataFrame()
print(f"YOLOv8x pred rows: {len(pred_v8_all)} (so sanh YOLOv11x: {len(pred_all)})")

distance_dfs_v8 = []
for seq_id in SELECTED_SEQUENCES:
    ddf = match_pred_to_gt_for_distance(seq_id, pred_v8_all, iou_thr=EVAL_IOU_THRESHOLD)
    if len(ddf):
        distance_dfs_v8.append(ddf)
distance_pairs_v8 = pd.concat(distance_dfs_v8, ignore_index=True) if distance_dfs_v8 else pd.DataFrame()

velocity_pairs_v8 = compute_velocity_pairs(distance_pairs_v8, fps=KITTI_FPS)

eval_v8 = distance_pairs_v8.merge(
    velocity_pairs_v8[["sequence_id", "frame", "pred_track_id", "gt_vz_mps", "pred_vz_sg_mps"]],
    on=["sequence_id", "frame", "pred_track_id"], how="inner",
)
eval_v8["gt_approaching"] = eval_v8["gt_vz_mps"] < 0
eval_v8["gt_ttc"] = np.where(eval_v8["gt_approaching"],
                              eval_v8["gt_z"] / np.maximum(-eval_v8["gt_vz_mps"], 1e-6), np.inf)
eval_v8["gt_is_risk"] = ((eval_v8["gt_z"] < GT_COLLISION_DIST_M)
                         & eval_v8["gt_approaching"]
                         & (eval_v8["gt_ttc"] < GT_COLLISION_TTC_S))
eval_v8["pred_approaching"] = eval_v8["pred_vz_sg_mps"] < 0
eval_v8["pred_ttc_eval"] = np.where(eval_v8["pred_approaching"],
                                    eval_v8["pred_z_pinhole"] / np.maximum(-eval_v8["pred_vz_sg_mps"], 1e-6), np.inf)

f1_v8, far_v8 = f1_far_for_threshold(eval_v8, BEST_TH, TOTAL_RECORDING_HOURS)
print(f"YOLOv8x + ByteTrack + Savitzky-Golay: F1={f1_v8:.3f}, FAR/gio={far_v8:.2f} (nguong {BEST_TH}s)")

ablation_df.loc[ablation_df["Config"].str.contains("YOLOv8x"), ["F1", "FAR_per_hour"]] = [round(f1_v8, 3), round(far_v8, 2)]
ablation_df.to_csv(TABLE_DIR / "table4_ablation_FIXED.csv", index=False)
display(ablation_df)

## Phan 5 - Ap dung chien luoc tu ban phan bien Q1 toan dien

Tai lieu phan bien co ~15 de xuat. 4 viec duoi day duoc chon vi: (a) Impact "Cao"
theo bang Muc 10 cua tai lieu, (b) kha thi NGAY voi du lieu/hardware hien co,
khong can them dataset moi (BDD100K/nuScenes/DAD/DoTA can Lan add vao Kaggle
truoc khi lam tiep, chua lam trong ban nay).

In [ ]:
# ==============================================================================
# 25. ABLATION BoT-SORT (THEM MOI)
# ==============================================================================
# So sanh ByteTrack vs BoT-SORT

# Luu config cu
_TRACKER_CFG_BACKUP = TRACKER_CFG
TRACKER_CFG = "botsort.yaml"

print("Chuyen sang BoT-SORT...")

# Chay lai voi BoT-SORT
pred_botsort_dfs = []
for seq_id in SELECTED_SEQUENCES:
    pred_seq_bs, _rt = run_yolo_bytetrack_on_sequence(seq_id)
    if len(pred_seq_bs):
        pred_botsort_dfs.append(pred_seq_bs)

# Khoi phuc config cu
TRACKER_CFG = _TRACKER_CFG_BACKUP

pred_botsort_all = pd.concat(pred_botsort_dfs, ignore_index=True)
print(f"BoT-SORT pred rows: {len(pred_botsort_all)} (ByteTrack: {len(pred_all)})")

# Danh gia
distance_dfs_bs = []
for seq_id in SELECTED_SEQUENCES:
    ddf = match_pred_to_gt_for_distance(seq_id, pred_botsort_all, iou_thr=EVAL_IOU_THRESHOLD)
    if len(ddf):
        distance_dfs_bs.append(ddf)

distance_pairs_bs = pd.concat(distance_dfs_bs, ignore_index=True)
velocity_pairs_bs = compute_velocity_pairs(distance_pairs_bs, fps=KITTI_FPS)

eval_bs = distance_pairs_bs.merge(
    velocity_pairs_bs[["sequence_id", "frame", "pred_track_id", "gt_vz_mps", "pred_vz_sg_mps"]],
    on=["sequence_id", "frame", "pred_track_id"], how="inner",
)

eval_bs["gt_approaching"] = eval_bs["gt_vz_mps"] < 0
eval_bs["gt_ttc"] = np.where(eval_bs["gt_approaching"],
                             eval_bs["gt_z"] / np.maximum(-eval_bs["gt_vz_mps"], 1e-6), np.inf)
eval_bs["gt_is_risk"] = ((eval_bs["gt_z"] < GT_COLLISION_DIST_M)
                         & eval_bs["gt_approaching"]
                         & (eval_bs["gt_ttc"] < GT_COLLISION_TTC_S))

eval_bs["pred_approaching"] = eval_bs["pred_vz_sg_mps"] < 0
eval_bs["pred_ttc_eval"] = np.where(eval_bs["pred_approaching"],
                                    eval_bs["pred_z_pinhole"] / np.maximum(-eval_bs["pred_vz_sg_mps"], 1e-6), np.inf)

# Tinh F1
TP_bs, FP_bs, FN_bs, TN_bs, _ = windowed_confusion_matrix(eval_bs, BEST_TH)
f1_bs = 2 * (TP_bs / (TP_bs + FP_bs)) * (TP_bs / (TP_bs + FN_bs)) / ((TP_bs / (TP_bs + FP_bs)) + (TP_bs / (TP_bs + FN_bs))) if (TP_bs + FP_bs) > 0 and (TP_bs + FN_bs) > 0 else 0
far_bs = FP_bs / TOTAL_RECORDING_HOURS

print(f"\n[KET QUA] YOLOv11x + BoT-SORT: F1={f1_bs:.3f}, FAR/gio={far_bs:.2f}")

# So sanh
f1_byte = 0.379  # tu Cell 22
if f1_bs > f1_byte:
    print("-> BoT-SORT cai thien F1 so voi ByteTrack.")
else:
    print("-> BoT-SORT KHONG cai thien.")

In [ ]:
# ==============================================================================
# 26. MULTI-SSM FUSION: TTC + DRAC (THEM MOI)
# ==============================================================================
from sklearn.linear_model import LogisticRegression

# Tinh DRAC
eval_fusion = eval_base.copy()
closing_speed = np.where(eval_fusion["pred_approaching"], -eval_fusion["pred_vz_sg_mps"], 0.0)
eval_fusion["drac"] = (closing_speed ** 2) / (2.0 * np.maximum(eval_fusion["pred_z_pinhole"], 1e-3))
eval_fusion["inv_ttc"] = 1.0 / np.maximum(eval_fusion["pred_ttc_eval"].replace(np.inf, 1e6), 1e-3)

# Train/Test split theo sequence
seq_list = sorted(eval_fusion["sequence_id"].unique())
n_train = int(len(seq_list) * 0.7)
train_seqs = set(seq_list[:n_train])
test_seqs = set(seq_list[n_train:])

print(f"Train sequences: {sorted(train_seqs)}")
print(f"Test sequences: {sorted(test_seqs)}")

# Chuan bi data
train_mask = eval_fusion["sequence_id"].isin(train_seqs)
test_mask = eval_fusion["sequence_id"].isin(test_seqs)

X_train = eval_fusion.loc[train_mask, ["inv_ttc", "drac"]].values
y_train = eval_fusion.loc[train_mask, "gt_is_risk"].values.astype(int)
X_test = eval_fusion.loc[test_mask, ["inv_ttc", "drac"]].values
y_test = eval_fusion.loc[test_mask, "gt_is_risk"].values.astype(int)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

# Train model
fusion_model = LogisticRegression(class_weight="balanced", max_iter=1000)
fusion_model.fit(X_train, y_train)

print(f"Weights: inv_ttc={fusion_model.coef_[0][0]:.4f}, drac={fusion_model.coef_[0][1]:.4f}")

# Danh gia tren test set
fusion_score = fusion_model.predict_proba(X_test)[:, 1]

# Tinh F1 tai nguong 0.5
fusion_pred = (fusion_score >= 0.5).astype(int)
TP = int(((fusion_pred == 1) & (y_test == 1)).sum())
FP = int(((fusion_pred == 1) & (y_test == 0)).sum())
FN = int(((fusion_pred == 0) & (y_test == 1)).sum())

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_fusion = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nF1 fusion (test set): {f1_fusion:.3f}")
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}")

In [ ]:
# ==============================================================================
# McNemar test — FIX: grouped_no / grouped_heur chưa được định nghĩa ở trên
# Hàm windowed_confusion_matrix_heur cũ chỉ trả về (TP,FP,FN,TN),
# không trả về DataFrame. Cell này tự build lại grouped trước khi test.
# ==============================================================================

from scipy.stats import binomtest   # import lại an toàn nếu chưa có

def build_grouped_windows(eval_df, th,
                          window_frames=EVENT_WINDOW_FRAMES,
                          apply_heur=False):
    """
    Tạo DataFrame per-window (sequence_id, pred_track_id, window_id,
    gt_is_risk, pred_warn) dùng cho McNemar test.

    Khác với windowed_confusion_matrix_heur ở chỗ trả về DataFrame thô
    thay vì chỉ trả về (TP, FP, FN, TN).
    """
    df = eval_df.copy()
    df["window_id"] = (df["frame"] // window_frames).astype(int)

    pred_warn = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
    if apply_heur:
        if "heuristics_passed" not in df.columns:
            raise KeyError("Cần chạy cell 4b trước để có cột heuristics_passed")
        pred_warn = pred_warn & df["heuristics_passed"]

    df["pred_warn"] = pred_warn.astype(bool)

    grouped = (
        df.groupby(["sequence_id", "pred_track_id", "window_id"])
        .agg(gt_is_risk=("gt_is_risk", "any"),
             pred_warn=("pred_warn", "any"))
        .reset_index()
    )
    return grouped


# Kiểm tra BEST_TH tồn tại; nếu không thì dùng 1.0 (hardcode fallback)
try:
    _th = float(BEST_TH)
except NameError:
    _th = 1.0
    print(f"[WARNING] BEST_TH chưa được định nghĩa, dùng fallback TH={_th}s")

print(f"Dùng TH = {_th}s  |  EVENT_WINDOW_FRAMES = {EVENT_WINDOW_FRAMES}")

# Build grouped DataFrames
grouped_no   = build_grouped_windows(eval_base, _th, apply_heur=False)
grouped_heur = build_grouped_windows(eval_heur, _th, apply_heur=True)

print(f"grouped_no  : {len(grouped_no):,} windows  "
      f"| pos={int(grouped_no['gt_is_risk'].sum())}  "
      f"| predicted={int(grouped_no['pred_warn'].sum())}")
print(f"grouped_heur: {len(grouped_heur):,} windows  "
      f"| pos={int(grouped_heur['gt_is_risk'].sum())}  "
      f"| predicted={int(grouped_heur['pred_warn'].sum())}")

# ── McNemar test ──────────────────────────────────────────────
merged = grouped_no.merge(
    grouped_heur,
    on=["sequence_id", "pred_track_id", "window_id"],
    suffixes=("_no", "_heur"),
)

# gt_is_risk_no == gt_is_risk_heur (cùng GT); dùng _no làm chuẩn
gt_col = "gt_is_risk_no"

b = int(((merged["pred_warn_no"] == merged[gt_col]) &
         (merged["pred_warn_heur"] != merged[gt_col])).sum())
c = int(((merged["pred_warn_no"] != merged[gt_col]) &
         (merged["pred_warn_heur"] == merged[gt_col])).sum())

print(f"\nMcNemar contingency: b (no đúng, heur sai) = {b}")
print(f"                     c (no sai, heur đúng) = {c}")

n_discordant = b + c
if n_discordant > 0:
    mcnemar_result = binomtest(min(b, c), n_discordant, p=0.5,
                               alternative="two-sided")
    print(f"McNemar p-value = {mcnemar_result.pvalue:.4f}")
    if mcnemar_result.pvalue < 0.05:
        winner = "Heuristics" if c > b else "No-heuristics"
        print(f"-> Khác biệt CÓ Ý NGHĨA THỐNG KÊ (p < 0.05)  —  {winner} tốt hơn")
    else:
        print("-> Khác biệt CHƯA đạt ý nghĩa thống kê (p ≥ 0.05)")
else:
    print("\nKhông có cặp discordant — hai cấu hình cho cùng kết quả trên mọi window")
    print("(Điều này có thể xảy ra khi dữ liệu quá ít hoặc heuristics không thay đổi quyết định)")


In [ ]:
# ==============================================================================
# 28. ROBUSTNESS CORRUPTION TEST (THEM MOI)
# ==============================================================================
import shutil

ROBUSTNESS_SEQUENCES = SELECTED_SEQUENCES[:2]
ROBUSTNESS_MAX_FRAMES = 60
ROBUSTNESS_WORK_DIR = Path("/kaggle/working/robustness_corrupted")

def corrupt_gaussian_noise(img, sigma):
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def corrupt_gaussian_blur(img, ksize):
    k = ksize if ksize % 2 == 1 else ksize + 1
    return cv2.GaussianBlur(img, (k, k), 0)

def corrupt_low_light(img, factor):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

def corrupt_fog(img, intensity):
    fog_layer = np.full_like(img, 200, dtype=np.uint8)
    return cv2.addWeighted(img, 1 - intensity, fog_layer, intensity, 0)

CORRUPTION_CONFIGS = [
    {"name": "clean_baseline", "fn": None, "param": None},
    {"name": "gaussian_noise_low", "fn": corrupt_gaussian_noise, "param": 15},
    {"name": "gaussian_noise_high", "fn": corrupt_gaussian_noise, "param": 30},
    {"name": "blur_low", "fn": corrupt_gaussian_blur, "param": 5},
    {"name": "blur_high", "fn": corrupt_gaussian_blur, "param": 11},
    {"name": "low_light", "fn": corrupt_low_light, "param": 0.5},
    {"name": "fog", "fn": corrupt_fog, "param": 0.3},
]

def build_corrupted_sequence_dir(seq_id, corruption_fn, param, max_frames):
    seq_id = str(seq_id).zfill(4)
    src_files = get_image_files(seq_id, max_frames=max_frames)
    dst_dir = ROBUSTNESS_WORK_DIR / seq_id
    
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    for fp in src_files:
        img = cv2.imread(str(fp))
        if img is None:
            continue
        out_img = corruption_fn(img, param) if corruption_fn is not None else img
        cv2.imwrite(str(dst_dir / fp.name), out_img)
    
    return dst_dir

_IMAGE_DIR_BACKUP = IMAGE_DIR
robustness_rows = []

for cfg in CORRUPTION_CONFIGS:
    print(f"\n=== Corruption: {cfg['name']} ===")
    
    pred_dfs_corrupt = []
    for seq_id in ROBUSTNESS_SEQUENCES:
        IMAGE_DIR = _IMAGE_DIR_BACKUP
        corrupted_dir = build_corrupted_sequence_dir(seq_id, cfg["fn"], cfg["param"], ROBUSTNESS_MAX_FRAMES)
        IMAGE_DIR = ROBUSTNESS_WORK_DIR
        
        pred_seq, _rt = run_yolo_bytetrack_on_sequence(seq_id)
        if len(pred_seq):
            pred_dfs_corrupt.append(pred_seq)
    
    IMAGE_DIR = _IMAGE_DIR_BACKUP
    pred_corrupt_all = pd.concat(pred_dfs_corrupt, ignore_index=True)
    
    # Danh gia
    distance_dfs_c = []
    for seq_id in ROBUSTNESS_SEQUENCES:
        ddf = match_pred_to_gt_for_distance(seq_id, pred_corrupt_all, iou_thr=EVAL_IOU_THRESHOLD)
        if len(ddf):
            distance_dfs_c.append(ddf)
    
    distance_pairs_c = pd.concat(distance_dfs_c, ignore_index=True)
    
    if len(distance_pairs_c) > 0:
        mae_c = float((distance_pairs_c["pred_distance_pinhole"] - distance_pairs_c["gt_distance"]).abs().mean())
        print(f"  Distance MAE: {mae_c:.2f}m")
        robustness_rows.append({"corruption": cfg["name"], "distance_mae_m": mae_c})
    else:
        print("  Khong co detection match")
        robustness_rows.append({"corruption": cfg["name"], "distance_mae_m": np.nan})

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(TABLE_DIR / "table_robustness_corruption.csv", index=False)
display(robustness_df)

In [ ]:
# ==============================================================================
# 5a. ABLATION TRACKER: ByteTrack -> BoT-SORT (Mục 2.1 tài liệu phản biện)
# ==============================================================================
# BoT-SORT có sẵn trong ultralytics (botsort.yaml), không cần cài thêm gì.
# Đây là "quick win" tài liệu đề xuất: không cần train, chỉ đổi config tracker.

# ------------------------------------------------------------------------------
# SỬA LỖI DÂY CHUYỀN: Ép lại weights về YOLOv11x để sửa lỗi kẹt 'yolo8x.pt' từ cell trước
# ------------------------------------------------------------------------------
YOLO_WEIGHTS = "/kaggle/working/yolo11x.pt"  # Đảm bảo đúng tên file chuẩn của YOLOv11x
# Nếu bạn chạy offline (Internet Off), hãy uncomment dòng dưới và điền đúng đường dẫn:
# YOLO_WEIGHTS = "/kaggle/input/yolov11-weights/yolov11x.pt" 

# Lưu cấu hình tracker cũ và chuyển sang BoT-SORT
_TRACKER_CFG_BACKUP = TRACKER_CFG
TRACKER_CFG = "botsort.yaml"
print(f"Đã ép lại YOLO_WEIGHTS -> {YOLO_WEIGHTS}")
print(f"Chuyển tạm thời TRACKER_CFG -> {TRACKER_CFG} để chạy ablation...")

# Chạy inference và tracking trên toàn bộ các chuỗi dữ liệu được chọn
pred_botsort_dfs = []
for seq_id in SELECTED_SEQUENCES:
    # Hàm này sẽ gọi model.track() bên trong. Hãy đảm bảo logic của hàm 
    # sử dụng biến toàn cục TRACKER_CFG thay vì hardcode "bytetrack.yaml".
    pred_seq_bs, _rt = run_yolo_bytetrack_on_sequence(seq_id)
    if len(pred_seq_bs):
        pred_botsort_dfs.append(pred_seq_bs)

# Khôi phục lại cấu hình tracker gốc sau khi chạy xong vòng lặp
TRACKER_CFG = _TRACKER_CFG_BACKUP
print(f"Đã khôi phục TRACKER_CFG -> {TRACKER_CFG}")

# Gộp kết quả dự đoán của BoT-SORT
pred_botsort_all = pd.concat(pred_botsort_dfs, ignore_index=True) if pred_botsort_dfs else pd.DataFrame()
print(f"BoT-SORT pred rows: {len(pred_botsort_all)} (so sánh ByteTrack: {len(pred_all)})")

# ------------------------------------------------------------------------------
# ĐÁNH GIÁ (EVALUATION) KẾT QUẢ ĐO KHOẢNG CÁCH & VẬN TỐC
# ------------------------------------------------------------------------------
distance_dfs_bs = []
for seq_id in SELECTED_SEQUENCES:
    ddf = match_pred_to_gt_for_distance(seq_id, pred_botsort_all, iou_thr=EVAL_IOU_THRESHOLD)
    if len(ddf):
        distance_dfs_bs.append(ddf)
distance_pairs_bs = pd.concat(distance_dfs_bs, ignore_index=True) if distance_dfs_bs else pd.DataFrame()

# Tính toán vận tốc từ khoảng cách
velocity_pairs_bs = compute_velocity_pairs(distance_pairs_bs, fps=KITTI_FPS)

# Merge dữ liệu ground truth và prediction phục vụ tính toán TTC
eval_bs = distance_pairs_bs.merge(
    velocity_pairs_bs[["sequence_id", "frame", "pred_track_id", "gt_vz_mps", "pred_vz_sg_mps"]],
    on=["sequence_id", "frame", "pred_track_id"], how="inner",
)

# Tính toán các chỉ số nguy cơ va chạm (Risk/TTC) cho Ground Truth và BoT-SORT
eval_bs["gt_approaching"] = eval_bs["gt_vz_mps"] < 0
eval_bs["gt_ttc"] = np.where(eval_bs["gt_approaching"],
                             eval_bs["gt_z"] / np.maximum(-eval_bs["gt_vz_mps"], 1e-6), np.inf)
eval_bs["gt_is_risk"] = ((eval_bs["gt_z"] < GT_COLLISION_DIST_M)
                         & eval_bs["gt_approaching"]
                         & (eval_bs["gt_ttc"] < GT_COLLISION_TTC_S))

eval_bs["pred_approaching"] = eval_bs["pred_vz_sg_mps"] < 0
eval_bs["pred_ttc_eval"] = np.where(eval_bs["pred_approaching"],
                                    eval_bs["pred_z_pinhole"] / np.maximum(-eval_bs["pred_vz_sg_mps"], 1e-6), np.inf)

# Tính F1-score và False Alarm Rate (FAR) trên giờ
f1_bs, far_bs = f1_far_for_threshold(eval_bs, BEST_TH, TOTAL_RECORDING_HOURS)
print(f"\n[KẾT QUẢ] YOLOv11x + BoT-SORT + Savitzky-Golay: F1={f1_bs:.3f}, FAR/giờ={far_bs:.2f} (ngưỡng {BEST_TH}s)")

# ------------------------------------------------------------------------------
# SO SÁNH TRỰC QUAN & CẬP NHẬT BẢNG ABLATION REPORT
# ------------------------------------------------------------------------------
bytetrack_row = ablation_df[ablation_df["Config"].str.contains("Savitzky-Golay") &
                             ~ablation_df["Config"].str.contains("Heuristics")]
if len(bytetrack_row):
    f1_byte = bytetrack_row.iloc[0]["F1"]
    print(f"(So sánh chéo) YOLOv11x + ByteTrack + Savitzky-Golay: F1={f1_byte}")
    if f1_bs > f1_byte:
        print("-> ĐÁNH GIÁ: BoT-SORT cải thiện F1 thực sự so với ByteTrack. Nên đổi tracker trong paper chính.")
    else:
        print("-> ĐÁNH GIÁ: BoT-SORT KHÔNG cải thiện ở dữ liệu này. Có thể giữ ByteTrack, báo cáo trung thực cả 2 vào paper.")

# Thêm dòng kết quả mới vào dataframe ablation_df
ablation_df = pd.concat([
    ablation_df,
    pd.DataFrame([{
        "Config": "YOLOv11x + BoT-SORT + Savitzky-Golay",
        "Detector": "YOLOv11x", 
        "Filter": "BoT-SORT + Savitzky-Golay",
        "F1": round(float(f1_bs), 3), 
        "FAR_per_hour": round(float(far_bs), 2),
    }])
], ignore_index=True)

# Lưu lại file kết quả ablation cuối cùng
ablation_df.to_csv(TABLE_DIR / "table4_ablation_FIXED.csv", index=False)
display(ablation_df)

In [ ]:
# ==============================================================================
# 5b. MULTI-SSM FUSION THAT: TTC + DRAC, HOC TRONG SO (Muc 2.3/8.2 tai lieu)
# ==============================================================================
# DRAC (Deceleration Rate to Avoid Crash) = v_rel^2 / (2 * distance) - chuan SSM
# trong literature an toan giao thong (khac TTC, nhay voi gia toc can de tranh va
# cham, khong chi khoang cach/van toc).
# PET (Post-Encroachment Time) KHONG implement duoc o day vi can 2 doi tuong cat
# nhau tai 1 diem xung dot chung - kien truc hien tai la ego-centric (xem ghi chu
# heuristics o tren) - gioi han nay ghi ro trong paper, khong gia mao.
#
# QUAN TRONG - tranh chinh loi leakage da sua truoc do: hoc trong so fusion tren
# TAP TRAIN (mot phan sequence), danh gia tren TAP TEST (sequence con lai) -
# KHONG hoc va danh gia tren cung du lieu (se lai la circular evaluation).

from sklearn.linear_model import LogisticRegression

DRAC_TRAIN_FRACTION = 0.7
seq_list_for_split = sorted(eval_base["sequence_id"].unique())
n_train_seq = max(1, int(len(seq_list_for_split) * DRAC_TRAIN_FRACTION))
train_sequences = set(seq_list_for_split[:n_train_seq])
test_sequences = set(seq_list_for_split[n_train_seq:])
print(f"Train sequences ({len(train_sequences)}): {sorted(train_sequences)}")
print(f"Test sequences  ({len(test_sequences)}): {sorted(test_sequences)}")

eval_fusion = eval_base.copy()
closing_speed = np.where(eval_fusion["pred_approaching"], -eval_fusion["pred_vz_sg_mps"], 0.0)
eval_fusion["drac"] = (closing_speed ** 2) / (2.0 * np.maximum(eval_fusion["pred_z_pinhole"], 1e-3))
eval_fusion["inv_ttc"] = 1.0 / np.maximum(eval_fusion["pred_ttc_eval"].replace(np.inf, 1e6), 1e-3)

train_mask = eval_fusion["sequence_id"].isin(train_sequences)
test_mask = eval_fusion["sequence_id"].isin(test_sequences)

X_train = eval_fusion.loc[train_mask, ["inv_ttc", "drac"]].values
y_train = eval_fusion.loc[train_mask, "gt_is_risk"].values.astype(int)
X_test = eval_fusion.loc[test_mask, ["inv_ttc", "drac"]].values
y_test = eval_fusion.loc[test_mask, "gt_is_risk"].values.astype(int)

print(f"Train samples: {len(X_train)} (positive: {int(y_train.sum())}) | "
      f"Test samples: {len(X_test)} (positive: {int(y_test.sum())})")

if y_train.sum() == 0 or len(test_sequences) == 0:
    print("CANH BAO: khong du positive de hoc fusion, hoac khong co sequence test rieng.")
    print("Can them sequence (Cell 1, Tuan 1) truoc khi chay cell nay.")
else:
    fusion_model = LogisticRegression(class_weight="balanced", max_iter=1000)
    fusion_model.fit(X_train, y_train)
    print(f"Trong so hoc duoc: inv_ttc={fusion_model.coef_[0][0]:.4f}, "
          f"drac={fusion_model.coef_[0][1]:.4f}, intercept={fusion_model.intercept_[0]:.4f}")

    fusion_score_test = fusion_model.predict_proba(X_test)[:, 1]
    eval_fusion.loc[test_mask, "fusion_score"] = fusion_score_test

    fusion_rows = []
    for fusion_th in [0.3, 0.4, 0.5, 0.6, 0.7]:
        df_test = eval_fusion.loc[test_mask].copy()
        df_test["window_id"] = (df_test["frame"] // EVENT_WINDOW_FRAMES).astype(int)
        df_test["pred_warn"] = df_test["fusion_score"] >= fusion_th
        grouped = (df_test.groupby(["sequence_id", "pred_track_id", "window_id"])
                   .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
                   .reset_index())
        TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
        FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        TN = int((~grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fusion_rows.append({"fusion_threshold": fusion_th, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                             "Precision": round(precision, 4), "Recall": round(recall, 4), "F1": round(f1, 4)})

    fusion_results_df = pd.DataFrame(fusion_rows)
    fusion_results_df.to_csv(TABLE_DIR / "table_fusion_ttc_drac_FIXED.csv", index=False)
    display(fusion_results_df)

    best_fusion = fusion_results_df.sort_values("F1", ascending=False).iloc[0]
    print(f"\nF1 tot nhat cua TTC+DRAC fusion (tren TEST sequence, khong leakage): {best_fusion['F1']:.3f}")
    print("So sanh voi F1 cua TTC-only/heuristics o cac cell truoc (CHU Y: cac so do la tren TOAN BO")
    print("du lieu, khong tach train/test rieng nhu o day - so sanh chi mang tinh tham khao, khong")
    print("hoan toan cong bang ve protocol. Neu muon so sanh chuan, can chay lai TTC-only/heuristics")
    print("cung tren test_sequences thoi, khong phai toan bo eval_base.")

In [ ]:
# ==============================================================================
# 5c. BOOTSTRAP CI + MCNEMAR TEST (Muc 3.3 tai lieu phan bien)
# ==============================================================================
N_BOOTSTRAP = 1000
rng_boot = np.random.default_rng(123)


def compute_f1_precision(grouped):
    TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
    FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall


def build_grouped_windows(eval_df, th, apply_heur=False):
    df = eval_df.copy()
    df["window_id"] = (df["frame"] // EVENT_WINDOW_FRAMES).astype(int)
    pred_warn = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
    if apply_heur:
        pred_warn = pred_warn & df["heuristics_passed"]
    df["pred_warn"] = pred_warn
    return (df.groupby(["sequence_id", "pred_track_id", "window_id"])
            .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
            .reset_index())


grouped_no_heur = build_grouped_windows(eval_base, BEST_TH, apply_heur=False)
grouped_with_heur = build_grouped_windows(eval_heur, BEST_TH, apply_heur=True)


def bootstrap_f1_ci(grouped, n_boot=N_BOOTSTRAP):
    f1_samples = []
    n = len(grouped)
    for _ in range(n_boot):
        idx = rng_boot.integers(0, n, size=n)
        sample = grouped.iloc[idx]
        f1, _, _ = compute_f1_precision(sample)
        f1_samples.append(f1)
    f1_samples = np.array(f1_samples)
    return float(np.percentile(f1_samples, 2.5)), float(np.percentile(f1_samples, 97.5)), float(np.mean(f1_samples))


ci_lo_no, ci_hi_no, mean_no = bootstrap_f1_ci(grouped_no_heur)
ci_lo_with, ci_hi_with, mean_with = bootstrap_f1_ci(grouped_with_heur)

print(f"F1 khong heuristics: mean={mean_no:.3f}, 95% CI=[{ci_lo_no:.3f}, {ci_hi_no:.3f}] (n_boot={N_BOOTSTRAP})")
print(f"F1 co heuristics   : mean={mean_with:.3f}, 95% CI=[{ci_lo_with:.3f}, {ci_hi_with:.3f}] (n_boot={N_BOOTSTRAP})")
if ci_lo_with > ci_hi_no or ci_lo_no > ci_hi_with:
    print("-> 2 khoang CI khong giao nhau: su khac biet co the xem la dang ke ve thong ke.")
else:
    print("-> 2 khoang CI co giao nhau: chua the khang dinh chac chan heuristics tot hon ve thong ke,")
    print("   du F1 trung binh co cao hon. Can bao cao trung thuc dieu nay trong paper.")

merged_compare = grouped_no_heur.merge(
    grouped_with_heur, on=["sequence_id", "pred_track_id", "window_id"],
    suffixes=("_no_heur", "_with_heur"),
)
merged_compare["gt_is_risk"] = merged_compare["gt_is_risk_no_heur"]
merged_compare["correct_no_heur"] = merged_compare["pred_warn_no_heur"] == merged_compare["gt_is_risk"]
merged_compare["correct_with_heur"] = merged_compare["pred_warn_with_heur"] == merged_compare["gt_is_risk"]

b = int(((merged_compare["correct_no_heur"]) & (~merged_compare["correct_with_heur"])).sum())
c = int(((~merged_compare["correct_no_heur"]) & (merged_compare["correct_with_heur"])).sum())
print(f"\nMcNemar contingency: b (chi no_heur dung)={b}, c (chi with_heur dung)={c}")

from scipy.stats import binomtest
n_discordant = b + c
if n_discordant == 0:
    print("Khong co cap discordant nao - 2 cau hinh cho ket qua giong het nhau tren tat ca sample.")
else:
    mcnemar_result = binomtest(min(b, c), n_discordant, p=0.5, alternative="two-sided")
    print(f"McNemar exact test (qua binomial sign test): p-value = {mcnemar_result.pvalue:.4f}")
    if mcnemar_result.pvalue < 0.05:
        print("-> Khac biet co Y NGHIA THONG KE (p<0.05) giua co/khong heuristics.")
    else:
        print("-> Khac biet CHUA dat y nghia thong ke (p>=0.05) o muc mau hien tai.")
        print("   Co the can them du lieu (them sequence) de tang power thong ke.")

## Phan 6 - Robustness corruption test (Muc 3.4 tai lieu phan bien)

Khong dung dataset moi - tao anh KITTI bi nhieu hoa (Gaussian noise, blur, toi,
suong mu) tu chinh anh da co, chay lai detection+tracking+eval THAT tren ban
nhieu, so sanh voi baseline sach. Gioi han 2 sequence (0000, 0001) x 60 frame
de kiem soat thoi gian chay - co the tang sau khi xac nhan code chay dung.

In [ ]:
# ==============================================================================
# 32. DEPTH ANYTHING V2 METRIC - LOAD MODEL (THEM MOI)
# ==============================================================================
!pip install -q transformers accelerate

import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

DEPTH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
depth_processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf")
depth_model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf").to(DEPTH_DEVICE)
depth_model.eval()

def get_depth_map(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = depth_processor(images=img, return_tensors="pt").to(DEPTH_DEVICE)
    
    with torch.no_grad():
        outputs = depth_model(**inputs)
        depth_map = outputs.predicted_depth.squeeze().float().cpu().numpy()
    
    return cv2.resize(depth_map, (img.width, img.height))

def extract_depth_at_bbox(depth_map, x1, y1, x2, y2):
    cx1, cy1 = int(max(x1, 0)), int(max(y1, 0))
    cx2 = int(min(x2, depth_map.shape[1] - 1))
    cy2 = int(min(y2, depth_map.shape[0] - 1))
    
    if cx2 <= cx1 or cy2 <= cy1:
        return float("nan")
    
    patch = depth_map[cy1:cy2, cx1:cx2]
    return float(np.median(patch))

print("Da load Depth Anything V2 Metric model")

In [ ]:
# ==============================================================================
# 33. DEPTH ANYTHING V2 - SO SANH VOI PINHOLE (THEM MOI)
# ==============================================================================
DEPTH_EVAL_SEQUENCES = SELECTED_SEQUENCES[:3]

# Lay bbox tu distance_pairs
if all(c in distance_pairs.columns for c in ["x1", "y1", "x2", "y2"]):
    bbox_source = distance_pairs
else:
    bbox_source = distance_pairs.merge(
        pred_all[["sequence_id", "frame", "track_id", "x1", "y1", "x2", "y2"]]
        .rename(columns={"track_id": "pred_track_id"}),
        on=["sequence_id", "frame", "pred_track_id"], how="left",
    )

depth_eval_rows = []
sub_df = bbox_source[bbox_source["sequence_id"].isin(DEPTH_EVAL_SEQUENCES)]
frames_to_process = sub_df[["sequence_id", "frame"]].drop_duplicates()

print(f"So frame can chay: {len(frames_to_process)}")

for _, row in frames_to_process.iterrows():
    seq_id, frame_num = row["sequence_id"], row["frame"]
    img_files = get_image_files(seq_id, max_frames=None)
    img_path = next((f for f in img_files if int(f.stem) == int(frame_num)), None)
    
    if img_path is None:
        continue
    
    depth_map = get_depth_map(img_path)
    frame_rows = sub_df[(sub_df["sequence_id"] == seq_id) & (sub_df["frame"] == frame_num)]
    
    for _, det in frame_rows.iterrows():
        if pd.isna(det.get("x1")):
            continue
        
        z_da = extract_depth_at_bbox(depth_map, det["x1"], det["y1"], det["x2"], det["y2"])
        depth_eval_rows.append({
            "sequence_id": seq_id, "frame": frame_num, "pred_track_id": det["pred_track_id"],
            "gt_z": det["gt_z"], "pred_z_pinhole": det["pred_z_pinhole"], "pred_z_depthanything": z_da,
        })

depth_compare_df = pd.DataFrame(depth_eval_rows).dropna(subset=["pred_z_depthanything"])
print(f"So sample: {len(depth_compare_df)}")

# So sanh
mae_pinhole = float((depth_compare_df["pred_z_pinhole"] - depth_compare_df["gt_z"]).abs().mean())
rmse_pinhole = float(np.sqrt(((depth_compare_df["pred_z_pinhole"] - depth_compare_df["gt_z"]) ** 2).mean()))
mae_da = float((depth_compare_df["pred_z_depthanything"] - depth_compare_df["gt_z"]).abs().mean())
rmse_da = float(np.sqrt(((depth_compare_df["pred_z_depthanything"] - depth_compare_df["gt_z"]) ** 2).mean()))

print(f"\nPinhole: MAE={mae_pinhole:.3f}m, RMSE={rmse_pinhole:.3f}m")
print(f"Depth Anything V2: MAE={mae_da:.3f}m, RMSE={rmse_da:.3f}m")

if mae_da < mae_pinhole:
    print("\n-> Depth Anything V2 TOT HON Pinhole")
else:
    print("\n-> Depth Anything V2 KHONG cai thien")

depth_compare_df.to_csv(TABLE_DIR / "table_depth_comparison.csv", index=False)

In [ ]:
# ==============================================================================
# 34. DEPTH ANYTHING V2 - LAN TRUYEN QUA PIPELINE (THEM MOI)
# ==============================================================================
from scipy.signal import savgol_filter

depth_compare_sorted = depth_compare_df.sort_values(["sequence_id", "pred_track_id", "frame"]).copy()

def smooth_z_savgol(z_values, window=7, poly=2):
    if len(z_values) < window:
        return pd.Series(z_values).rolling(min(3, len(z_values)), min_periods=1, center=True).mean().values
    return savgol_filter(z_values, window, poly)

def build_eval_for_depth_source(df_sorted, z_col):
    rows = []
    
    for (seq, tid), g in df_sorted.groupby(["sequence_id", "pred_track_id"]):
        g = g.sort_values("frame").reset_index(drop=True)
        if len(g) < 3:
            continue
        
        z_smooth = smooth_z_savgol(g[z_col].values)
        vz_smooth = np.gradient(z_smooth) * KITTI_FPS
        
        gt_z_arr = g["gt_z"].values
        gt_vz_arr = np.gradient(gt_z_arr) * KITTI_FPS
        
        for i in range(len(g)):
            gt_approaching = gt_vz_arr[i] < 0
            gt_ttc = gt_z_arr[i] / max(-gt_vz_arr[i], 1e-6) if gt_approaching else np.inf
            gt_is_risk = (gt_z_arr[i] < GT_COLLISION_DIST_M) and gt_approaching and (gt_ttc < GT_COLLISION_TTC_S)
            
            pred_approaching = vz_smooth[i] < 0
            pred_ttc_eval = z_smooth[i] / max(-vz_smooth[i], 1e-6) if pred_approaching else np.inf
            
            rows.append({
                "sequence_id": seq, "pred_track_id": tid, "frame": g.loc[i, "frame"],
                "gt_is_risk": gt_is_risk, "pred_approaching": pred_approaching,
                "pred_ttc_eval": pred_ttc_eval
            })
    
    return pd.DataFrame(rows)

# Danh gia ca 2 phuong an
eval_pinhole = build_eval_for_depth_source(depth_compare_sorted, "pred_z_pinhole")
eval_da = build_eval_for_depth_source(depth_compare_sorted, "pred_z_depthanything")

print(f"Eval Pinhole: {len(eval_pinhole)} samples")
print(f"Eval Depth-Anything: {len(eval_da)} samples")

# Tinh F1 cho ca 2
TOTAL_HOURS = sum(
    (g["frame"].max() - g["frame"].min() + 1) / KITTI_FPS
    for _, g in depth_compare_sorted.groupby("sequence_id")
) / 3600.0

rows_compare = []
for label, eval_df in [("Pinhole", eval_pinhole), ("Depth Anything V2", eval_da)]:
    for th in [1.0, 1.5, 2.0]:
        df = eval_df.copy()
        df["window_id"] = (df["frame"] // 10).astype(int)
        df["pred_warn"] = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
        
        grouped = (df.groupby(["sequence_id", "pred_track_id", "window_id"])
                   .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any"))
                   .reset_index())
        
        TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
        FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        far = FP / TOTAL_HOURS if TOTAL_HOURS > 0 else float("nan")
        
        rows_compare.append({
            "Depth_source": label, "TTC_Threshold": th,
            "Precision": round(precision, 4), "Recall": round(recall, 4),
            "F1": round(f1, 4), "FAR": round(far, 2)
        })

depth_propagation_df = pd.DataFrame(rows_compare)
depth_propagation_df.to_csv(TABLE_DIR / "table_depth_propagation.csv", index=False)
display(depth_propagation_df)

# Tim F1 tot nhat
best_pinhole = depth_propagation_df[depth_propagation_df["Depth_source"] == "Pinhole"].sort_values("F1", ascending=False).iloc[0]
best_da = depth_propagation_df[depth_propagation_df["Depth_source"] == "Depth Anything V2"].sort_values("F1", ascending=False).iloc[0]

print(f"\nF1 tot nhat - Pinhole: {best_pinhole['F1']:.3f}")
print(f"F1 tot nhat - Depth Anything V2: {best_da['F1']:.3f}")

if best_da["F1"] > best_pinhole["F1"]:
    print("-> Depth Anything V2 cai thien F1")
else:
    print("-> Depth Anything V2 KHONG cai thien F1")

In [ ]:
# ==============================================================================
# 5. DO RUNTIME THAT - FULL PIPELINE (detection + ByteTrack + IPM + TTC)
# ==============================================================================
# DA SUA: ban truoc chi do detection+IPM. Ban nay them (b) detection+ByteTrack
# (model.track) de tach rieng overhead cua ByteTrack, va (d) thoi gian tinh TTC
# tren toan bo trajectory cua 1 sequence mau.

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    gpu_name = "unknown"

print("Hardware dang do runtime tren:", gpu_name, "|", platform.platform())
print("CANH BAO: he thong nay khac voi hardware paper dang khai bao o Section IV.A")
print("(Intel i5 + RTX 3050 4GB). Notebook nay thuc te chay tren Kaggle.")
print("-> Can sua lai Section IV.A cho khop hardware thuc, hoac do lai dung tren may RTX 3050.")

seq_for_timing = SELECTED_SEQUENCES[0]
files_for_timing = get_image_files(seq_for_timing, max_frames=30)
calib_for_timing = read_kitti_calib(CALIB_DIR / f"{seq_for_timing}.txt")

# --- (a) Detection only ---
t_det_ms = []
model_det = YOLO(YOLO_WEIGHTS)
for fp in files_for_timing:
    t0 = time.perf_counter()
    res = model_det.predict(str(fp), imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)
    t1 = time.perf_counter()
    t_det_ms.append((t1 - t0) * 1000)

# --- (b) Detection + ByteTrack (model.track) - tach overhead ByteTrack ---
t_track_ms = []
model_track = YOLO(YOLO_WEIGHTS)
for fp in files_for_timing:
    t0 = time.perf_counter()
    res = model_track.track(source=str(fp), persist=True, tracker=TRACKER_CFG,
                             conf=CONF_THRES, iou=IOU_THRES, imgsz=IMG_SIZE, verbose=False)
    t1 = time.perf_counter()
    t_track_ms.append((t1 - t0) * 1000)

# --- (c) IPM depth/position (rieng, da tru phan detection) ---
t_ipm_total_ms = []
for fp in files_for_timing:
    t2 = time.perf_counter()
    res = model_det.predict(str(fp), imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)
    if len(res[0].boxes):
        for b in res[0].boxes.xyxy.cpu().numpy():
            _ = estimate_depth_from_bbox({"y1": float(b[1]), "y2": float(b[3]), "class_name": "car"}, calib_for_timing)
    t3 = time.perf_counter()
    t_ipm_total_ms.append((t3 - t2) * 1000)
ipm_only_ms = max(float(np.mean(t_ipm_total_ms)) - float(np.mean(t_det_ms)), 0.0)

# --- (d) TTC computation tren toan bo 1 sequence mau ---
sample_pred = pred_all[pred_all["sequence_id"].astype(str).str.zfill(4) == seq_for_timing].copy()
t0 = time.perf_counter()
traj_sample = build_pred_trajectory_from_predictions(sample_pred)
traj_sample = add_ttc_to_trajectory(traj_sample, fps=KITTI_FPS)
t1 = time.perf_counter()
n_frames_sample = max(int(sample_pred["frame"].nunique()), 1)
ttc_ms_per_frame = (t1 - t0) * 1000 / n_frames_sample

det_ms = float(np.mean(t_det_ms))
track_ms = float(np.mean(t_track_ms))
bytetrack_overhead_ms = max(track_ms - det_ms, 0.0)

runtime_summary = pd.DataFrame([
    {"Module": "YOLOv11x detection only", "Latency_ms": round(det_ms, 2)},
    {"Module": "Detection + ByteTrack (model.track)", "Latency_ms": round(track_ms, 2)},
    {"Module": "ByteTrack overhead (track - detect)", "Latency_ms": round(bytetrack_overhead_ms, 2)},
    {"Module": "IPM depth/position (per frame)", "Latency_ms": round(ipm_only_ms, 3)},
    {"Module": "TTC computation (per frame, sequence avg)", "Latency_ms": round(ttc_ms_per_frame, 3)},
])
runtime_summary.to_csv(TABLE_DIR / "table5_runtime_FIXED.csv", index=False)
display(runtime_summary)

total_latency_full = track_ms + ipm_only_ms + ttc_ms_per_frame
print(f"Tong latency FULL PIPELINE (track+IPM+TTC): {total_latency_full:.1f} ms -> "
      f"{1000 / total_latency_full:.1f} FPS")
print("Day la so FPS thuc te nen dua vao paper, khong phai so detection-only truoc do.")

In [ ]:
# ==============================================================================
# 6a. ROBUSTNESS CORRUPTION TEST - THAT, KHONG CAN DATASET MOI
# ==============================================================================
# Ap dung 4 loai nhieu pho bien trong robustness benchmark (kieu ImageNet-C/
# RoboDepth): Gaussian noise (sensor), Gaussian blur (mat net/motion blur),
# giam sang (chieu toi/hoang hon), fog (suong mu). Chay lai THAT detection +
# tracking + distance + eval tren anh da nhieu, so voi baseline sach cung
# sequence/frame.
#
# GIOI HAN COMPUTE: chi dung 2 sequence dau (khong phai het 11) va 60 frame/seq
# de kiem soat thoi gian. Tang ROBUSTNESS_SEQUENCES/ROBUSTNESS_MAX_FRAMES neu
# muon ket qua manh hon, nhung se ton GPU-hour tuyen tinh theo so cau hinh.

import shutil
import cv2

ROBUSTNESS_SEQUENCES = SELECTED_SEQUENCES[:2]
ROBUSTNESS_MAX_FRAMES = 60
ROBUSTNESS_WORK_DIR = Path("/kaggle/working/robustness_corrupted")


def corrupt_gaussian_noise(img, sigma):
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def corrupt_gaussian_blur(img, ksize):
    k = ksize if ksize % 2 == 1 else ksize + 1
    return cv2.GaussianBlur(img, (k, k), 0)


def corrupt_low_light(img, factor):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def corrupt_fog(img, intensity):
    fog_layer = np.full_like(img, 200, dtype=np.uint8)
    return cv2.addWeighted(img, 1 - intensity, fog_layer, intensity, 0)


CORRUPTION_CONFIGS = [
    {"name": "clean_baseline", "fn": None, "param": None},
    {"name": "gaussian_noise_low", "fn": corrupt_gaussian_noise, "param": 15},
    {"name": "gaussian_noise_high", "fn": corrupt_gaussian_noise, "param": 30},
    {"name": "blur_low", "fn": corrupt_gaussian_blur, "param": 5},
    {"name": "blur_high", "fn": corrupt_gaussian_blur, "param": 11},
    {"name": "low_light_dusk", "fn": corrupt_low_light, "param": 0.5},
    {"name": "low_light_night", "fn": corrupt_low_light, "param": 0.25},
    {"name": "fog_light", "fn": corrupt_fog, "param": 0.3},
    {"name": "fog_heavy", "fn": corrupt_fog, "param": 0.6},
]


def build_corrupted_sequence_dir(seq_id, corruption_fn, param, max_frames):
    seq_id = str(seq_id).zfill(4)
    src_files = get_image_files(seq_id, max_frames=max_frames)
    dst_dir = ROBUSTNESS_WORK_DIR / seq_id
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for fp in src_files:
        img = cv2.imread(str(fp))
        if img is None:
            continue
        out_img = corruption_fn(img, param) if corruption_fn is not None else img
        cv2.imwrite(str(dst_dir / fp.name), out_img)
    return dst_dir


_IMAGE_DIR_BACKUP = IMAGE_DIR
robustness_rows = []

for cfg in CORRUPTION_CONFIGS:
    print(f"\n=== Corruption: {cfg['name']} ===")
    pred_dfs_corrupt = []
    for seq_id in ROBUSTNESS_SEQUENCES:
        if cfg["fn"] is None:
            IMAGE_DIR = _IMAGE_DIR_BACKUP
        else:
            # DA SUA: dam bao doc anh GOC (khong phai anh corrupt cua seq truoc con sot lai
            # trong IMAGE_DIR) truoc khi build corrupted dir cho seq nay.
            IMAGE_DIR = _IMAGE_DIR_BACKUP
            corrupted_dir = build_corrupted_sequence_dir(seq_id, cfg["fn"], cfg["param"], ROBUSTNESS_MAX_FRAMES)
            IMAGE_DIR = ROBUSTNESS_WORK_DIR
        pred_seq, _rt = run_yolo_bytetrack_on_sequence(seq_id)
        if len(pred_seq):
            pred_dfs_corrupt.append(pred_seq)
    IMAGE_DIR = _IMAGE_DIR_BACKUP

    pred_corrupt_all = pd.concat(pred_dfs_corrupt, ignore_index=True) if pred_dfs_corrupt else pd.DataFrame()

    distance_dfs_c = []
    for seq_id in ROBUSTNESS_SEQUENCES:
        ddf = match_pred_to_gt_for_distance(seq_id, pred_corrupt_all, iou_thr=EVAL_IOU_THRESHOLD)
        if len(ddf):
            distance_dfs_c.append(ddf)
    distance_pairs_c = pd.concat(distance_dfs_c, ignore_index=True) if distance_dfs_c else pd.DataFrame()

    if len(distance_pairs_c) == 0:
        print(f"  Khong co detection match nao voi corruption {cfg['name']} - YOLO co the da mat het object.")
        robustness_rows.append({"corruption": cfg["name"], "n_matched_pairs": 0,
                                 "distance_mae_m": np.nan, "distance_rmse_m": np.nan, "n_detections": len(pred_corrupt_all)})
        continue

    abs_err = (distance_pairs_c["pred_distance_pinhole"] - distance_pairs_c["gt_distance"]).abs()
    sq_err = (distance_pairs_c["pred_distance_pinhole"] - distance_pairs_c["gt_distance"]) ** 2
    mae_c = float(abs_err.mean())
    rmse_c = float(np.sqrt(sq_err.mean()))
    print(f"  n_detections={len(pred_corrupt_all)} n_matched_pairs={len(distance_pairs_c)} "
          f"distance_MAE={mae_c:.2f}m distance_RMSE={rmse_c:.2f}m")
    robustness_rows.append({"corruption": cfg["name"], "n_matched_pairs": len(distance_pairs_c),
                             "n_detections": len(pred_corrupt_all),
                             "distance_mae_m": round(mae_c, 3), "distance_rmse_m": round(rmse_c, 3)})

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(TABLE_DIR / "table_robustness_corruption_FIXED.csv", index=False)
display(robustness_df)

baseline_row = robustness_df[robustness_df["corruption"] == "clean_baseline"]
if len(baseline_row) and not pd.isna(baseline_row.iloc[0]["distance_mae_m"]):
    baseline_mae = baseline_row.iloc[0]["distance_mae_m"]
    baseline_n = baseline_row.iloc[0]["n_matched_pairs"]
    print(f"\nBaseline sach: MAE={baseline_mae}m, n_matched={baseline_n}")
    print("Do suy giam (n_matched_pairs giam = nhieu object bi mat detection, KHONG chi la MAE tang):")
    for _, row in robustness_df.iterrows():
        if row["corruption"] == "clean_baseline":
            continue
        drop_pct = (1 - row["n_matched_pairs"] / baseline_n) * 100 if baseline_n > 0 else float("nan")
        mae_delta = (row["distance_mae_m"] - baseline_mae) if not pd.isna(row["distance_mae_m"]) else float("nan")
        print(f"  {row['corruption']:22s} n_matched giam {drop_pct:5.1f}% | MAE thay doi: {mae_delta:+.2f}m")
print("\nLUU Y: corruption nang co the lam YOLO mat hoan toan object (n_matched_pairs=0),")
print("day la ket qua THAT can bao cao (he thong khong robust duoi dieu kien do), khong phai loi.")

In [ ]:
# ==============================================================================
# 6. TONG HOP SO LIEU DA SUA - DUNG BANG NAY DE VIET LAI PAPER
# ==============================================================================
print("=" * 70)
print("TONG HOP SO LIEU TUAN 2 (DA SUA - khong fabrication, khong leakage)")
print("=" * 70)
print("\n--- Table 3 / ROC sweep (FPR THAT) ---")
display(roc_points_df)
print("\n--- Table 4 / Ablation (chua xong dong YOLOv8x) ---")
display(ablation_df)
print("\n--- Table 5 / Runtime ---")
display(runtime_summary)
print(f"\nVelocity MAE - Savitzky-Golay: {mae_raw_kmh:.2f} km/h")
print(f"Velocity MAE - Monte Carlo (no leakage): {mae_mc_kmh:.2f} km/h")
print(f"\nGT collision definition dang dung: distance < {GT_COLLISION_DIST_M}m, "
      f"TTC < {GT_COLLISION_TTC_S}s (tu LiDAR thuc).")
print("Neu paper muon giu nguyen '1.5m / 3.0s' nhu da viet, doi GT_COLLISION_DIST_M = 1.5")
print("o Cell 1 va chay lai - nhung kiem tra n_risk co > 0 hay khong truoc.")

import zipfile
zip_path = PROJECT_ROOT / "results_FIXED.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [TABLE_DIR, FIGURE_DIR]:
        for p in Path(folder).rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(PROJECT_ROOT))
print("\nDa dong goi:", zip_path)

In [ ]:
# ==============================================================================
# 3b. FIG. 3 (THAY THE) - QUY DAO THAT, TU DONG CHON TRACK, KHONG DUNG np.random
# ==============================================================================
# Ban goc: raw_noisy_trajectory = 25.0 - 0.4*frame + np.random.normal(...) -> BIA HOAN TOAN.
# Ban thay the: tu dong chon 1 track THAT (sequence_id, pred_track_id) co nhieu frame
# lien tuc nhat trong distance_pairs (co ca pred_z_pinhole VA gt_z LiDAR doi chieu),
# roi ve quy dao thuc + uncertainty band tu Monte Carlo nhieu hoa dau vao (khong dung GT).

MIN_TRACK_LEN_FOR_FIG3 = 15

track_lengths = (
    distance_pairs.groupby(["sequence_id", "pred_track_id"])
    .size()
    .reset_index(name="n_frames")
    .sort_values("n_frames", ascending=False)
)
display(track_lengths.head(10))

candidate = track_lengths[track_lengths["n_frames"] >= MIN_TRACK_LEN_FOR_FIG3]
if len(candidate) == 0:
    raise RuntimeError(
        f"Khong co track nao dat toi thieu {MIN_TRACK_LEN_FOR_FIG3} frame trong distance_pairs. "
        "Giam MIN_TRACK_LEN_FOR_FIG3 hoac kiem tra lai distance_pairs."
    )

chosen_seq = candidate.iloc[0]["sequence_id"]
chosen_tid = int(candidate.iloc[0]["pred_track_id"])
chosen_n = int(candidate.iloc[0]["n_frames"])
print(f"Tu dong chon: sequence_id={chosen_seq}, pred_track_id={chosen_tid}, "
      f"so frame={chosen_n} (track dai/on dinh nhat dap ung dieu kien).")

track_df = (
    distance_pairs[
        (distance_pairs["sequence_id"] == chosen_seq)
        & (distance_pairs["pred_track_id"] == chosen_tid)
    ]
    .drop_duplicates(subset=["frame"])
    .sort_values("frame")
    .reset_index(drop=True)
)

z_raw = track_df["pred_z_pinhole"].values
z_gt = track_df["gt_z"].values
frames_idx = track_df["frame"].values

# Monte Carlo nhieu hoa dau vao (pixel jitter + class-height variance), khong dung gt_z.
def monte_carlo_depth_uncertainty(z_array, n_samples=100, pixel_jitter_std=1.0,
                                   relative_height_std=0.06, seed=42):
    rng = np.random.default_rng(seed)
    means, stds = [], []
    for z in z_array:
        noise_scale = abs(z) * relative_height_std + pixel_jitter_std * 0.05
        samples = z + rng.normal(0, max(noise_scale, 1e-3), n_samples)
        means.append(np.mean(samples))
        stds.append(np.std(samples))
    return np.array(means), np.array(stds)

z_mc_mean, z_mc_std = monte_carlo_depth_uncertainty(z_raw)
mae_fig3_kmh_equiv = float(mean_absolute_error(z_gt, z_mc_mean))
print(f"MAE giua quy dao Monte Carlo va LiDAR thuc cho track nay: {mae_fig3_kmh_equiv:.2f} m")

plt.figure(figsize=(8.5, 4.5))
plt.plot(frames_idx, z_raw, linestyle=":", alpha=0.6, color="#d62728",
         label="Raw Pinhole/IPM (camera, chua loc)")
plt.plot(frames_idx, z_mc_mean, linewidth=2.2, color="#1f77b4",
         label="Monte Carlo smoothed (khong dung GT)")
plt.fill_between(frames_idx, z_mc_mean - 1.96 * z_mc_std, z_mc_mean + 1.96 * z_mc_std,
                  color="#1f77b4", alpha=0.15, label="95% CI")
plt.plot(frames_idx, z_gt, linewidth=2, color="#2ca02c", linestyle="--",
         label="Ground Truth (LiDAR thuc)")
plt.xlabel("Frame (Temporal Index)")
plt.ylabel("Estimated Distance Z (m)")
plt.title(f"Spatial Depth Refinement - sequence {chosen_seq}, track {chosen_tid} (du lieu THAT)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig3_path = FIGURE_DIR / "figure3_depth_mc_FIXED.png"
plt.savefig(fig3_path, dpi=300)
plt.show()
print("Da luu:", fig3_path)
print("Neu muon chon track khac thay vi tu dong, sua chon_seq/chosen_tid o tren bang tay roi chay lai cell nay.")

## Phan 7 - Thay the depth estimation bang Depth Anything V2 Metric (VKITTI)

Day la upgrade duoc chon dua tren literature 2024 thuc (NeurIPS 2024), nham
vao diem yeu CO HE THONG xuyen suot moi lan sua truoc (velocity MAE ~17km/h
khong doi, distance RMSE 3.8-4.85m): uoc luong Z hien tai dung gia dinh chieu
cao co dinh theo class (Eq.7-8 trong paper), de nhay loi voi xe khong chuan
(SUV/sedan thap), nguoi lon/tre em.

**Khong xoa pipeline cu** - them depth model nhu MOT MODULE SONG SONG, so sanh
THAT tren cung sequence/frame, cung tieu chi (distance MAE/RMSE, sau do velocity/
F1/FAR qua dung windowed eval da co). Neu Depth Anything V2 thuc su tot hon ->
de xuat thay; neu khong -> bao cao trung thuc ca hai, KHONG ep so lieu.

In [ ]:
# ==============================================================================
# 7a. DEPTH ANYTHING V2 METRIC (VKITTI) - PRETRAINED, KHONG CAN TRAIN LAI
# ==============================================================================
!pip install -q transformers accelerate

import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image
import cv2

DEPTH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_CANDIDATES = [
    "depth-anything/Depth-Anything-V2-Metric-VKITTI-Large-hf",
    "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf",
    "depth-anything/Depth-Anything-V2-Small-hf",  # fallback: RELATIVE depth, kem chinh xac hon
]

depth_model = None
depth_processor = None
DEPTH_MODEL_IS_METRIC = True
for model_id in DEPTH_MODEL_CANDIDATES:
    try:
        depth_processor = AutoImageProcessor.from_pretrained(model_id)
        depth_model = AutoModelForDepthEstimation.from_pretrained(model_id).to(DEPTH_DEVICE)
        depth_model.eval()
        DEPTH_MODEL_ID = model_id
        DEPTH_MODEL_IS_METRIC = "Metric" in model_id
        print(f"Da tai thanh cong: {model_id} (metric={DEPTH_MODEL_IS_METRIC})")
        break
    except Exception as e:
        print(f"Khong tai duoc {model_id}: {e}")

if depth_model is None:
    raise RuntimeError("Khong tai duoc bat ky model Depth Anything V2 nao. Kiem tra ket noi internet "
                        "cua Kaggle notebook (Settings -> Internet -> On) va thu lai.")

if not DEPTH_MODEL_IS_METRIC:
    print("CANH BAO: dang dung ban RELATIVE depth, ket qua se kem chinh xac hon Metric-VKITTI.")
    print("Can them global scale-shift alignment thu cong de co metric scale (chua lam o day).")


def get_depth_map(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = depth_processor(images=img, return_tensors="pt").to(DEPTH_DEVICE)
    with torch.no_grad():
        outputs = depth_model(**inputs)
    depth_map = outputs.predicted_depth.squeeze().float().cpu().numpy()
    depth_map_resized = cv2.resize(depth_map, (img.width, img.height))
    return depth_map_resized


def extract_depth_at_bbox(depth_map, x1, y1, x2, y2):
    cx1, cy1 = int(max(x1, 0)), int(max(y1, 0))
    cx2 = int(min(x2, depth_map.shape[1] - 1))
    cy2 = int(min(y2, depth_map.shape[0] - 1))
    if cx2 <= cx1 or cy2 <= cy1:
        return float("nan")
    patch = depth_map[cy1:cy2, cx1:cx2]
    return float(np.median(patch))

In [ ]:
# ==============================================================================
# 7b. SO SANH THAT: DEPTH ANYTHING V2 vs PINHOLE+CLASS-HEIGHT (CUNG SEQUENCE)
# ==============================================================================
# GIOI HAN COMPUTE: depth model chay tren TOAN BO PIXEL moi frame (nang hon YOLO
# detection-only) -> chi dung DEPTH_EVAL_SEQUENCES (mac dinh 3 sequence dau) de
# kiem soat thoi gian. Tang neu ket qua khả quan va co du GPU-hour.

DEPTH_EVAL_SEQUENCES = SELECTED_SEQUENCES[:3]

if all(c in distance_pairs.columns for c in ["x1", "y1", "x2", "y2"]):
    bbox_source = distance_pairs
else:
    bbox_source = distance_pairs.merge(
        pred_all[["sequence_id", "frame", "track_id", "x1", "y1", "x2", "y2"]]
        .rename(columns={"track_id": "pred_track_id"}),
        on=["sequence_id", "frame", "pred_track_id"], how="left",
    )

depth_eval_rows = []
sub_df = bbox_source[bbox_source["sequence_id"].isin(DEPTH_EVAL_SEQUENCES)]
frames_to_process = sub_df[["sequence_id", "frame"]].drop_duplicates()
print(f"So frame can chay Depth Anything V2: {len(frames_to_process)} "
      f"(tren {len(DEPTH_EVAL_SEQUENCES)} sequence: {DEPTH_EVAL_SEQUENCES})")

for _, row in frames_to_process.iterrows():
    seq_id, frame_num = row["sequence_id"], row["frame"]
    img_files = get_image_files(seq_id, max_frames=None)
    img_path = next((f for f in img_files if int(f.stem) == int(frame_num)), None)
    if img_path is None:
        continue

    depth_map = get_depth_map(img_path)
    frame_rows = sub_df[(sub_df["sequence_id"] == seq_id) & (sub_df["frame"] == frame_num)]
    for _, det in frame_rows.iterrows():
        if pd.isna(det.get("x1")):
            continue
        z_da = extract_depth_at_bbox(depth_map, det["x1"], det["y1"], det["x2"], det["y2"])
        depth_eval_rows.append({
            "sequence_id": seq_id, "frame": frame_num, "pred_track_id": det["pred_track_id"],
            "gt_z": det["gt_z"], "pred_z_pinhole": det["pred_z_pinhole"], "pred_z_depthanything": z_da,
        })

depth_compare_df = pd.DataFrame(depth_eval_rows).dropna(subset=["pred_z_depthanything"])
print(f"So sample so sanh duoc: {len(depth_compare_df)}")

if len(depth_compare_df) == 0:
    print("CANH BAO: khong co sample nao. Kiem tra lai duong dan anh / DEPTH_EVAL_SEQUENCES.")
else:
    mae_pinhole = float((depth_compare_df["pred_z_pinhole"] - depth_compare_df["gt_z"]).abs().mean())
    rmse_pinhole = float(np.sqrt(((depth_compare_df["pred_z_pinhole"] - depth_compare_df["gt_z"]) ** 2).mean()))
    mae_da = float((depth_compare_df["pred_z_depthanything"] - depth_compare_df["gt_z"]).abs().mean())
    rmse_da = float(np.sqrt(((depth_compare_df["pred_z_depthanything"] - depth_compare_df["gt_z"]) ** 2).mean()))

    print(f"\n--- KET QUA THAT (tren {len(DEPTH_EVAL_SEQUENCES)} sequence, {len(depth_compare_df)} sample) ---")
    print(f"Pinhole + class-height (cu)      : MAE={mae_pinhole:.3f}m RMSE={rmse_pinhole:.3f}m")
    print(f"Depth Anything V2 ({'Metric' if DEPTH_MODEL_IS_METRIC else 'Relative, CHUA align scale'})"
          f": MAE={mae_da:.3f}m RMSE={rmse_da:.3f}m")

    depth_compare_df.to_csv(TABLE_DIR / "table_depth_comparison_FIXED.csv", index=False)

    if mae_da < mae_pinhole and DEPTH_MODEL_IS_METRIC:
        print("\n-> Depth Anything V2 Metric THAT SU tot hon. De xuat dung lam pred_z chinh,")
        print("   roi chay lai toan bo velocity/TTC/F1 (Phan 2-6) voi pred_z moi de danh gia day du.")
    elif not DEPTH_MODEL_IS_METRIC:
        print("\n-> Dang dung ban RELATIVE (khong metric) do model Metric-VKITTI tai khong duoc.")
        print("   Ket qua nay CHUA the ket luan - can thu lai voi dung checkpoint metric.")
    else:
        print("\n-> Depth Anything V2 KHONG cai thien so voi pinhole+class-height tren du lieu nay.")
        print("   Day la ket qua THAT - co the do object trong KITTI da kha chuan kich thuoc class,")
        print("   hoac do bbox tu YOLO chua đu chinh xac de cat dung vung object trong depth map.")
        print("   Nen bao cao trung thuc ca hai phuong an trong paper, KHONG chon mot chieu.")

In [ ]:
# ==============================================================================
# 7c. LAN TRUYEN DEPTH ANYTHING V2 QUA TOAN BO PIPELINE (velocity/TTC/F1/FAR)
# ==============================================================================
# Dung pred_z_depthanything da co o Phan 7b, tinh lai velocity (Savitzky-Golay),
# roi tinh F1/Precision/FAR THEO DUNG ham windowed eval da dung cho pipeline cu.
# SO SANH TREN CUNG 3 sequence (0000,0001,0002) cho CA HAI phuong an depth -
# KHONG so voi so 11-sequence cu (pham vi du lieu khac nhau, khong cong bang).

from scipy.signal import savgol_filter

depth_compare_sorted = depth_compare_df.sort_values(["sequence_id", "pred_track_id", "frame"]).copy()

def smooth_z_savgol(z_values, window=7, poly=2):
    if len(z_values) < window:
        return pd.Series(z_values).rolling(min(3, len(z_values)), min_periods=1, center=True).mean().values
    return savgol_filter(z_values, window, poly)

def build_eval_for_depth_source(df_sorted, z_col):
    rows = []
    for (seq, tid), g in df_sorted.groupby(["sequence_id", "pred_track_id"]):
        g = g.sort_values("frame").reset_index(drop=True)
        if len(g) < 3:
            continue
        z_smooth = smooth_z_savgol(g[z_col].values)
        vz_smooth = np.gradient(z_smooth) * KITTI_FPS  # dZ/dt: am = dang tien gan (approaching)
        gt_z_arr = g["gt_z"].values
        gt_vz_arr = np.gradient(gt_z_arr) * KITTI_FPS
        for i in range(len(g)):
            gt_approaching = gt_vz_arr[i] < 0
            gt_ttc = gt_z_arr[i] / max(-gt_vz_arr[i], 1e-6) if gt_approaching else np.inf
            gt_is_risk = (gt_z_arr[i] < GT_COLLISION_DIST_M) and gt_approaching and (gt_ttc < GT_COLLISION_TTC_S)
            pred_approaching = vz_smooth[i] < 0
            pred_ttc_eval = z_smooth[i] / max(-vz_smooth[i], 1e-6) if pred_approaching else np.inf
            rows.append({"sequence_id": seq, "pred_track_id": tid, "frame": g.loc[i, "frame"],
                         "gt_is_risk": gt_is_risk, "pred_approaching": pred_approaching,
                         "pred_ttc_eval": pred_ttc_eval})
    return pd.DataFrame(rows)

eval_pinhole_3seq = build_eval_for_depth_source(depth_compare_sorted, "pred_z_pinhole")
eval_da_3seq = build_eval_for_depth_source(depth_compare_sorted, "pred_z_depthanything")

print(f"Eval pinhole (3 seq): {len(eval_pinhole_3seq)} sample | "
      f"Eval Depth-Anything (3 seq): {len(eval_da_3seq)} sample")

TOTAL_HOURS_3SEQ = sum(
    (g["frame"].max() - g["frame"].min() + 1) / KITTI_FPS
    for _, g in depth_compare_sorted.groupby("sequence_id")
) / 3600.0
print(f"Tong thoi luong 3 sequence nay: {TOTAL_HOURS_3SEQ * 3600:.1f}s")

rows_compare = []
for label, eval_df in [("Pinhole + class-height", eval_pinhole_3seq), ("Depth Anything V2 Metric", eval_da_3seq)]:
    for th in [1.0, 1.5, 2.0]:
        # Tinh truc tiep cong thuc windowed (khong tai dung ham cu vi schema cot khac nhau)
        df = eval_df.copy()
        df["window_id"] = (df["frame"] // EVENT_WINDOW_FRAMES).astype(int)
        df["pred_warn"] = df["pred_approaching"] & (df["pred_ttc_eval"] <= th)
        grouped = (df.groupby(["sequence_id", "pred_track_id", "window_id"])
                   .agg(gt_is_risk=("gt_is_risk", "any"), pred_warn=("pred_warn", "any")).reset_index())
        TP = int((grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        FP = int((grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
        FN = int((~grouped["pred_warn"] & grouped["gt_is_risk"]).sum())
        TN = int((~grouped["pred_warn"] & ~grouped["gt_is_risk"]).sum())
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        far = FP / TOTAL_HOURS_3SEQ if TOTAL_HOURS_3SEQ > 0 else float("nan")
        rows_compare.append({"Depth_source": label, "TTC_Threshold_s": th,
                              "Precision": round(precision, 4), "Recall": round(recall, 4),
                              "F1": round(f1, 4), "FAR_per_hour": round(far, 2)})

depth_propagation_df = pd.DataFrame(rows_compare)
depth_propagation_df.to_csv(TABLE_DIR / "table_depth_propagation_FIXED.csv", index=False)
display(depth_propagation_df)

best_pinhole = depth_propagation_df[depth_propagation_df["Depth_source"] == "Pinhole + class-height"].sort_values("F1", ascending=False).iloc[0]
best_da = depth_propagation_df[depth_propagation_df["Depth_source"] == "Depth Anything V2 Metric"].sort_values("F1", ascending=False).iloc[0]
print(f"\nF1 tot nhat - Pinhole (tren 3 sequence nay)        : {best_pinhole['F1']:.3f}")
print(f"F1 tot nhat - Depth Anything V2 (tren 3 sequence nay): {best_da['F1']:.3f}")
if best_da["F1"] > best_pinhole["F1"]:
    print("-> Depth Anything V2 cai thien CA distance VA F1/Precision/FAR thuc su.")
    print("   De xuat: mo rong chay Depth Anything V2 cho toan bo 11 sequence (Phan 1-6)")
    print("   de co ket qua day du, roi dung lam pipeline chinh trong paper.")
else:
    print("-> Depth Anything V2 cai thien distance MAE/RMSE nhung CHUA chuyen thanh cai thien")
    print("   F1/Precision/FAR ro ret tren 3 sequence nay. Co the do mau con nho, hoac do")
    print("   bottleneck nam o buoc khac (vi du smoothing/TTC logic), khong chi o depth.")
    print("   Bao cao trung thuc ca hai ket qua (distance va F1) trong paper, khong chi noi mot chieu.")

## Phan 8 - Mo rong dataset ngoai + trajectory forecasting (scoped)

Da kiem tra cau truc thuc te truoc khi viet:
- **nuScenes**: dung `nuscenes-devkit`, co LiDAR 3D thuc.
- **BDD100K**: KHONG co nhan 3D -> chi sanity check detection 2D.
- **CCD**: cell "kham pha" truoc - mo ta Kaggle ("tabular, numpy") goi y co the
  la dac trung trich xuat san, khong phai video tho. Gui lai output discovery
  de viet tiep dung, khong doan mu.
- **DoTA, DAD**: KHONG tich hop - DoTA can pipeline YouTube+ffmpeg rieng, DAD o
  Baidu Pan 53-116GB va la bai toan driver-attention khac collision-proxy.
- **MTR++/Social-GAT**: KHONG implement nguyen ban (qui mo nghien cuu nhieu
  tuan). Thay bang constant-velocity vs GRU don gian tren du lieu KITTI da co -
  ghi ro la scoped-down, MTR++/Social-GAT day du van la Future Work.

In [ ]:
# ==============================================================================
# 8a. NUSCENES - GENERALIZATION TEST CHO DISTANCE/DEPTH (CO LIDAR 3D THUC)
# ==============================================================================
# DA SUA: tim dung "v1.0-trainval" (khong phai v1.0-mini), va phan biet 2 thu
# muc trung ten long nhau - chi lay thu muc NGOAI (co samples/sweeps/maps ben
# trong) lam dataroot, khong lay thu muc trong (chi co file JSON).
!pip install -q nuscenes-devkit

import glob
from nuscenes.nuscenes import NuScenes

print("Danh sach tat ca thu muc /kaggle/input cap 1-2 (de doi chieu neu can):")
for p in sorted(Path("/kaggle/input").glob("*")):
    print(" ", p)

NUSCENES_VERSION = "v1.0-trainval"  # DA SUA: truoc la "v1.0-mini", sai voi du lieu Lan dang dung
_v1_candidates = glob.glob(f"/kaggle/input/**/{NUSCENES_VERSION}", recursive=True)
print(f"\nTat ca thu muc '{NUSCENES_VERSION}' tim duoc:", _v1_candidates)

NUSCENES_DATAROOT = None
for cand in _v1_candidates:
    cand_path = Path(cand)
    # dataroot DUNG phai co samples/ va sweeps/ la con truc tiep (thu muc long ben
    # trong chi co file JSON, khong co samples/sweeps) -> loai duoc nham lan 2 ten trung.
    if (cand_path / "samples").exists() and (cand_path / "sweeps").exists():
        NUSCENES_DATAROOT = str(cand_path)
        break

if NUSCENES_DATAROOT is None:
    print(f"\nKhong tu dong xac dinh duoc dataroot dung (can co ca samples/ va sweeps/ ben trong).")
    print(f"Danh sach candidates da tim: {_v1_candidates}")
    print("Hay chon dung duong dan tu danh sach tren va gan thu cong, vi du:")
    print("  NUSCENES_DATAROOT = '/kaggle/input/<ten-dataset>/v1.0-trainval'")
    raise RuntimeError("Xem huong dan ngay tren de gan NUSCENES_DATAROOT thu cong roi chay lai cell nay.")

print("\nDang dung NUSCENES_DATAROOT =", NUSCENES_DATAROOT)
nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=True)

NUSCENES_SAMPLE_LIMIT = 150  # v1.0-trainval co 850 scene (rat lon) - gioi han de kiem soat thoi gian
nusc_model = YOLO(YOLO_WEIGHTS)
nusc_rows = []
n_processed = 0

for scene in nusc.scene:
    sample_token = scene["first_sample_token"]
    while sample_token and n_processed < NUSCENES_SAMPLE_LIMIT:
        sample = nusc.get("sample", sample_token)
        cam_token = sample["data"].get("CAM_FRONT")
        if cam_token is None:
            sample_token = sample["next"]
            continue

        data_path, boxes, camera_intrinsic = nusc.get_sample_data(cam_token)
        res = nusc_model.predict(str(data_path), imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]

        gt_boxes_cam = []
        for box in boxes:
            cat = box.name.split(".")[0]
            if cat not in ("vehicle", "human"):
                continue
            if box.center[2] <= 0:
                continue
            gt_boxes_cam.append({"gt_x": float(box.center[0]), "gt_z": float(box.center[2]), "category": cat})

        if res.boxes is not None and len(res.boxes) > 0 and len(gt_boxes_cam) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            fy_nusc = float(camera_intrinsic[1, 1])
            for box_xyxy in xyxy:
                pred_h_pixel = box_xyxy[3] - box_xyxy[1]
                if pred_h_pixel <= 1:
                    continue
                pred_z_nusc = (fy_nusc * 1.6) / pred_h_pixel
                closest_gt = min(gt_boxes_cam, key=lambda g: abs(g["gt_z"] - pred_z_nusc))
                nusc_rows.append({"pred_z": pred_z_nusc, "gt_z": closest_gt["gt_z"]})

        n_processed += 1
        sample_token = sample["next"]
    if n_processed >= NUSCENES_SAMPLE_LIMIT:
        break

nusc_eval_df = pd.DataFrame(nusc_rows)
print(f"\nSo sample da xu ly: {n_processed} | so cap pred-gt (xap xi): {len(nusc_eval_df)}")

if len(nusc_eval_df) == 0:
    print("CANH BAO: khong co cap pred-gt nao. Kiem tra lai NUSCENES_DATAROOT/NUSCENES_VERSION.")
else:
    nusc_mae = float((nusc_eval_df["pred_z"] - nusc_eval_df["gt_z"]).abs().mean())
    nusc_rmse = float(np.sqrt(((nusc_eval_df["pred_z"] - nusc_eval_df["gt_z"]) ** 2).mean()))
    print(f"nuScenes (zero-shot): distance MAE={nusc_mae:.2f}m RMSE={nusc_rmse:.2f}m")
    print("\nLUU Y: ghep cap pred-gt o day dung 'gt_z gan nhat' (XAP XI, khong qua IoU 2D thuc).")
    print("Chi mang tinh chi bao xu huong generalization, khong nen bao cao nhu so sanh cong bang 1-1.")
    nusc_eval_df.to_csv(TABLE_DIR / "nuscenes_generalization_FIXED.csv", index=False)

In [ ]:
# ==============================================================================
# 8b. BDD100K - SANITY CHECK DETECTION 2D (KHONG CO NHAN 3D)
# ==============================================================================
from pathlib import Path
import numpy as np

# BẠN DÁN ĐƯỜNG DẪN TRỰC TIẾP TỚI THƯ MỤC ẢNH VAL VÀO ĐÂY:
DIRECT_BDD_PATH = "/kaggle/input/datasets/awsaf49/bdd100k-dataset/bdd100k/bdd100k/images/100k/val" 
BDD_IMG_DIR = Path(DIRECT_BDD_PATH)

# Kiểm tra nhanh xem đường dẫn có tồn tại hay không
if not BDD_IMG_DIR.exists():
    raise RuntimeError(f"Đường dẫn bạn cung cấp không tồn tại: {BDD_IMG_DIR}")

print("Dang dung BDD_IMG_DIR =", BDD_IMG_DIR)

BDD_SAMPLE_N = 200
bdd_images = sorted(BDD_IMG_DIR.glob("*.jpg"))[:BDD_SAMPLE_N]
print(f"Lay mau {len(bdd_images)} anh de sanity check.")

if len(bdd_images) == 0:
    raise RuntimeError(f"Thu muc {BDD_IMG_DIR} khong co file .jpg nao. Kiem tra lai duong dan!")

# Giả định YOLO, CONF_THRES, IOU_THRES, IMG_SIZE và pred_all đã được khai báo trước đó
bdd_model = YOLO(YOLO_WEIGHTS)
bdd_detection_counts, bdd_conf_scores = [], []

for img_path in bdd_images:
    res = bdd_model.predict(str(img_path), imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    n_det = len(res.boxes) if res.boxes is not None else 0
    bdd_detection_counts.append(n_det)
    if res.boxes is not None and len(res.boxes) > 0:
        bdd_conf_scores.extend(res.boxes.conf.cpu().numpy().tolist())

# Tính toán so sánh
kitti_det_per_frame = pred_all.groupby(["sequence_id", "frame"]).size().mean()
print(f"\nSo detection trung binh/anh tren BDD100K: {np.mean(bdd_detection_counts):.2f}")
print(f"So detection trung binh/frame tren KITTI : {kitti_det_per_frame:.2f}")

bdd_mean_conf = np.mean(bdd_conf_scores) if len(bdd_conf_scores) > 0 else 0
print(f"Confidence trung binh tren BDD100K: {bdd_mean_conf:.3f} (n={len(bdd_conf_scores)})")
print(f"Confidence trung binh tren KITTI  : {pred_all['score'].mean():.3f}")

print("\nDay la sanity check domain shift, KHONG phai benchmark Recall/mAP day du (thieu nhan 3D).")

In [ ]:
# ==============================================================================
# 8c. CCD - CELL KHAM PHA CAU TRUC THUC (CHAY TRUOC, CHUA DANH GIA GI)
# ==============================================================================
from pathlib import Path

# BẠN DÁN ĐƯỜNG DẪN TRỰC TIẾP TỚI THƯ MỤC ROOT CỦA DATASET CCD VÀO ĐÂY:
DIRECT_CCD_PATH = "/kaggle/input/datasets/asefjamilajwad/car-crash-dataset-ccd/CrashBest" 
CCD_ROOT = Path(DIRECT_CCD_PATH)

# Kiểm tra nhanh xem đường dẫn có tồn tại hay không
if not CCD_ROOT.exists():
    raise RuntimeError(f"Đường dẫn bạn cung cấp không tồn tại: {CCD_ROOT}. Hãy kiểm tra lại!")

print(f"\nCau truc thu muc duoi {CCD_ROOT} (toi da 60 dong, sau 3 cap):")
count = 0
for p in sorted(CCD_ROOT.rglob("*")):
    depth = len(p.relative_to(CCD_ROOT).parts)
    if depth > 3:
        continue
    print(" " * (depth * 2), p.name, "(dir)" if p.is_dir() else f"({p.suffix})")
    count += 1
    if count >= 60:
        print("  ... (con nua, da cat o 60 dong)")
        break

print("\nGui lai output nay de viet tiep cell danh gia chinh xac (can biet co video/.mp4")
print("hay chi co .npy dac trung, va ten/cot file metadata thoi diem va cham).")

In [ ]:
# ==============================================================================
# 8d. TRAJECTORY FORECASTING - SCOPED: CONSTANT-VELOCITY vs GRU DON GIAN
# ==============================================================================
# KHONG implement MTR++/Social-GAT day du (du an nhieu tuan). So sanh constant-
# velocity (vat ly hien tai) voi 1 GRU don gian hoc tu chinh du lieu KITTI -
# ghi ro day la "simplified learned trajectory baseline", khong claim tuong
# duong MTR++/Social-GAT.

import torch
import torch.nn as nn

SEQ_LEN_IN, SEQ_LEN_OUT = 5, 3
traj_dfs = []
for (seq, tid), g in distance_pairs.sort_values(["sequence_id", "pred_track_id", "frame"]).groupby(
        ["sequence_id", "pred_track_id"]):
    g = g.drop_duplicates(subset=["frame"]).sort_values("frame").reset_index(drop=True)
    if len(g) < SEQ_LEN_IN + SEQ_LEN_OUT:
        continue
    xz = g[["gt_x", "gt_z"]].values
    for i in range(len(g) - SEQ_LEN_IN - SEQ_LEN_OUT + 1):
        traj_dfs.append({"sequence_id": seq, "track_id": tid,
                          "input": xz[i:i + SEQ_LEN_IN], "target": xz[i + SEQ_LEN_IN:i + SEQ_LEN_IN + SEQ_LEN_OUT]})

print(f"So mau trajectory: {len(traj_dfs)}")
if len(traj_dfs) < 20:
    print("CANH BAO: qua it mau. Can them sequence (Cell 1, Tuan 1).")
else:
    seqs_unique = sorted(set(d["sequence_id"] for d in traj_dfs))
    n_train_seq = max(1, int(len(seqs_unique) * 0.7))
    train_seqs_set = set(seqs_unique[:n_train_seq])
    train_data = [d for d in traj_dfs if d["sequence_id"] in train_seqs_set]
    test_data = [d for d in traj_dfs if d["sequence_id"] not in train_seqs_set]
    print(f"Train: {len(train_data)} (seq {sorted(train_seqs_set)}) | Test: {len(test_data)}")

    def constant_velocity_predict(input_xz, n_out):
        v = input_xz[-1] - input_xz[-2]
        return np.array([input_xz[-1] + v * (k + 1) for k in range(n_out)])

    def compute_ade_fde(preds, targets):
        ade = np.mean([np.linalg.norm(p - t, axis=1).mean() for p, t in zip(preds, targets)])
        fde = np.mean([np.linalg.norm(p[-1] - t[-1]) for p, t in zip(preds, targets)])
        return float(ade), float(fde)

    cv_preds = [constant_velocity_predict(d["input"], SEQ_LEN_OUT) for d in test_data]
    cv_targets = [d["target"] for d in test_data]
    cv_ade, cv_fde = compute_ade_fde(cv_preds, cv_targets)
    print(f"\nConstant-velocity: ADE={cv_ade:.3f}m, FDE={cv_fde:.3f}m")

    # DA SUA: GRU truoc du doan toa do TUYET DOI (X,Z toi 80m) khong chuan hoa -> khong hoc duoc
    # (loss khong giam, ADE=23m). Sua sang du doan DICH CHUYEN TUONG DOI so voi vi tri cuoi quan
    # sat - chuan trong trajectory forecasting, on dinh hon nhieu cho mang nho.
    def to_relative(sample):
        last_pos = sample["input"][-1]
        return sample["input"] - last_pos, sample["target"] - last_pos, last_pos

    class SimpleGRUPredictor(nn.Module):
        def __init__(self, hidden=32, out_steps=SEQ_LEN_OUT):
            super().__init__()
            self.gru = nn.GRU(input_size=2, hidden_size=hidden, batch_first=True)
            self.head = nn.Linear(hidden, out_steps * 2)
            self.out_steps = out_steps

        def forward(self, x):
            _, h = self.gru(x)
            return self.head(h.squeeze(0)).view(-1, self.out_steps, 2)

    train_rel = [to_relative(d) for d in train_data]
    test_rel = [to_relative(d) for d in test_data]
    X_train = torch.tensor(np.stack([r[0] for r in train_rel]), dtype=torch.float32)
    Y_train = torch.tensor(np.stack([r[1] for r in train_rel]), dtype=torch.float32)
    X_test = torch.tensor(np.stack([r[0] for r in test_rel]), dtype=torch.float32)
    test_last_pos = np.stack([r[2] for r in test_rel])
    Y_test_np_rel = np.stack([r[1] for r in test_rel])
    Y_test_np = Y_test_np_rel + test_last_pos[:, None, :]

    device_gru = "cuda" if torch.cuda.is_available() else "cpu"
    gru_model = SimpleGRUPredictor().to(device_gru)
    optimizer = torch.optim.Adam(gru_model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    X_train, Y_train = X_train.to(device_gru), Y_train.to(device_gru)
    gru_model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        loss = loss_fn(gru_model(X_train), Y_train)
        loss.backward()
        optimizer.step()
    print(f"GRU training xong, loss cuoi: {loss.item():.4f}")

    gru_model.eval()
    with torch.no_grad():
        gru_preds_rel = gru_model(X_test.to(device_gru)).cpu().numpy()
    gru_preds = gru_preds_rel + test_last_pos[:, None, :]
    gru_ade, gru_fde = compute_ade_fde(list(gru_preds), list(Y_test_np))
    print(f"GRU don gian: ADE={gru_ade:.3f}m, FDE={gru_fde:.3f}m")

    if gru_ade < cv_ade:
        print(f"\n-> GRU cai thien ADE thuc su ({gru_ade:.3f}m < {cv_ade:.3f}m).")
    else:
        print(f"\n-> GRU KHONG cai thien so voi constant-velocity. Day la ket qua THAT.")

    pd.DataFrame([
        {"Method": "Constant-velocity", "ADE_m": round(cv_ade, 3), "FDE_m": round(cv_fde, 3)},
        {"Method": "Simple GRU (scoped)", "ADE_m": round(gru_ade, 3), "FDE_m": round(gru_fde, 3)},
    ]).to_csv(TABLE_DIR / "table_trajectory_forecasting_FIXED.csv", index=False)